# Stage 0: Connect GitHub repository

In [ ]:
from pathlib import Path
from google.colab import userdata
import subprocess


GITHUB_USERNAME = "swanksenia"
REPOSITORY_NAME = "health-psychology-rag-kb"

PROJECT_ROOT = Path(
    f"/content/{REPOSITORY_NAME}"
)

GITHUB_TOKEN = userdata.get(
    "GITHUB_TOKEN"
)

if not GITHUB_TOKEN:
    raise RuntimeError(
        "GITHUB_TOKEN is missing. "
        "Add it in Colab Secrets and enable notebook access."
    )

authenticated_url = (
    f"https://{GITHUB_USERNAME}:"
    f"{GITHUB_TOKEN}@github.com/"
    f"{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}.git"
)

if PROJECT_ROOT.exists():
    print(
        "Repository already exists:",
        PROJECT_ROOT
    )
else:
    clone_result = subprocess.run(
        [
            "git",
            "clone",
            authenticated_url,
            str(PROJECT_ROOT),
        ],
        capture_output=True,
        text=True,
    )

    if clone_result.returncode != 0:
        raise RuntimeError(
            "Git clone failed:\n"
            + clone_result.stderr
        )

    print(
        "Repository cloned:",
        PROJECT_ROOT
    )


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Repository exists:",
    PROJECT_ROOT.exists()
)


Repository cloned: /content/health-psychology-rag-kb
Project root: /content/health-psychology-rag-kb
Repository exists: True


In [ ]:
RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

NORMALIZED_DIR = (
    PROJECT_ROOT
    / "data"
    / "normalized"
)

ASSETS_DIR = (
    NORMALIZED_DIR
    / "assets"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


for directory in [
    RAW_DIR,
    NORMALIZED_DIR,
]:
    assert directory.exists(), (
        f"Missing directory: {directory}"
    )


for directory in [
    ASSETS_DIR,
    PROCESSED_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("RAW_DIR:", RAW_DIR)
print("NORMALIZED_DIR:", NORMALIZED_DIR)
print("ASSETS_DIR:", ASSETS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

RAW_DIR: /content/health-psychology-rag-kb/data/raw
NORMALIZED_DIR: /content/health-psychology-rag-kb/data/normalized
ASSETS_DIR: /content/health-psychology-rag-kb/data/normalized/assets
PROCESSED_DIR: /content/health-psychology-rag-kb/data/processed


In [ ]:
!git -C /content/health-psychology-rag-kb config user.name "Kseniia Lebedeva"
!git -C /content/health-psychology-rag-kb config user.email "lebedevaky@gmail.com"

In [ ]:
!pip -q install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 43.3 MB/s eta 0:00:00


# Document 1: Course Syllabus


In [ ]:
SYLLABUS_SOURCE_PATH = (
    RAW_DIR
    / "course_syllabus.pdf"
)

SYLLABUS_OUTPUT_PATH = (
    NORMALIZED_DIR
    / "course_syllabus.md"
)

assert SYLLABUS_SOURCE_PATH.exists(), (
    f"Missing source file: "
    f"{SYLLABUS_SOURCE_PATH}"
)

print(
    "Syllabus source:",
    SYLLABUS_SOURCE_PATH
)

Syllabus source: /content/health-psychology-rag-kb/data/raw/course_syllabus.pdf


In [ ]:
import fitz


document = fitz.open(
    SYLLABUS_SOURCE_PATH
)

In [ ]:
import pymupdf


PDF_PATH = RAW_DIR / "course_syllabus.pdf"

print("PDF path:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())

assert PDF_PATH.exists(), (
    f"PDF не знайдено: {PDF_PATH}"
)


with pymupdf.open(PDF_PATH) as pdf_document:
    print(
        "Кількість сторінок:",
        len(pdf_document)
    )

    for page_index in range(
        min(5, len(pdf_document))
    ):
        page = pdf_document[page_index]

        text = page.get_text(
            "text",
            sort=True,
        ).strip()

        print("\n" + "=" * 80)
        print(f"PDF page {page_index + 1}")
        print("Characters:", len(text))
        print("=" * 80)
        print(text[:1500])

PDF path: /content/health-psychology-rag-kb/data/raw/course_syllabus.pdf
PDF exists: True
Кількість сторінок: 8

PDF page 1
Characters: 2769
Syllabus for Introduction to Health Psychology

                              Credits: 3
                     PSYC 1111


Instructor Contact Information:

You can always send your instructor a private message through the Brightspace Messaging system,
accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click
your instructor’s profile page to see all the ways you can communicate with them, including their email
address.

Course Description

Health psychology focuses on the dynamic interaction between biological, social, and psychological factors
that influence physical health and illness, aiming to promote overall well-being and prevent diseases. This
course is designed to provide students with an introduction to the field of health psychology. Students will
learn the crucial role of behavior in shapi

In [ ]:
PAGE_NUMBER = 5

page = document[PAGE_NUMBER - 1]
page_5_text = page.get_text("text", sort=True).strip()

print(page_5_text)

Discussion  40%   1.  Discussion Forum Unit 1               CLO 1, CLO3
     Forums           2.  Discussion Forum Unit 2               CLO3
                           3.  Discussion Forum Unit 4               CLO 3, CLO4
                           4.  Discussion Forum Unit 5               CLO 1, CLO3, CLO4
                           5.  Discussion Forum Unit 6               CLO 1, CLO3, CLO4
                           6.  Discussion Forum Unit 8               CLO 1, CLO4, CLO5

  Assignment  40%    1.  Assignment Activity Unit 2              CLO3
     Activities           2.  Assignment Activity Unit 3              CLO1, CLO 2
                           3.  Assignment Activity Unit 5              CLO 1, CLO3, CLO4
                           4.  Assignment Activity Unit 7              CLO 3, CLO 4

 Graded Quiz  10%    1.  Graded Quiz Unit 3                    CLO1, CLO 2
                           2.  Graded Quiz Unit 6                    CLO 1, CLO3, CLO4

   Final Exam  

In [ ]:
table_finder = page.find_tables()

print("Знайдено таблиць:", len(table_finder.tables))

Знайдено таблиць: 0


### Table extraction decision

PyMuPDF successfully extracted the syllabus text layer, but `find_tables()` did not detect the grading tables on PDF pages 4–5. Because the document contains only eight pages and the tables are small and stable, they were manually normalized into structured Markdown after visual verification against the source PDF.

This preserves the relationships between categories, percentages, assessment items, learning outcomes, letter grades, and grade points.

In [ ]:
grading_weights_rows = [
    {
        "category": "Discussion Forums",
        "weight": "40%",
        "grade_item": "Discussion Forum Unit 1",
        "learning_outcomes": "CLO 1, CLO 3",
    },
    {
        "category": "Discussion Forums",
        "weight": "40%",
        "grade_item": "Discussion Forum Unit 2",
        "learning_outcomes": "CLO 3",
    },
    {
        "category": "Discussion Forums",
        "weight": "40%",
        "grade_item": "Discussion Forum Unit 4",
        "learning_outcomes": "CLO 3, CLO 4",
    },
    {
        "category": "Discussion Forums",
        "weight": "40%",
        "grade_item": "Discussion Forum Unit 5",
        "learning_outcomes": "CLO 1, CLO 3, CLO 4",
    },
    {
        "category": "Discussion Forums",
        "weight": "40%",
        "grade_item": "Discussion Forum Unit 6",
        "learning_outcomes": "CLO 1, CLO 3, CLO 4",
    },
    {
        "category": "Discussion Forums",
        "weight": "40%",
        "grade_item": "Discussion Forum Unit 8",
        "learning_outcomes": "CLO 1, CLO 4, CLO 5",
    },
    {
        "category": "Assignment Activities",
        "weight": "40%",
        "grade_item": "Assignment Activity Unit 2",
        "learning_outcomes": "CLO 3",
    },
    {
        "category": "Assignment Activities",
        "weight": "40%",
        "grade_item": "Assignment Activity Unit 3",
        "learning_outcomes": "CLO 1, CLO 2",
    },
    {
        "category": "Assignment Activities",
        "weight": "40%",
        "grade_item": "Assignment Activity Unit 5",
        "learning_outcomes": "CLO 1, CLO 3, CLO 4",
    },
    {
        "category": "Assignment Activities",
        "weight": "40%",
        "grade_item": "Assignment Activity Unit 7",
        "learning_outcomes": "CLO 3, CLO 4",
    },
    {
        "category": "Graded Quiz",
        "weight": "10%",
        "grade_item": "Graded Quiz Unit 3",
        "learning_outcomes": "CLO 1, CLO 2",
    },
    {
        "category": "Graded Quiz",
        "weight": "10%",
        "grade_item": "Graded Quiz Unit 6",
        "learning_outcomes": "CLO 1, CLO 3, CLO 4",
    },
    {
        "category": "Final Exam",
        "weight": "10%",
        "grade_item": "Final Exam",
        "learning_outcomes": "CLO 1, CLO 2, CLO 3, CLO 4, CLO 5",
    },
]
grading_weights_rows[:2]

[{'category': 'Discussion Forums',
  'weight': '40%',
  'grade_item': 'Discussion Forum Unit 1',
  'learning_outcomes': 'CLO 1, CLO 3'},
 {'category': 'Discussion Forums',
  'weight': '40%',
  'grade_item': 'Discussion Forum Unit 2',
  'learning_outcomes': 'CLO 3'}]

In [ ]:
grading_scale_rows = [
    {"letter_grade": "A+", "percentage": "98%-100%", "grade_points": "4.00"},
    {"letter_grade": "A",  "percentage": "93%-97%",  "grade_points": "4.00"},
    {"letter_grade": "A-", "percentage": "90%-92%",  "grade_points": "3.67"},
    {"letter_grade": "B+", "percentage": "88%-89%",  "grade_points": "3.33"},
    {"letter_grade": "B",  "percentage": "83%-87%",  "grade_points": "3.00"},
    {"letter_grade": "B-", "percentage": "80%-82%",  "grade_points": "2.67"},
    {"letter_grade": "C+", "percentage": "78%-79%",  "grade_points": "2.33"},
    {"letter_grade": "C",  "percentage": "73%-77%",  "grade_points": "2.00"},
    {"letter_grade": "C-", "percentage": "70%-72%",  "grade_points": "1.67"},
    {"letter_grade": "D+", "percentage": "68%-69%",  "grade_points": "1.33"},
    {"letter_grade": "D",  "percentage": "63%-67%",  "grade_points": "1.00"},
    {"letter_grade": "D-", "percentage": "60%-62%",  "grade_points": "0.67"},
    {"letter_grade": "F",  "percentage": "<60%",      "grade_points": "0.00"},
    {"letter_grade": "W",  "percentage": "N/A",      "grade_points": "N/A"},
]
grading_scale_rows[:2]

[{'letter_grade': 'A+', 'percentage': '98%-100%', 'grade_points': '4.00'},
 {'letter_grade': 'A', 'percentage': '93%-97%', 'grade_points': '4.00'}]

In [ ]:
DOCUMENT_METADATA = {
    "document_id": "health_psychology_course_syllabus",
    "source_file": "data/raw/course_syllabus.pdf",
    "title": "Syllabus for Introduction to Health Psychology",
    "course_id": "PSYC_1111",
    "source_type": "pdf",
    "document_type": "syllabus",
    "content_role": "course_structure_and_assessment",
    "language": "en",
    "access_level": "course_use",
}

print(DOCUMENT_METADATA)

{'document_id': 'health_psychology_course_syllabus', 'source_file': 'data/raw/course_syllabus.pdf', 'title': 'Syllabus for Introduction to Health Psychology', 'course_id': 'PSYC_1111', 'source_type': 'pdf', 'document_type': 'syllabus', 'content_role': 'course_structure_and_assessment', 'language': 'en', 'access_level': 'course_use'}


In [ ]:
def dict_rows_to_markdown(rows, columns):
    """
    Convert structured dictionary rows into a Markdown table.

    columns:
        list of tuples:
        (dictionary_key, visible_column_name)
    """
    header = "| " + " | ".join(
        label for _, label in columns
    ) + " |"

    separator = "| " + " | ".join(
        "---" for _ in columns
    ) + " |"

    body = []

    for row in rows:
        values = [
            str(row.get(key, ""))
            .replace("|", r"\|")
            .strip()
            for key, _ in columns
        ]

        body.append(
            "| " + " | ".join(values) + " |"
        )

    return "\n".join([header, separator, *body])

In [ ]:
grading_weights_markdown = dict_rows_to_markdown(
    grading_weights_rows,
    columns=[
        ("category", "Category"),
        ("weight", "Weight"),
        ("grade_item", "Grade Item"),
        (
            "learning_outcomes",
            "Associated Learning Outcomes"
        ),
    ],
)

print(grading_weights_markdown)

| Category | Weight | Grade Item | Associated Learning Outcomes |
| --- | --- | --- | --- |
| Discussion Forums | 40% | Discussion Forum Unit 1 | CLO 1, CLO 3 |
| Discussion Forums | 40% | Discussion Forum Unit 2 | CLO 3 |
| Discussion Forums | 40% | Discussion Forum Unit 4 | CLO 3, CLO 4 |
| Discussion Forums | 40% | Discussion Forum Unit 5 | CLO 1, CLO 3, CLO 4 |
| Discussion Forums | 40% | Discussion Forum Unit 6 | CLO 1, CLO 3, CLO 4 |
| Discussion Forums | 40% | Discussion Forum Unit 8 | CLO 1, CLO 4, CLO 5 |
| Assignment Activities | 40% | Assignment Activity Unit 2 | CLO 3 |
| Assignment Activities | 40% | Assignment Activity Unit 3 | CLO 1, CLO 2 |
| Assignment Activities | 40% | Assignment Activity Unit 5 | CLO 1, CLO 3, CLO 4 |
| Assignment Activities | 40% | Assignment Activity Unit 7 | CLO 3, CLO 4 |
| Graded Quiz | 10% | Graded Quiz Unit 3 | CLO 1, CLO 2 |
| Graded Quiz | 10% | Graded Quiz Unit 6 | CLO 1, CLO 3, CLO 4 |
| Final Exam | 10% | Final Exam | CLO 1, CLO 2, CLO 3

In [ ]:
grading_scale_markdown = dict_rows_to_markdown(
    grading_scale_rows,
    columns=[
        ("letter_grade", "Letter Grade"),
        ("percentage", "Percentage Grade"),
        ("grade_points", "Grade Points"),
    ],
)

print(grading_scale_markdown)

| Letter Grade | Percentage Grade | Grade Points |
| --- | --- | --- |
| A+ | 98%-100% | 4.00 |
| A | 93%-97% | 4.00 |
| A- | 90%-92% | 3.67 |
| B+ | 88%-89% | 3.33 |
| B | 83%-87% | 3.00 |
| B- | 80%-82% | 2.67 |
| C+ | 78%-79% | 2.33 |
| C | 73%-77% | 2.00 |
| C- | 70%-72% | 1.67 |
| D+ | 68%-69% | 1.33 |
| D | 63%-67% | 1.00 |
| D- | 60%-62% | 0.67 |
| F | <60% | 0.00 |
| W | N/A | N/A |


In [ ]:
grading_tables_markdown = f"""
<!-- section: Evaluation and Grading -->

## Evaluation and Grading

<!-- table_id: syllabus_grading_weights -->
<!-- page_start: 4 -->
<!-- page_end: 5 -->
<!-- content_type: table -->
<!-- normalization_method: manual_structured_transcription -->

### Grading Weights

{grading_weights_markdown}

<!-- table_id: syllabus_grading_scale -->
<!-- page_start: 5 -->
<!-- page_end: 5 -->
<!-- content_type: table -->
<!-- normalization_method: manual_structured_transcription -->

### Grading Scale

{grading_scale_markdown}

### Additional Grade Statuses

Students may also be granted Withdrawal (W) if they withdraw from the course, or an Incomplete (I) should their circumstances permit. A student who feels they were graded unfairly, or who seeks to dispute a grade, may initiate a grade appeal process. Refer to University Policies for more information on withdrawals and appeals.
""".strip()

print(grading_tables_markdown)

<!-- section: Evaluation and Grading -->

## Evaluation and Grading

<!-- table_id: syllabus_grading_weights -->
<!-- page_start: 4 -->
<!-- page_end: 5 -->
<!-- content_type: table -->
<!-- normalization_method: manual_structured_transcription -->

### Grading Weights

| Category | Weight | Grade Item | Associated Learning Outcomes |
| --- | --- | --- | --- |
| Discussion Forums | 40% | Discussion Forum Unit 1 | CLO 1, CLO 3 |
| Discussion Forums | 40% | Discussion Forum Unit 2 | CLO 3 |
| Discussion Forums | 40% | Discussion Forum Unit 4 | CLO 3, CLO 4 |
| Discussion Forums | 40% | Discussion Forum Unit 5 | CLO 1, CLO 3, CLO 4 |
| Discussion Forums | 40% | Discussion Forum Unit 6 | CLO 1, CLO 3, CLO 4 |
| Discussion Forums | 40% | Discussion Forum Unit 8 | CLO 1, CLO 4, CLO 5 |
| Assignment Activities | 40% | Assignment Activity Unit 2 | CLO 3 |
| Assignment Activities | 40% | Assignment Activity Unit 3 | CLO 1, CLO 2 |
| Assignment Activities | 40% | Assignment Activity Unit 5 | CLO

In [ ]:
import re


def clean_syllabus_text(text: str) -> str:
    """
    Clean extracted PDF text while preserving meaning.
    """

    # Службові символи PDF
    text = text.replace("\u00ad", "")   # soft hyphen
    text = text.replace("\xa0", " ")    # non-breaking space
    text = text.replace("\t", " ")

    # Зламані bullet symbols → нормальний Markdown bullet
    text = text.replace("", "-")
    text = text.replace("", "-")
    text = text.replace("●", "-")
    text = text.replace("▪", "-")

    # Склеюємо слова, розірвані переносом
    # psycho-\nlogical → psychological
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Прибираємо окремі номери сторінок
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Прибираємо зайві пробіли на початку рядків
    text = re.sub(r"(?m)^[ \t]+", "", text)

    # Кілька пробілів → один
    text = re.sub(r"[ ]{2,}", " ", text)

    # Пробіли перед переносом прибираємо
    text = re.sub(r"[ \t]+\n", "\n", text)

    # Три й більше порожніх рядків → один порожній рядок
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
page_1_raw = document[0].get_text("text", sort=True)
page_1_clean = clean_syllabus_text(page_1_raw)

print(page_1_clean[:2000])

Syllabus for Introduction to Health Psychology

Credits: 3
PSYC 1111

Instructor Contact Information:

You can always send your instructor a private message through the Brightspace Messaging system,
accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click
your instructor’s profile page to see all the ways you can communicate with them, including their email
address.

Course Description

Health psychology focuses on the dynamic interaction between biological, social, and psychological factors
that influence physical health and illness, aiming to promote overall well-being and prevent diseases. This
course is designed to provide students with an introduction to the field of health psychology. Students will
learn the crucial role of behavior in shaping health practices and the influence of culture and gender on
health and health outcomes measured through quality of life, health-related quality of life, and life
expectancy. Additionally, th

In [ ]:
def extract_and_trim_page(page_number: int) -> str:
    """
    Extract and clean one PDF page.
    Applies page-specific trimming where needed.
    """

    raw_text = document[page_number - 1].get_text("text", sort=True)
    clean_text = clean_syllabus_text(raw_text)

    # На сторінці 4 залишаємо лише текст до початку таблиці.
    if page_number == 4:
        marker = "Discussion"
        if marker in clean_text:
            clean_text = clean_text.split(marker, 1)[0].strip()

    # На сторінці 7 залишаємо лише текст до університетських policies.
    if page_number == 7:
        marker = "University Policies & Processes"
        if marker in clean_text:
            clean_text = clean_text.split(marker, 1)[0].strip()

    return clean_text

In [ ]:
for page_number in [1, 2, 3, 4, 6, 7]:
    page_text = extract_and_trim_page(page_number)

    print("\n" + "=" * 60)
    print(f"PAGE {page_number}")
    print("Characters:", len(page_text))
    print("=" * 60)
    print(page_text[:1000])


PAGE 1
Characters: 2649
Syllabus for Introduction to Health Psychology

Credits: 3
PSYC 1111

Instructor Contact Information:

You can always send your instructor a private message through the Brightspace Messaging system,
accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click
your instructor’s profile page to see all the ways you can communicate with them, including their email
address.

Course Description

Health psychology focuses on the dynamic interaction between biological, social, and psychological factors
that influence physical health and illness, aiming to promote overall well-being and prevent diseases. This
course is designed to provide students with an introduction to the field of health psychology. Students will
learn the crucial role of behavior in shaping health practices and the influence of culture and gender on
health and health outcomes measured through quality of life, health-related quality of life, and life
exp

In [ ]:
normalized_parts = []

# Метадані документа
normalized_parts.append(
    f"""<!-- document_id: {DOCUMENT_METADATA["document_id"]} -->
<!-- source_file: {DOCUMENT_METADATA["source_file"]} -->
<!-- title: {DOCUMENT_METADATA["title"]} -->
<!-- course_id: {DOCUMENT_METADATA["course_id"]} -->
<!-- document_type: {DOCUMENT_METADATA["document_type"]} -->
<!-- content_role: {DOCUMENT_METADATA["content_role"]} -->
<!-- language: {DOCUMENT_METADATA["language"]} -->
<!-- access_level: {DOCUMENT_METADATA["access_level"]} -->

# {DOCUMENT_METADATA["title"]}
"""
)

# Звичайні сторінки до таблиць
for page_number in [1, 2, 3, 4]:
    page_text = extract_and_trim_page(page_number)

    normalized_parts.append(
        f"""<!-- pdf_page: {page_number} -->

{page_text}
"""
    )

# Відновлені таблиці зі сторінок 4–5
normalized_parts.append(grading_tables_markdown)

# Наступні релевантні сторінки
for page_number in [6, 7]:
    page_text = extract_and_trim_page(page_number)

    normalized_parts.append(
        f"""<!-- pdf_page: {page_number} -->

{page_text}
"""
    )

normalized_syllabus = "\n\n".join(normalized_parts).strip()

print(normalized_syllabus[:5000])

<!-- document_id: health_psychology_course_syllabus -->
<!-- source_file: data/raw/course_syllabus.pdf -->
<!-- title: Syllabus for Introduction to Health Psychology -->
<!-- course_id: PSYC_1111 -->
<!-- document_type: syllabus -->
<!-- content_role: course_structure_and_assessment -->
<!-- language: en -->
<!-- access_level: course_use -->

# Syllabus for Introduction to Health Psychology


<!-- pdf_page: 1 -->

Syllabus for Introduction to Health Psychology

Credits: 3
PSYC 1111

Instructor Contact Information:

You can always send your instructor a private message through the Brightspace Messaging system,
accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click
your instructor’s profile page to see all the ways you can communicate with them, including their email
address.

Course Description

Health psychology focuses on the dynamic interaction between biological, social, and psychological factors
that influence physical health and 

In [ ]:
print("Кількість символів:", len(normalized_syllabus))

print("Є grading table:", "### Grading Weights" in normalized_syllabus)
print("Є grading scale:", "### Grading Scale" in normalized_syllabus)
print("Є дивні bullets:", "" in normalized_syllabus or "" in normalized_syllabus)
print("Є policies:", "University Policies & Processes" in normalized_syllabus)

Кількість символів: 14411
Є grading table: True
Є grading scale: True
Є дивні bullets: False
Є policies: False


In [ ]:
OUTPUT_PATH = (
    NORMALIZED_DIR
    / "course_syllabus.md"
)

OUTPUT_PATH.write_text(
    normalized_syllabus,
    encoding="utf-8",
)

print(
    "Файл збережено:",
    OUTPUT_PATH,
)

print(
    "Розмір у символах:",
    len(normalized_syllabus),
)

print(
    "\n".join(
        OUTPUT_PATH
        .read_text(encoding="utf-8")
        .splitlines()[:10]
    )
)

Файл збережено: /content/health-psychology-rag-kb/data/normalized/course_syllabus.md
Розмір у символах: 14411
<!-- document_id: health_psychology_course_syllabus -->
<!-- source_file: data/raw/course_syllabus.pdf -->
<!-- title: Syllabus for Introduction to Health Psychology -->
<!-- course_id: PSYC_1111 -->
<!-- document_type: syllabus -->
<!-- content_role: course_structure_and_assessment -->
<!-- language: en -->
<!-- access_level: course_use -->

# Syllabus for Introduction to Health Psychology


In [ ]:
print(
    OUTPUT_PATH.read_text(
        encoding="utf-8"
    )[:2000]
)

<!-- document_id: health_psychology_course_syllabus -->
<!-- source_file: data/raw/course_syllabus.pdf -->
<!-- title: Syllabus for Introduction to Health Psychology -->
<!-- course_id: PSYC_1111 -->
<!-- document_type: syllabus -->
<!-- content_role: course_structure_and_assessment -->
<!-- language: en -->
<!-- access_level: course_use -->

# Syllabus for Introduction to Health Psychology


<!-- pdf_page: 1 -->

Syllabus for Introduction to Health Psychology

Credits: 3
PSYC 1111

Instructor Contact Information:

You can always send your instructor a private message through the Brightspace Messaging system,
accessible via the envelope (Messages) icon in the top navigation bar. Once logged into your course, click
your instructor’s profile page to see all the ways you can communicate with them, including their email
address.

Course Description

Health psychology focuses on the dynamic interaction between biological, social, and psychological factors
that influence physical health and 

# Document 2: Wright et al. (2019) — 3P-Disease Model

In [ ]:
!pip -q install beautifulsoup4 requests

In [ ]:
import requests
from pathlib import Path

ARTICLE_URL = "https://pmc.ncbi.nlm.nih.gov/articles/PMC6879427/"

response = requests.get(
    ARTICLE_URL,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=30
)

response.raise_for_status()

RAW_HTML_PATH = Path("/content/wright_2019_3p_disease_model.html")
RAW_HTML_PATH.write_text(response.text, encoding="utf-8")

print("HTTP status:", response.status_code)
print("HTML characters:", len(response.text))
print("Saved to:", RAW_HTML_PATH)

HTTP status: 200
HTML characters: 293474
Saved to: /content/wright_2019_3p_disease_model.html


In [ ]:
ARTICLE_TITLE = (
    "A Framework for Understanding the Role of Psychological "
    "Processes in Disease Development, Maintenance, and Treatment: "
    "The 3P-Disease Model"
)

print("Article title found:", ARTICLE_TITLE in response.text)

Article title found: True


In [ ]:
WRIGHT_METADATA = {
    "document_id": "wright_2019_3p_disease_model",
    "source_file": "wright_2019_3p_disease_model.html",
    "source_url": ARTICLE_URL,
    "title": (
        "A Framework for Understanding the Role of Psychological "
        "Processes in Disease Development, Maintenance, and Treatment: "
        "The 3P-Disease Model"
    ),
    "authors": [
        "Casey D. Wright",
        "Alaina G. Tiani",
        "Amber L. Billingsley",
        "Shari A. Steinman",
        "Kevin T. Larkin",
        "Daniel W. McNeil",
    ],
    "publication_year": 2019,
    "journal": "Frontiers in Psychology",
    "doi": "10.3389/fpsyg.2019.02498",
    "pmcid": "PMC6879427",
    "source_type": "html",
    "document_type": "journal_article",
    "content_role": "supplementary_research",
    "domain": "health_psychology",
    "language": "en",
    "access_level": "open_access",
    "license": "CC BY",
}

print("Metadata created:", WRIGHT_METADATA["document_id"])

Metadata created: wright_2019_3p_disease_model


In [ ]:
from bs4 import BeautifulSoup

html = RAW_HTML_PATH.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

print("Page title:")
print(soup.title.get_text(" ", strip=True) if soup.title else "No title")

print("\nHeadings found:")

for heading in soup.find_all(["h1", "h2", "h3", "h4"])[:30]:
    print(
        heading.name,
        "→",
        heading.get_text(" ", strip=True)
    )

Page title:
A Framework for Understanding the Role of Psychological Processes in Disease Development, Maintenance, and Treatment: The 3P-Disease Model - PMC

Headings found:
h2 → PERMALINK
h1 → A Framework for Understanding the Role of Psychological Processes in Disease Development, Maintenance, and Treatment: The 3P-Disease Model
h3 → Casey D Wright
h3 → Alaina G Tiani
h3 → Amber L Billingsley
h3 → Shari A Steinman
h3 → Kevin T Larkin
h3 → Daniel W McNeil
h2 → Abstract
h2 → Introduction
h3 → Insomnia and the 3P Model
h4 → FIGURE 1.
h4 → Predisposing Factors
h4 → Precipitating Factors
h4 → Perpetuating Factors
h2 → The 3P-Disease Model
h3 → Importance of Thoughts, Feelings, and Behavior in Disease
h4 → FIGURE 2.
h3 → Examples of 3P-Health Model Applications to Specific Diseases
h4 → Chronic Pain
h4 → Gastrointestinal Disease and Disorders
h4 → Dental, Oral, and Craniofacial Diseases
h4 → Heart Disease
h2 → Discussion and Implications for the Future
h2 → Conclusion
h2 → Author Contribut

In [ ]:
article = soup.find("article")

print("Article container found:", article is not None)

Article container found: True


In [ ]:
for heading in article.find_all(["h1", "h2", "h3", "h4"])[:30]:
    print(
        heading.name,
        "→",
        heading.get_text(" ", strip=True)
    )

h1 → A Framework for Understanding the Role of Psychological Processes in Disease Development, Maintenance, and Treatment: The 3P-Disease Model
h3 → Casey D Wright
h3 → Alaina G Tiani
h3 → Amber L Billingsley
h3 → Shari A Steinman
h3 → Kevin T Larkin
h3 → Daniel W McNeil
h2 → Abstract
h2 → Introduction
h3 → Insomnia and the 3P Model
h4 → FIGURE 1.
h4 → Predisposing Factors
h4 → Precipitating Factors
h4 → Perpetuating Factors
h2 → The 3P-Disease Model
h3 → Importance of Thoughts, Feelings, and Behavior in Disease
h4 → FIGURE 2.
h3 → Examples of 3P-Health Model Applications to Specific Diseases
h4 → Chronic Pain
h4 → Gastrointestinal Disease and Disorders
h4 → Dental, Oral, and Craniofacial Diseases
h4 → Heart Disease
h2 → Discussion and Implications for the Future
h2 → Conclusion
h2 → Author Contributions
h2 → Conflict of Interest
h2 → Acknowledgments
h2 → Footnotes
h2 → References


In [ ]:
from urllib.parse import urljoin
from bs4.element import Tag, NavigableString

BASE_URL = ARTICLE_URL


def inline_html_to_markdown(element: Tag) -> str:
    """
    Convert inline HTML content to Markdown while preserving links.
    """

    parts = []

    for child in element.children:
        if isinstance(child, NavigableString):
            parts.append(str(child))

        elif isinstance(child, Tag):
            text = child.get_text(" ", strip=True)

            if child.name == "a":
                href = child.get("href")

                if href and text:
                    absolute_url = urljoin(BASE_URL, href)
                    parts.append(f"[{text}]({absolute_url})")
                else:
                    parts.append(text)

            elif child.name in {"strong", "b"}:
                parts.append(f"**{text}**")

            elif child.name in {"em", "i"}:
                parts.append(f"*{text}*")

            elif child.name in {"sup", "sub"}:
                parts.append(text)

            else:
                parts.append(
                    inline_html_to_markdown(child)
                    if list(child.children)
                    else text
                )

    result = "".join(parts)
    result = " ".join(result.split())

    return result.strip()

In [ ]:
first_linked_paragraph = article.find(
    lambda tag: (
        tag.name == "p"
        and tag.find("a") is not None
    )
)

if first_linked_paragraph:
    print("PLAIN TEXT:")
    print(first_linked_paragraph.get_text(" ", strip=True))

    print("\nMARKDOWN WITH LINKS:")
    print(inline_html_to_markdown(first_linked_paragraph))
else:
    print("Paragraph with link not found")

PLAIN TEXT:
Scientific and technological advances in 1969 led to “giant leaps for mankind,” including the publication of a paper applying psychological principles to health services ( Schofield, 1969 ). Over the next several years, the American Psychological Association (APA) established a task force to explicate the role of behavior and psychology in health-related processes, systems, and diseases ( APA Task Force on Health Research, 1976 ). As a result, health psychology and behavioral medicine were established as unique fields of study ( Wallston, 1996 ). Contemporary uses often use the terms health psychology and behavioral medicine interchangeably.

MARKDOWN WITH LINKS:
Scientific and technological advances in 1969 led to “giant leaps for mankind,” including the publication of a paper applying psychological principles to health services ([Schofield, 1969](https://pmc.ncbi.nlm.nih.gov/articles/PMC6879427/#B99)). Over the next several years, the American Psychological Association (A

In [ ]:
import re


def is_noise_text(text: str) -> bool:
    text = text.strip()

    if text.casefold() == "open in a new tab":
        return True

    if re.fullmatch(
        r"FIGURE\s+\d+\.?",
        text,
        flags=re.IGNORECASE
    ):
        return True

    return False


def clean_figure_caption(text: str) -> str:
    text = text.strip()

    text = re.sub(
        r"\bOpen in a new tab\b",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^FIGURE\s+\d+\.?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
from bs4.element import Tag


FIGURE_ASSETS = {
    "FIGURE 1.": {
        "title": "Figure 1",
        "alt": "The 3P-Disease Model",
        "path": "assets/wright_2019_figure_1.jpeg",
    },
    "FIGURE 2.": {
        "title": "Figure 2",
        "alt": (
            "Summary of potential predisposing, precipitating, "
            "and perpetuating factors across biopsychosocial domains"
        ),
        "path": "assets/wright_2019_figure_2.jpeg",
    },
}


def article_to_markdown(article: Tag, expected_title: str) -> str:
    """
    Convert the retrieval-relevant PMC article content
    to normalized Markdown.

    Preserves headings, paragraphs, lists, citation links,
    figures, and figure captions.

    Excludes article back matter such as author contributions,
    conflict statements, acknowledgments, footnotes, and references.
    """

    allowed_tags = {
        "h1",
        "h2",
        "h3",
        "h4",
        "p",
        "li",
        "figcaption",
    }

    parts = []
    started = False
    pending_figure = None

    for element in article.find_all(
        list(allowed_tags),
        recursive=True
    ):
        if not isinstance(element, Tag):
            continue

        plain_text = element.get_text(" ", strip=True)

        excluded_back_matter_sections = {
            "Author Contributions",
            "Conflict of Interest",
            "Acknowledgments",
            "Footnotes",
            "References",
        }

        if (
            element.name == "h2"
            and plain_text in excluded_back_matter_sections
        ):
            break

        if not started:
            if (
                element.name == "h1"
                and plain_text == expected_title
            ):
                started = True
            else:
                continue

        # Figure heading: preserve it semantically,
        # but replace the duplicated PMC label with clean Markdown.
        figure_key = plain_text.upper()

        if element.name == "h4" and figure_key in FIGURE_ASSETS:
            pending_figure = FIGURE_ASSETS[figure_key]

            parts.append(f"#### {pending_figure['title']}")
            parts.append(
                f"![{pending_figure['alt']}]"
                f"({pending_figure['path']})"
            )
            continue

        if is_noise_text(plain_text):
            continue

        # Avoid nested duplicates such as p inside li or figcaption.
        parent = element.parent
        nested_inside_selected_block = False

        while parent is not None and parent is not article:
            if (
                isinstance(parent, Tag)
                and parent.name in allowed_tags
            ):
                nested_inside_selected_block = True
                break

            parent = parent.parent

        if nested_inside_selected_block:
            continue

        text = inline_html_to_markdown(element)

        if not text:
            continue

        if element.name == "h1":
            parts.append(f"# {text}")

        elif element.name == "h2":
            parts.append(f"## {text}")

        elif element.name == "h3":
            parts.append(f"### {text}")

        elif element.name == "h4":
            parts.append(f"#### {text}")

        elif element.name == "li":
            parts.append(f"- {text}")

        elif element.name == "figcaption":
            caption_text = clean_figure_caption(text)

            if caption_text:
                parts.append(
                    f"**Figure caption:** {caption_text}"
                )

            pending_figure = None

        elif element.name == "p":
            parts.append(text)

    return "\n\n".join(parts).strip()

In [ ]:
wright_article_markdown = article_to_markdown(
    article=article,
    expected_title=WRIGHT_METADATA["title"]
)

print("Characters:", len(wright_article_markdown))
print(wright_article_markdown[:5000])

Characters: 71436
# A Framework for Understanding the Role of Psychological Processes in Disease Development, Maintenance, and Treatment: The 3P-Disease Model

### Casey D Wright

### Alaina G Tiani

### Amber L Billingsley

### Shari A Steinman

### Kevin T Larkin

### Daniel W McNeil

- Author information

- Article notes

- Copyright and License information

Edited by: Silvia Serino, Lausanne University Hospital (CHUV), Switzerland

Reviewed by: Anna Sedda, Heriot-Watt University, United Kingdom; Paola Cardinali, University of Genoa, Italy

*Correspondence: Casey D. Wright, cdw0022@mix.wvu.edu

Daniel W. McNeil, dmcneil@wvu.edu

This article was submitted to Psychology for Clinical Settings, a section of the journal Frontiers in Psychology

Received 2019 Mar 12; Accepted 2019 Oct 22; Collection date 2019.

This is an open-access article distributed under the terms of the Creative Commons Attribution License (CC BY). The use, distribution or reproduction in other forums is permitted,

In [ ]:
from collections import Counter

blocks = [
    block.strip()
    for block in wright_article_markdown.split("\n\n")
    if block.strip()
]

block_counts = Counter(blocks)

duplicates = [
    (count, block)
    for block, count in block_counts.items()
    if count > 1 and len(block) > 80
]

duplicates.sort(reverse=True)

print("Total blocks:", len(blocks))
print("Unique blocks:", len(block_counts))
print("Repeated substantial blocks:", len(duplicates))

for count, block in duplicates[:20]:
    print("\n" + "=" * 70)
    print("Repeated:", count, "times")
    print(block[:500])

Total blocks: 103
Unique blocks: 103
Repeated substantial blocks: 0


In [ ]:
section_checks = [
    "# A Framework for Understanding",
    "## Abstract",
    "## Introduction",
    "## The 3P-Disease Model",
    "## Discussion and Implications for the Future",
    "## Conclusion"
]

for section in section_checks:
    print(
        section,
        "→",
        wright_article_markdown.count(section)
    )

# A Framework for Understanding → 1
## Abstract → 1
## Introduction → 1
## The 3P-Disease Model → 1
## Discussion and Implications for the Future → 1
## Conclusion → 1


In [ ]:
test_phrase = "Scientific and technological advances in 1969"

print(
    "Introduction paragraph occurrences:",
    wright_article_markdown.count(test_phrase)
)

Introduction paragraph occurrences: 1


In [ ]:
print(wright_article_markdown[-7000:])

cope. These are important aspects of disease development for therapists to consider.

An additional issue to consider is how to use this model to explain sudden heart attacks or other health concerns in individuals who do engage in adaptive health behaviors by eating nutritious foods and regularly exercising. For these individuals, perhaps their genetic predisposition is what determines their ultimate outcome.

The 3P model advances the conceptualization of CAD as more than just a biological disease resulting from poor genetic luck or from purely internal, physiological factors. The model considers both internal factors (e.g., genetic) and external factors (e.g., life events, stressors, and behaviors) that all culminate and contribute to one’s overall risk of disease development. While it is important to attend check-ups and monitor blood pressure, cholesterol levels, and the like, it is just as important to consider the breadth of precipitating and perpetuating factors that psychologi

In [ ]:
tables = article.find_all("table")

print("Tables found:", len(tables))

Tables found: 0


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving michie_2011_figure_1.jpeg to michie_2011_figure_1.jpeg
Saving michie_2011_figure_2.jpeg to michie_2011_figure_2.jpeg


In [ ]:
import re


def is_noise_text(text: str) -> bool:
    """
    Detect PMC interface text and duplicate figure labels
    that should not appear in normalized Markdown.
    """

    text = text.strip()

    if text == "Open in a new tab":
        return True

    if re.fullmatch(
        r"FIGURE\s+\d+\.?",
        text,
        flags=re.IGNORECASE
    ):
        return True

    return False

In [ ]:
test_values = [
    "Open in a new tab",
    "FIGURE 1.",
    "FIGURE 1",
    "FIGURE 2.",
    "The 3P-Disease Model.",
]

for value in test_values:
    print(
        repr(value),
        "→",
        is_noise_text(value)
    )

'Open in a new tab' → True
'FIGURE 1.' → True
'FIGURE 1' → True
'FIGURE 2.' → True
'The 3P-Disease Model.' → False


In [ ]:
wright_article_markdown = article_to_markdown(
    article=article,
    expected_title=WRIGHT_METADATA["title"]
)

print("Characters:", len(wright_article_markdown))

Characters: 71436


In [ ]:
final_checks = {
    "title_once": (
        wright_article_markdown.count(
            "# " + WRIGHT_METADATA["title"]
        ) == 1
    ),
    "abstract_present": (
        "## Abstract" in wright_article_markdown
    ),
    "back_matter_removed": all(
    heading not in wright_article_markdown
    for heading in [
        "## Author Contributions",
        "## Conflict of Interest",
        "## Acknowledgments",
        "## Footnotes",
        "## References",
    ]
    ),
    "open_in_new_tab_removed": (
        "Open in a new tab"
        not in wright_article_markdown
    ),
    "figure_1_duplicate_removed": (
        "FIGURE 1." not in wright_article_markdown
    ),
    "figure_2_duplicate_removed": (
        "FIGURE 2." not in wright_article_markdown
    ),
    "figure_1_link_present": (
        "assets/wright_2019_figure_1.jpeg"
        in wright_article_markdown
    ),
    "figure_2_link_present": (
        "assets/wright_2019_figure_2.jpeg"
        in wright_article_markdown
    ),
    "figure_captions_present": (
        wright_article_markdown.count(
            "**Figure caption:**"
        ) == 2
    ),
    "citation_links_present": (
        "https://pmc.ncbi.nlm.nih.gov/articles/"
        "PMC6879427/#B99"
        in wright_article_markdown
    ),
}

for check_name, passed in final_checks.items():
    print(check_name, "→", passed)

title_once → True
abstract_present → True
back_matter_removed → True
open_in_new_tab_removed → True
figure_1_duplicate_removed → True
figure_2_duplicate_removed → True
figure_1_link_present → True
figure_2_link_present → True
figure_captions_present → True
citation_links_present → True


In [ ]:
from google.colab import files

files.download(
    "/content/wright_2019_3p_disease_model.md"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Document 3: Michie et al. (2011) — Behaviour Change Wheel

In [ ]:
MICHIE_ARTICLE_TITLE = (
    "The behaviour change wheel: "
    "A new method for characterising and designing behaviour change interventions"
)

In [ ]:
MICHIE_RAW_HTML_PATH = (
    RAW_DIR
    / "michie_2011_behaviour_change_wheel.html"
)

assert MICHIE_RAW_HTML_PATH.exists(), (
    f"Missing file: {MICHIE_RAW_HTML_PATH}"
)

michie_html = MICHIE_RAW_HTML_PATH.read_text(
    encoding="utf-8"
)

print("Exists:", MICHIE_RAW_HTML_PATH.exists())
print("Size:", MICHIE_RAW_HTML_PATH.stat().st_size)
print("HTML characters:", len(michie_html))
print(
    "Article title found:",
    MICHIE_ARTICLE_TITLE in michie_html
)

Exists: True
Size: 199594
HTML characters: 199418
Article title found: True


In [ ]:
from bs4 import BeautifulSoup

michie_soup = BeautifulSoup(
    michie_html,
    "html.parser"
)

michie_article = michie_soup.find("article")

print(
    "Article container found:",
    michie_article is not None
)

Article container found: True


In [ ]:
print("HEADINGS")

for heading in michie_article.find_all(
    ["h1", "h2", "h3", "h4"]
)[:60]:
    print(
        heading.name,
        "→",
        heading.get_text(" ", strip=True)
    )

print("\nFIGURES:", len(michie_article.find_all("figure")))
print("TABLES:", len(michie_article.find_all("table")))

HEADINGS
h1 → The behaviour change wheel: A new method for characterising and designing behaviour change interventions
h3 → Susan Michie
h3 → Maartje M van Stralen
h3 → Robert West
h2 → Abstract
h3 → Background
h3 → Methods
h3 → Results
h3 → Conclusions
h2 → Background
h2 → Methods
h3 → Establishing criteria of usefulness
h4 → Figure 1.
h3 → Systematic literature review of current frameworks
h3 → Develop a new framework
h3 → Test the reliability of the framework
h2 → Results
h3 → Systematic literature review of existing frameworks
h4 → Table 1.
h3 → Development of a new framework
h4 → Figure 2.
h4 → Table 2.
h4 → Table 3.
h3 → Testing the reliability of the new framework
h2 → Discussion
h2 → Competing interests
h2 → Authors' contributions
h2 → Supplementary Material
h2 → Contributor Information
h2 → Acknowledgements
h2 → References
h2 → Associated Data
h3 → Supplementary Materials

FIGURES: 2
TABLES: 3


In [ ]:
MICHIE_ARTICLE_URL = (
    "https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/"
)

MICHIE_ARTICLE_TITLE = (
    "The behaviour change wheel: "
    "A new method for characterising and designing "
    "behaviour change interventions"
)

MICHIE_METADATA = {
    "document_id": "michie_2011_behaviour_change_wheel",
    "source_file": "michie_2011_behaviour_change_wheel.html",
    "source_path": (
        "data/raw/"
        "michie_2011_behaviour_change_wheel.html"
    ),
    "source_url": MICHIE_ARTICLE_URL,
    "title": MICHIE_ARTICLE_TITLE,
    "authors": [
        "Susan Michie",
        "Maartje M. van Stralen",
        "Robert West",
    ],
    "publication_year": 2011,
    "journal": "Implementation Science",
    "volume": 6,
    "article_number": 42,
    "doi": "10.1186/1748-5908-6-42",
    "pmcid": "PMC3096582",
    "pmid": "21513547",
    "source_type": "html",
    "document_type": "journal_article",
    "content_role": "core_behaviour_change_framework",
    "domain": "health_psychology",
    "language": "en",
    "access_level": "open_access",
    "license": "CC BY 2.0",
    "figure_count": 2,
    "table_count": 3,
}

In [ ]:
import json

print(
    json.dumps(
        MICHIE_METADATA,
        indent=2,
        ensure_ascii=False
    )
)

{
  "document_id": "michie_2011_behaviour_change_wheel",
  "source_file": "michie_2011_behaviour_change_wheel.html",
  "source_path": "data/raw/michie_2011_behaviour_change_wheel.html",
  "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/",
  "title": "The behaviour change wheel: A new method for characterising and designing behaviour change interventions",
  "authors": [
    "Susan Michie",
    "Maartje M. van Stralen",
    "Robert West"
  ],
  "publication_year": 2011,
  "journal": "Implementation Science",
  "volume": 6,
  "article_number": 42,
  "doi": "10.1186/1748-5908-6-42",
  "pmcid": "PMC3096582",
  "pmid": "21513547",
  "source_type": "html",
  "document_type": "journal_article",
  "content_role": "core_behaviour_change_framework",
  "domain": "health_psychology",
  "language": "en",
  "access_level": "open_access",
  "license": "CC BY 2.0",
  "figure_count": 2,
  "table_count": 3
}


In [ ]:
figures = michie_article.find_all("figure")

for index, figure in enumerate(figures, start=1):
    image = figure.find("img")
    caption = figure.find("figcaption")

    print(f"FIGURE {index}")
    print("src:", image.get("src") if image else None)
    print(
        "caption:",
        caption.get_text(" ", strip=True)
        if caption
        else None
    )
    print()

FIGURE 1
src: https://cdn.ncbi.nlm.nih.gov/pmc/blobs/73d6/3096582/3c4529e4ae2b/1748-5908-6-42-1.jpg
caption: The COM-B system - a framework for understanding behaviour .

FIGURE 2
src: https://cdn.ncbi.nlm.nih.gov/pmc/blobs/73d6/3096582/ac7b6688eaa3/1748-5908-6-42-2.jpg
caption: The Behaviour Change Wheel .



In [ ]:
import requests


MICHIE_ASSETS_DIR = (
    PROJECT_ROOT
    / "data"
    / "normalized"
    / "assets"
)

MICHIE_ASSETS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


michie_figure_urls = [
    "https://cdn.ncbi.nlm.nih.gov/pmc/blobs/73d6/3096582/3c4529e4ae2b/1748-5908-6-42-1.jpg",
    "https://cdn.ncbi.nlm.nih.gov/pmc/blobs/73d6/3096582/ac7b6688eaa3/1748-5908-6-42-2.jpg",
]


for index, url in enumerate(
    michie_figure_urls,
    start=1,
):
    output_path = (
        MICHIE_ASSETS_DIR
        / f"michie_2011_figure_{index}.jpeg"
    )

    response = requests.get(
        url,
        timeout=30,
    )

    response.raise_for_status()

    output_path.write_bytes(
        response.content
    )

    print(
        "Saved:",
        output_path,
        "bytes:",
        output_path.stat().st_size,
    )

Saved: /content/health-psychology-rag-kb/data/normalized/assets/michie_2011_figure_1.jpeg bytes: 30018
Saved: /content/health-psychology-rag-kb/data/normalized/assets/michie_2011_figure_2.jpeg bytes: 61027


In [ ]:
MICHIE_ASSETS_DIR = ASSETS_DIR

for expected_name in [
    "michie_2011_figure_1.jpeg",
    "michie_2011_figure_2.jpeg",
]:
    path = MICHIE_ASSETS_DIR / expected_name

    print(
        expected_name,
        "exists:",
        path.exists(),
        "size:",
        path.stat().st_size
        if path.exists()
        else 0,
    )

michie_2011_figure_1.jpeg exists: True size: 30018
michie_2011_figure_2.jpeg exists: True size: 61027


In [ ]:
tables = michie_article.find_all("table")

print("TABLES FOUND:", len(tables))

for index, table in enumerate(tables, start=1):
    parent = table.find_parent(
        ["section", "figure"]
    )

    caption = None

    if parent:
        caption_block = parent.find(
            class_="caption"
        )

        if caption_block:
            caption = caption_block.get_text(
                " ",
                strip=True
            )

    rows = table.find_all("tr")

    print(f"\nTABLE {index}")
    print("Caption:", caption)
    print("Rows:", len(rows))

    first_rows = rows[:3]

    for row_index, row in enumerate(
        first_rows,
        start=1
    ):
        cells = row.find_all(
            ["th", "td"]
        )

        values = [
            cell.get_text(
                " ",
                strip=True
            )
            for cell in cells
        ]

        print(
            f"Row {row_index}:",
            values
        )

TABLES FOUND: 3

TABLE 1
Caption: Definitions of interventions and policies
Rows: 34
Row 1: ['Interventions', 'Definition', 'Examples']
Row 2: ['Education', 'Increasing knowledge or understanding', 'Providing information to promote healthy eating']
Row 3: ['']

TABLE 2
Caption: Links between the components of the 'COM-B' model of behaviour and the intervention functions
Rows: 12
Row 1: ['Model of behaviour: sources', 'Educa-tion', 'Persua-sion', 'Incentiv-isation', 'Coercion', 'Training', 'Restric-tion', 'Environ-mental restructuring', 'Model-ling', 'Enable-ment']
Row 2: ['C-Ph', '', '', '', '', '√', '', '', '', '√']
Row 3: ['']

TABLE 3
Caption: Links between policy categories and intervention functions
Rows: 14
Row 1: ['', 'Educat-ion', 'Persuas-ion', 'Incent-ivisation', 'Coerc-ion', 'Training', 'Restrict-ion', 'Environ-mental restructuring', 'Model-ling', 'Enable-ment']
Row 2: ['Communication/Marketing', '√', '√', '√', '√', '', '', '', '√', '']
Row 3: ['']


In [ ]:
import re

TABLE_WORD_FIXES = {
    "Educa-tion": "Education",
    "Educat-ion": "Education",
    "Persua-sion": "Persuasion",
    "Persuas-ion": "Persuasion",
    "Incentiv-isation": "Incentivisation",
    "Incent-ivisation": "Incentivisation",
    "Coerc-ion": "Coercion",
    "Restric-tion": "Restriction",
    "Restrict-ion": "Restriction",
    "Environ-mental": "Environmental",
    "Model-ling": "Modelling",
    "Enable-ment": "Enablement",
}


def clean_table_text(text: str) -> str:
    text = text.strip()

    # Виправляємо штучно розірвані слова в заголовках таблиць
    for broken_word, corrected_word in TABLE_WORD_FIXES.items():
        text = text.replace(
            broken_word,
            corrected_word
        )

    # Нормалізуємо зайві пробіли
    text = re.sub(r"\s+", " ", text)

    # Екрануємо вертикальну риску,
    # щоб вона не ламала Markdown-таблицю
    text = text.replace("|", r"\|")

    return text.strip()


def html_table_to_markdown(table) -> str:
    rows = []

    for row in table.find_all("tr"):
        cells = row.find_all(["th", "td"])

        if not cells:
            continue

        values = [
            clean_table_text(
                cell.get_text(" ", strip=True)
            )
            for cell in cells
        ]

        # Пропускаємо службові separator-рядки:
        # <tr><td colspan="..."><hr></td></tr>
        if not any(values):
            continue

        rows.append(values)

    if not rows:
        return ""

    column_count = max(len(row) for row in rows)

    normalized_rows = [
        row + [""] * (column_count - len(row))
        for row in rows
    ]

    header = normalized_rows[0]
    body = normalized_rows[1:]

    # У Table 3 перша клітинка заголовка порожня.
    # Додаємо змістовну назву колонки.
    if header and not header[0]:
        header[0] = "Policy category"

    markdown_lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(["---"] * column_count) + " |",
    ]

    for row in body:
        markdown_lines.append(
            "| " + " | ".join(row) + " |"
        )

    return "\n".join(markdown_lines)

In [ ]:
michie_markdown_tables = {}

for index, table in enumerate(tables, start=1):
    parent = table.find_parent(
        ["section", "figure"]
    )

    caption = None

    if parent:
        caption_block = parent.find(
            class_="caption"
        )

        if caption_block:
            caption = clean_table_text(
                caption_block.get_text(
                    " ",
                    strip=True
                )
            )

    table_markdown = html_table_to_markdown(
        table
    )

    michie_markdown_tables[index] = {
        "caption": caption,
        "markdown": table_markdown,
    }

    print(f"\n{'=' * 80}")
    print(f"TABLE {index}")
    print("Caption:", caption)
    print(f"{'=' * 80}\n")
    print(table_markdown[:3000])


TABLE 1
Caption: Definitions of interventions and policies

| Interventions | Definition | Examples |
| --- | --- | --- |
| Education | Increasing knowledge or understanding | Providing information to promote healthy eating |
| Persuasion | Using communication to induce positive or negative feelings or stimulate action | Using imagery to motivate increases in physical activity |
| Incentivisation | Creating expectation of reward | Using prize draws to induce attempts to stop smoking |
| Coercion | Creating expectation of punishment or cost | Raising the financial cost to reduce excessive alcohol consumption |
| Training | Imparting skills | Advanced driver training to increase safe driving |
| Restriction | Using rules to reduce the opportunity to engage in the target behaviour (or to increase the target behaviour by reducing the opportunity to engage in competing behaviours) | Prohibiting sales of solvents to people under 18 to reduce use for intoxication |
| Environmental restructur

In [ ]:
michie_table_notes = {}

for index, table in enumerate(tables, start=1):
    parent_section = table.find_parent(
        "section",
        class_="tw"
    )

    notes = []

    if parent_section:
        footnote_block = parent_section.find(
            class_="tw-foot"
        )

        if footnote_block:
            for paragraph in footnote_block.find_all("p"):
                note_text = clean_table_text(
                    paragraph.get_text(
                        " ",
                        strip=True
                    )
                )

                if note_text:
                    notes.append(note_text)

    michie_table_notes[index] = notes

    print(f"\nTABLE {index} NOTES:")

    if notes:
        for note in notes:
            print("-", note)
    else:
        print("No notes")


TABLE 1 NOTES:
- 1 Capability beyond education and training; opportunity beyond environmental restructuring

TABLE 2 NOTES:
- 1. Physical capability can be achieved through physical skill development which is the focus of training or potentially through enabling interventions such as medication, surgery or prostheses
- 2. Psychological capability can be achieved through imparting knowledge or understanding, training emotional, cognitive and/or behavioural skills or through enabling interventions such as medication
- 3. Reflective motivation can be achieved through increasing knowledge and understanding, eliciting positive (or negative) feelings about behavioural target
- 4. Automatic motivation can be achieved through associative learning that elicit positive (or negative) feelings and impulses and counter-impulses relating to the behavioural target, imitative learning, habit formation or direct influences on automatic motivational processes ( e.g ., via medication)
- 5. Physical and 

In [ ]:
from urllib.parse import urljoin
from bs4.element import Tag, NavigableString
import re

MICHIE_BASE_URL = MICHIE_ARTICLE_URL


def michie_inline_to_markdown(element: Tag) -> str:
    parts = []

    for child in element.children:
        if isinstance(child, NavigableString):
            parts.append(str(child))

        elif isinstance(child, Tag):
            text = child.get_text(" ", strip=True)

            if child.name == "a":
                href = child.get("href")

                if href and text:
                    absolute_url = urljoin(
                        MICHIE_BASE_URL,
                        href
                    )
                    parts.append(
                        f"[{text}]({absolute_url})"
                    )
                else:
                    parts.append(text)

            elif child.name in {"strong", "b"}:
                parts.append(f"**{text}**")

            elif child.name in {"em", "i"}:
                parts.append(f"*{text}*")

            elif child.name in {"sup", "sub"}:
                parts.append(text)

            else:
                nested = michie_inline_to_markdown(child)

                if nested:
                    parts.append(nested)
                else:
                    parts.append(text)

        result = "".join(parts)
        result = re.sub(r"\s+([.,;:!?])", r"\1", result)

    # Прибираємо зайві зовнішні дужки навколо citation links
    previous_result = None

    while previous_result != result:
        previous_result = result

        result = re.sub(
            r"\[\s*(\[[0-9]+\]\([^)]+\)"
            r"(?:\s*,\s*\[[0-9]+\]\([^)]+\))*)\s*\]",
            r"\1",
            result
        )

    # Виправляємо залишковий випадок:
    # [[1](url) → [1](url)
    result = re.sub(
        r"\[\[(?=\d+\]\()",
        "[",
        result
    )

    return result.strip()

    return result.strip()

In [ ]:
MICHIE_FIGURE_ASSETS = {
    "F1": {
        "title": "Figure 1",
        "alt": "The COM-B system - a framework for understanding behaviour",
        "path": "assets/michie_2011_figure_1.jpeg",
    },
    "F2": {
        "title": "Figure 2",
        "alt": "The Behaviour Change Wheel",
        "path": "assets/michie_2011_figure_2.jpeg",
    },
}


MICHIE_TABLE_URLS = {
    1: "https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/table/T1/",
    2: "https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/table/T2/",
    3: "https://pmc.ncbi.nlm.nih.gov/articles/PMC3096582/table/T3/",
}

In [ ]:
def metadata_to_html_comments(
    metadata: dict,
) -> list[str]:
    comments = []

    for key, value in metadata.items():
        if isinstance(value, list):
            value = "; ".join(
                str(item)
                for item in value
            )

        comments.append(
            f"<!-- {key}: {value} -->"
        )

    comments.append("")

    return comments

In [ ]:
def michie_article_to_markdown(
    article: Tag,
) -> str:
    lines = []

    table_counter = 0

    # Основна назва статті
    title = article.find("h1")

    if title:
        lines.append(
            "# " + title.get_text(
                " ",
                strip=True,
            )
        )
        lines.append("")

    # Provenance metadata залишається
    # у normalized Markdown як HTML-коментарі,
    # але не утворює retrieval section.
    lines.extend(
        metadata_to_html_comments(
            MICHIE_METADATA
        )
    )

    # Беремо тільки body статті
    article_body = article.find(
        "section",
        class_="main-article-body",
    )

    if article_body is None:
        raise ValueError(
            "Main article body was not found."
        )

    for element in article_body.children:
        if not isinstance(element, Tag):
            continue

        # Основні секції обробляємо рекурсивно
        if element.name != "section":
            continue

        section_id = element.get(
            "id",
            "",
        )

        # Не додаємо дубль supplementary materials
        if section_id == "_ad93_":
            continue

        for node in element.descendants:
            if not isinstance(node, Tag):
                continue

            # Не обробляємо вкладені елементи повторно,
            # якщо вони належать figure.
            if node.find_parent("figure"):
                continue

            # Не дублюємо внутрішній текст таблиць,
            # captions і приміток.
            if (
                node.find_parent(
                    "section",
                    class_="tw",
                )
                and node.name != "table"
            ):
                continue

            if node.name == "h2":
                text = node.get_text(
                    " ",
                    strip=True,
                )

                # References додаємо окремо нижче.
                if (
                    text.casefold()
                    == "references"
                ):
                    continue

                if text:
                    lines.append(
                        f"## {text}"
                    )
                    lines.append("")

            elif node.name == "h3":
                text = node.get_text(
                    " ",
                    strip=True,
                )

                if text:
                    lines.append(
                        f"### {text}"
                    )
                    lines.append("")

            elif node.name == "p":
                # Не дублюємо captions,
                # table captions і table notes.
                if node.find_parent(
                    "figcaption"
                ):
                    continue

                if node.find_parent(
                    class_="caption"
                ):
                    continue

                if node.find_parent(
                    class_="tw-foot"
                ):
                    continue

                text = (
                    michie_inline_to_markdown(
                        node
                    )
                )

                if (
                    text
                    and text.casefold()
                    != "open in a new tab"
                ):
                    lines.append(text)
                    lines.append("")

            elif node.name == "figure":
                figure_id = node.get("id")

                asset = (
                    MICHIE_FIGURE_ASSETS.get(
                        figure_id
                    )
                )

                if not asset:
                    continue

                caption_tag = node.find(
                    "figcaption"
                )

                caption = (
                    caption_tag.get_text(
                        " ",
                        strip=True,
                    )
                    if caption_tag
                    else ""
                )

                caption = re.sub(
                    r"\s+([.,;:!?])",
                    r"\1",
                    caption,
                )

                lines.append(
                    f"#### {asset['title']}"
                )
                lines.append("")

                lines.append(
                    f"![{asset['alt']}]"
                    f"({asset['path']})"
                )
                lines.append("")

                if caption:
                    lines.append(
                        "**Figure caption:** "
                        f"{caption}"
                    )
                    lines.append("")

            elif node.name == "table":
                table_counter += 1

                table_data = (
                    michie_markdown_tables[
                        table_counter
                    ]
                )

                caption = (
                    table_data["caption"]
                )

                markdown = (
                    table_data["markdown"]
                )

                table_heading = (
                    f"#### Table {table_counter}"
                )

                if caption:
                    table_heading += (
                        f". {caption}"
                    )

                lines.append(
                    table_heading
                )
                lines.append("")

                lines.append(markdown)
                lines.append("")

                notes = (
                    michie_table_notes.get(
                        table_counter,
                        [],
                    )
                )

                if notes:
                    lines.append(
                        "**Table notes:**"
                    )
                    lines.append("")

                    for note in notes:
                        lines.append(
                            f"- {note}"
                        )

                    lines.append("")

                table_url = (
                    MICHIE_TABLE_URLS.get(
                        table_counter
                    )
                )

                if table_url:
                    lines.append(
                        "[Open original HTML table]"
                        f"({table_url})"
                    )
                    lines.append("")

    # References додаємо до full Markdown,
    # а retrieval cleanup видалить їх пізніше.
    references = article.find(
        "section",
        class_="ref-list",
    )

    if references:
        lines.append("## References")
        lines.append("")

        reference_items = (
            references.find_all(
                "li",
                recursive=True,
            )
        )

        for index, item in enumerate(
            reference_items,
            start=1,
        ):
            text = (
                michie_inline_to_markdown(
                    item
                )
            )

            if text:
                lines.append(
                    f"{index}. {text}"
                )

        lines.append("")

    markdown = "\n".join(lines)

    # Прибираємо зайві порожні рядки.
    markdown = re.sub(
        r"\n{3,}",
        "\n\n",
        markdown,
    )

    return markdown.strip() + "\n"

In [ ]:
MICHIE_EXCLUDED_SECTIONS = {
    "competing interests",
    "authors' contributions",
    "author contributions",
    "contributor information",
    "acknowledgements",
    "acknowledgments",
    "supplementary material",
    "associated data",
    "references",
}


def clean_michie_retrieval_markdown(
    markdown_text: str,
) -> str:
    """
    Remove article back matter and standalone table-source links
    from the Michie retrieval-ready Markdown.
    """

    cleaned_lines = []
    skip_current_section = False

    for line in markdown_text.splitlines():
        stripped = line.strip()

        # New level-2 section.
        if stripped.startswith("## "):
            section_title = (
                stripped[3:]
                .strip()
                .casefold()
            )

            skip_current_section = (
                section_title
                in MICHIE_EXCLUDED_SECTIONS
            )

            if skip_current_section:
                continue

        # Skip all content belonging to an excluded H2 section.
        if skip_current_section:
            continue

        # Do not create standalone chunks containing only
        # links to original HTML tables.
        if re.fullmatch(
            r"\[Open original HTML table\]\([^)]+\)",
            stripped,
        ):
            continue

        cleaned_lines.append(line)

    cleaned_text = "\n".join(
        cleaned_lines
    )

    # Normalize excessive blank lines.
    cleaned_text = re.sub(
        r"\n{3,}",
        "\n\n",
        cleaned_text,
    )

    return cleaned_text.strip() + "\n"

In [ ]:
michie_full_markdown = (
    michie_article_to_markdown(
        michie_article
    )
)

michie_article_markdown = (
    clean_michie_retrieval_markdown(
        michie_full_markdown
    )
)

print(
    "Full Markdown characters:",
    len(michie_full_markdown)
)

print(
    "Clean retrieval Markdown characters:",
    len(michie_article_markdown)
)

Full Markdown characters: 74722
Clean retrieval Markdown characters: 49087


In [ ]:
assert "## Document metadata" not in michie_article_markdown

assert (
    "<!-- document_id: "
    "michie_2011_behaviour_change_wheel -->"
    in michie_article_markdown
)

assert (
    "<!-- doi: 10.1186/1748-5908-6-42 -->"
    in michie_article_markdown
)

assert (
    "## References"
    not in michie_article_markdown
)

assert (
    "[Open original HTML table]"
    not in michie_article_markdown
)

print(
    "Michie metadata cleanup passed"
)

Michie metadata cleanup passed


In [ ]:
for line in michie_article_markdown.splitlines():
    if line.startswith("#"):
        print(line)

# The behaviour change wheel: A new method for characterising and designing behaviour change interventions
## Abstract
### Background
### Methods
### Results
### Conclusions
## Background
## Methods
### Establishing criteria of usefulness
#### Figure 1
### Systematic literature review of current frameworks
### Develop a new framework
### Test the reliability of the framework
## Results
### Systematic literature review of existing frameworks
#### Table 1. Definitions of interventions and policies
### Development of a new framework
#### Figure 2
#### Table 2. Links between the components of the 'COM-B' model of behaviour and the intervention functions
#### Table 3. Links between policy categories and intervention functions
### Testing the reliability of the new framework
## Discussion


In [ ]:
MICHIE_NORMALIZED_PATH = (
    NORMALIZED_DIR
    / "michie_2011_behaviour_change_wheel.md"
)

MICHIE_NORMALIZED_PATH.write_text(
    michie_article_markdown,
    encoding="utf-8",
)

print("Saved:", MICHIE_NORMALIZED_PATH)
print("Size:", MICHIE_NORMALIZED_PATH.stat().st_size)

Saved: /content/health-psychology-rag-kb/data/normalized/michie_2011_behaviour_change_wheel.md
Size: 49215


In [ ]:
%cd {PROJECT_ROOT}

!git status --short

/content/health-psychology-rag-kb
 M data/normalized/michie_2011_behaviour_change_wheel.md
?? data/normalized/ogden_2019_health_psychology_BROKEN.md


# Document 4 REBUILD: Ogden (2019)

Цей розділ повністю перебудовує normalized Markdown для підручника Jane Ogden з нуля.

Стара версія не використовується як джерело істини. Новий pipeline читає тільки:

- raw PDF;
- figure assets із GitHub;
- явно задані metadata та figure descriptions.

Результат спочатку буде створено окремо й перевірено до заміни основного normalized файла.

In [ ]:
from pathlib import Path


OGDEN_RAW_PDF_PATH = (
    RAW_DIR
    / "ogden_2019_health_psychology.pdf"
)

OGDEN_CURRENT_PATH = (
    NORMALIZED_DIR
    / "ogden_2019_health_psychology.md"
)

OGDEN_BROKEN_PATH = (
    NORMALIZED_DIR
    / "ogden_2019_health_psychology_BROKEN.md"
)

OGDEN_ASSETS_DIR = ASSETS_DIR


assert OGDEN_RAW_PDF_PATH.exists(), (
    f"Missing PDF: {OGDEN_RAW_PDF_PATH}"
)

assert OGDEN_BROKEN_PATH.exists(), (
    f"Missing broken backup file: {OGDEN_BROKEN_PATH}"
)

assert OGDEN_ASSETS_DIR.exists(), (
    f"Missing assets directory: {OGDEN_ASSETS_DIR}"
)


print("Raw PDF:", OGDEN_RAW_PDF_PATH)
print(
    "New normalized output:",
    OGDEN_CURRENT_PATH
)
print(
    "Broken backup:",
    OGDEN_BROKEN_PATH
)
print(
    "Assets directory:",
    OGDEN_ASSETS_DIR
)

Raw PDF: /content/health-psychology-rag-kb/data/raw/ogden_2019_health_psychology.pdf
New normalized output: /content/health-psychology-rag-kb/data/normalized/ogden_2019_health_psychology.md
Broken backup: /content/health-psychology-rag-kb/data/normalized/ogden_2019_health_psychology_BROKEN.md
Assets directory: /content/health-psychology-rag-kb/data/normalized/assets


In [ ]:
import fitz


ogden_pdf = fitz.open(
    OGDEN_RAW_PDF_PATH
)

print("PDF pages:", len(ogden_pdf))
print("PDF title:", ogden_pdf.metadata.get("title"))
print("PDF author:", ogden_pdf.metadata.get("author"))

PDF pages: 108
PDF title: Microsoft Word - Ogden - The Psychology of Health and Illness_v2.docx
PDF author: 


In [ ]:
OGDEN_METADATA = {
    "document_id": "ogden_2019_health_psychology",
    "source_file": "ogden_2019_health_psychology.pdf",
    "title": "The Psychology of Health and Illness: An Open Access Course",
    "author": "Jane Ogden",
    "publication_year": 2019,
    "source_type": "pdf",
    "document_type": "textbook",
    "content_role": "core_reading",
    "domain": "health_psychology",
    "language": "en",
    "access_level": "open_access",
    "license": "CC BY 4.0",
}

OGDEN_UNIT_RANGES = {
    1: {
        "title": (
            "An Introduction to the Key Theoretical Frameworks "
            "of Psychology and Health"
        ),
        "page_start": 4,
        "page_end": 12,
    },
    2: {
        "title": "The Role of Behavior in Health",
        "page_start": 13,
        "page_end": 28,
    },
    3: {
        "title": "Behavior Change",
        "page_start": 29,
        "page_end": 36,
    },
    4: {
        "title": (
            "Becoming Ill and the Role of Illness Cognitions, "
            "Help-Seeking, and Communication"
        ),
        "page_start": 37,
        "page_end": 51,
    },
    5: {
        "title": "Being Ill and the Experience of Stress and Pain",
        "page_start": 52,
        "page_end": 67,
    },
    6: {
        "title": (
            "The Role of Psychology in Chronic Illnesses such as "
            "Obesity, Coronary Heart Disease, and Cancer"
        ),
        "page_start": 69,
        "page_end": 82,
    },
    7: {
        "title": "Gender, Health, and Illness",
        "page_start": 83,
        "page_end": 91,
    },
    8: {
        "title": "Health Outcomes and Quality of Life",
        "page_start": 92,
        "page_end": 99,
    },
}

OGDEN_OVERVIEW_PAGES = [2, 3]

print("Metadata ready:", OGDEN_METADATA["document_id"])
print("Units defined:", len(OGDEN_UNIT_RANGES))
print("Overview pages:", OGDEN_OVERVIEW_PAGES)

Metadata ready: ogden_2019_health_psychology
Units defined: 8
Overview pages: [2, 3]


In [ ]:
def preview_pdf_page(
    pdf_document,
    page_number: int,
    max_chars: int = 500,
) -> str:
    """
    page_number — номер сторінки як у PDF для користувача,
    починаючи з 1.
    """
    assert 1 <= page_number <= len(pdf_document)

    page_text = pdf_document[
        page_number - 1
    ].get_text("text")

    page_text = " ".join(
        page_text.split()
    )

    return page_text[:max_chars]


print("Overview page 2:")
print(preview_pdf_page(ogden_pdf, 2))
print()

for unit_number, unit_data in OGDEN_UNIT_RANGES.items():
    page_number = unit_data["page_start"]

    print(
        f'Unit {unit_number}, '
        f'page {page_number}:'
    )

    print(
        preview_pdf_page(
            ogden_pdf,
            page_number,
        )
    )

    print("-" * 80)

Overview page 2:
The Psychology of Health and Illness: Jane Ogden Overview For centuries health professionals have recognized that there are psychological consequences of being ill. A diagnosis of cancer or diabetes can make people anxious or depressed. This course will draw upon health psychology, public health, and community psychology to emphasize how psychology can also contribute to the cause, progression, experience, and outcomes of any physical illness. This course will highlight the many roles that psych

Unit 1, page 4:
The Psychology of Health and Illness: Jane Ogden Unit 1: An Introduction to the Key Theoretical Frameworks of Psychology and Health Overview Health psychology is the study of physical illness and addresses problems such as obesity, diabetes, cancer, and coronary heart disease (CHD) with a focus on health behaviors (eg. diet, exercise, sleep, help-seeking, medication adherence), illness beliefs, behavior change, and health outcomes. This first unit will describe

In [ ]:
def extract_pdf_pages(
    pdf_document,
    page_numbers,
):
    extracted_pages = []

    for page_number in page_numbers:
        assert 1 <= page_number <= len(pdf_document)

        page_text = pdf_document[
            page_number - 1
        ].get_text("text")

        extracted_pages.append(
            {
                "page_number": page_number,
                "text": page_text.strip(),
            }
        )

    return extracted_pages


ogden_selected_pages = []

ogden_selected_pages.extend(
    extract_pdf_pages(
        ogden_pdf,
        OGDEN_OVERVIEW_PAGES,
    )
)

for unit_number, unit_data in OGDEN_UNIT_RANGES.items():
    unit_page_numbers = range(
        unit_data["page_start"],
        unit_data["page_end"] + 1,
    )

    unit_pages = extract_pdf_pages(
        ogden_pdf,
        unit_page_numbers,
    )

    for page in unit_pages:
        page["unit_number"] = unit_number
        page["unit_title"] = unit_data["title"]

    ogden_selected_pages.extend(
        unit_pages
    )


print(
    "Selected pages:",
    len(ogden_selected_pages)
)

print(
    "First selected page:",
    ogden_selected_pages[0]["page_number"]
)

print(
    "Last selected page:",
    ogden_selected_pages[-1]["page_number"]
)

print(
    "Empty extracted pages:",
    sum(
        not page["text"]
        for page in ogden_selected_pages
    )
)

Selected pages: 97
First selected page: 2
Last selected page: 99
Empty extracted pages: 0


In [ ]:
from collections import Counter


first_line_counts = Counter()
last_line_counts = Counter()

for page in ogden_selected_pages:
    lines = [
        line.strip()
        for line in page["text"].splitlines()
        if line.strip()
    ]

    if lines:
        first_line_counts[lines[0]] += 1
        last_line_counts[lines[-1]] += 1


print("Most common first lines:")
for line, count in first_line_counts.most_common(10):
    print(f"{count:>3} × {line}")

print("\nMost common last lines:")
for line, count in last_line_counts.most_common(10):
    print(f"{count:>3} × {line}")

Most common first lines:
 94 × The psychology of health and illness: Jane Ogden
  3 × The Psychology of Health and Illness: Jane Ogden

Most common last lines:
  1 × 2
  1 × 3
  1 × such as chemical imbalances, bacteria, viruses, and genetic predisposition.
  1 × 5
  1 × 6
  1 × 7
  1 × 8
  1 × 9
  1 × 10
  1 × 11


In [ ]:
import re


OGDEN_HEADER_PATTERN = re.compile(
    r"^The psychology of health and illness:\s*Jane Ogden$",
    flags=re.IGNORECASE,
)

OGDEN_PAGE_NUMBER_PATTERN = re.compile(
    r"^\d+$"
)


def clean_ogden_page_lines(page_text: str) -> list[str]:
    lines = [
        line.strip()
        for line in page_text.splitlines()
        if line.strip()
    ]

    cleaned_lines = []

    for line_index, line in enumerate(lines):
        # Видаляємо повторюваний header.
        if (
            line_index == 0
            and OGDEN_HEADER_PATTERN.fullmatch(line)
        ):
            continue

        # Видаляємо номер сторінки лише в кінці сторінки.
        if (
            line_index == len(lines) - 1
            and OGDEN_PAGE_NUMBER_PATTERN.fullmatch(line)
        ):
            continue

        cleaned_lines.append(line)

    return cleaned_lines


ogden_cleaned_pages = []

for page in ogden_selected_pages:
    cleaned_page = dict(page)

    cleaned_page["lines"] = clean_ogden_page_lines(
        page["text"]
    )

    cleaned_page["text"] = "\n".join(
        cleaned_page["lines"]
    )

    ogden_cleaned_pages.append(
        cleaned_page
    )


print(
    "Cleaned pages:",
    len(ogden_cleaned_pages)
)

print(
    "Pages still starting with repeated header:",
    sum(
        bool(
            page["lines"]
            and OGDEN_HEADER_PATTERN.fullmatch(
                page["lines"][0]
            )
        )
        for page in ogden_cleaned_pages
    )
)

print(
    "Pages still ending with page number:",
    sum(
        bool(
            page["lines"]
            and OGDEN_PAGE_NUMBER_PATTERN.fullmatch(
                page["lines"][-1]
            )
        )
        for page in ogden_cleaned_pages
    )
)

print(
    "Empty pages after cleanup:",
    sum(
        not page["lines"]
        for page in ogden_cleaned_pages
    )
)

Cleaned pages: 97
Pages still starting with repeated header: 0
Pages still ending with page number: 0
Empty pages after cleanup: 0


In [ ]:
for page_number in [2, 4, 13, 29, 62]:
    page = next(
        item
        for item in ogden_cleaned_pages
        if item["page_number"] == page_number
    )

    print(f"\nPAGE {page_number}")
    print("=" * 80)

    for line_number, line in enumerate(
        page["lines"][:40],
        start=1,
    ):
        print(
            f"{line_number:>2}: {line}"
        )


PAGE 2
 1: Overview
 2: For centuries health professionals have recognized that there are psychological
 3: consequences of being ill. A diagnosis of cancer or diabetes can make people anxious or
 4: depressed. This course will draw upon health psychology, public health, and community
 5: psychology to emphasize how psychology can also contribute to the cause, progression,
 6: experience, and outcomes of any physical illness. This course will highlight the many roles
 7: that psychology plays in physical illness from i) being and staying well and the role of health
 8: behaviors and behavior change; ii) becoming ill with a focus on illness beliefs, symptom
 9: perception, help-seeking and communication with health professionals; iii) being ill in terms
10: of stress, pain, and chronic illnesses such as obesity, coronary heart disease, and cancer; iv)
11: the role of gender in health, and v) health outcomes in terms of Quality of Life and
12: longevity.
13: Learning objectives and outc

In [ ]:
OGDEN_UNIT_TITLES = {
    1: (
        "Unit 1: An Introduction to the Key Theoretical "
        "Frameworks of Psychology and Health"
    ),
    2: "Unit 2: The Role of Behavior in Health",
    3: "Unit 3: Behavior Change",
    4: (
        "Unit 4: Becoming Ill and the Role of Illness "
        "Cognitions, Help-Seeking, and Communication"
    ),
    5: (
        "Unit 5: Being Ill and the Experience "
        "of Stress and Pain"
    ),
    6: (
        "Unit 6: The Role of Psychology in Chronic "
        "Illnesses such as Obesity, Coronary Heart "
        "Disease, and Cancer"
    ),
    7: "Unit 7: Gender, Health, and Illness",
    8: (
        "Unit 8: Health Outcomes and "
        "Quality of Life"
    ),
}


def get_unit_number_for_page(
    page_number: int,
) -> int | None:
    for unit_number, unit_data in OGDEN_UNIT_RANGES.items():
        if (
            unit_data["page_start"]
            <= page_number
            <= unit_data["page_end"]
        ):
            return unit_number

    return None


ogden_pages_with_canonical_units = []

for page in ogden_cleaned_pages:
    updated_page = dict(page)
    updated_lines = list(page["lines"])

    unit_number = get_unit_number_for_page(
        page["page_number"]
    )

    if (
        unit_number is not None
        and page["page_number"]
        == OGDEN_UNIT_RANGES[unit_number]["page_start"]
    ):
        canonical_title = OGDEN_UNIT_TITLES[
            unit_number
        ]

        # Видаляємо сирий PDF-заголовок unit.
        updated_lines.pop(0)

        # Unit 1, 4 і 6 мають продовження
        # заголовка на наступному рядку.
        if (
            updated_lines
            and updated_lines[0] != "Overview"
        ):
            updated_lines.pop(0)

        updated_lines.insert(
            0,
            canonical_title,
        )

    updated_page["lines"] = updated_lines
    updated_page["text"] = "\n".join(
        updated_lines
    )

    ogden_pages_with_canonical_units.append(
        updated_page
    )


for unit_number, unit_data in OGDEN_UNIT_RANGES.items():
    page = next(
        item
        for item in ogden_pages_with_canonical_units
        if item["page_number"]
        == unit_data["page_start"]
    )

    print(
        f'Unit {unit_number}: '
        f'{page["lines"][0]}'
    )

    print(
        "Next line:",
        page["lines"][1],
    )

    print("-" * 80)

Unit 1: Unit 1: An Introduction to the Key Theoretical Frameworks of Psychology and Health
Next line: Overview
--------------------------------------------------------------------------------
Unit 2: Unit 2: The Role of Behavior in Health
Next line: Overview
--------------------------------------------------------------------------------
Unit 3: Unit 3: Behavior Change
Next line: Overview
--------------------------------------------------------------------------------
Unit 4: Unit 4: Becoming Ill and the Role of Illness Cognitions, Help-Seeking, and Communication
Next line: Overview
--------------------------------------------------------------------------------
Unit 5: Unit 5: Being Ill and the Experience of Stress and Pain
Next line: Overview
--------------------------------------------------------------------------------
Unit 6: Unit 6: The Role of Psychology in Chronic Illnesses such as Obesity, Coronary Heart Disease, and Cancer
Next line: Overview
--------------------------------

In [ ]:
page_62 = next(
    page
    for page in ogden_pages_with_canonical_units
    if page["page_number"] == 62
)

assert page_62["lines"][-2:] == [
    "Figure 4",
    "The Gate Control Theory of Pain",
]

page_62["lines"][-2:] = [
    "Figure 4: The Gate Control Theory of Pain"
]

page_62["text"] = "\n".join(
    page_62["lines"]
)

print(
    "Page 62 Figure 4 caption repaired:"
)

print(
    page_62["lines"][-1]
)

assert (
    page_62["lines"][-1]
    == "Figure 4: The Gate Control Theory of Pain"
)

Page 62 Figure 4 caption repaired:
Figure 4: The Gate Control Theory of Pain


In [ ]:
OGDEN_FIXED_HEADINGS = {
    "Overview",
    "Learning objectives and outcomes",
    "About the author",
    "The Background of Health Psychology",
    "The Biomedical Model",
    "Health Psychology",
    "The Four Key Theoretical Frameworks",
    "To Conclude",
    "A Key Role for Behavior",
    "What are Health Behaviors?",
    "Individual Beliefs about Behavior",
    "Models of Behavior",
    "In Summary",
    "The Example of Eating Behavior",
    "Four Main Theories Informing Behavior Change",
    "What are Illness Beliefs?",
    "Final Take-Home Message",
}


OGDEN_NUMBERED_HEADINGS = {
    "1.The Biopsychosocial Model",
    "2. Health and Illness as a Continuum",
    "3. The Direct and Indirect Pathways between Psychology and Health",
    "3. A Focus on Variability",
    "1.The Stages of Change Model",
    "2.The Health Belief Model",
    "3. The Protection Motivation Theory",
    "4. The Theory of Planned Behavior",
    "1. Cognition Models",
    "2. The Developmental Model",
    "3. A Weight Concern Model of Eating Behavior",
    "1. Learning Theory (with added cognitions)",
    "2. Social Cognition Theory and the Use of Planning",
    "3. Stages of Change Model and Motivational Interviewing",
    "4. Using Emotion",
}


def classify_ogden_line(line: str) -> str:
    line = line.strip()

    if re.match(
        r"^Unit\s+\d+[.:]",
        line,
    ):
        return "unit_heading"

    if re.match(
        r"^(?:Fig\.?|Figure)\s*\d+\s*[:.]?\s+"
        r"(?!It\s+suggested\b)",
        line,
        flags=re.IGNORECASE,
    ):
        return "figure_heading"

    if line in OGDEN_FIXED_HEADINGS:
        return "section_heading"

    if line in OGDEN_NUMBERED_HEADINGS:
        return "numbered_section_heading"

    return "body"


ogden_line_type_counts = Counter()

for page in ogden_pages_with_canonical_units:
    for line in page["lines"]:
        line_type = classify_ogden_line(line)
        ogden_line_type_counts[line_type] += 1


print("Line types:")

for line_type, count in ogden_line_type_counts.items():
    print(
        f"{line_type}: {count}"
    )


print("\nDetected structural lines:")

for page in ogden_pages_with_canonical_units:
    for line_number, line in enumerate(
        page["lines"],
        start=1,
    ):
        line_type = classify_ogden_line(line)

        if line_type != "body":
            print(
                f'Page {page["page_number"]:>2}, '
                f'line {line_number:>2}, '
                f'{line_type}: {line}'
            )


assert (
    ogden_line_type_counts["unit_heading"]
    == 8
)

assert (
    ogden_line_type_counts["figure_heading"]
    == 15
)

assert (
    ogden_line_type_counts[
        "numbered_section_heading"
    ]
    == 15
)

print(
    "\nOgden structural classification passed"
)

Line types:
section_heading: 35
body: 1864
unit_heading: 8
numbered_section_heading: 15
figure_heading: 15

Detected structural lines:
Page  2, line  1, section_heading: Overview
Page  2, line 13, section_heading: Learning objectives and outcomes
Page  2, line 29, section_heading: About the author
Page  4, line  1, unit_heading: Unit 1: An Introduction to the Key Theoretical Frameworks of Psychology and Health
Page  4, line  2, section_heading: Overview
Page  4, line 11, section_heading: The Background of Health Psychology
Page  4, line 16, section_heading: The Biomedical Model
Page  5, line 14, section_heading: Health Psychology
Page  7, line  1, section_heading: The Four Key Theoretical Frameworks
Page  7, line  2, numbered_section_heading: 1.The Biopsychosocial Model
Page  7, line 13, figure_heading: Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980)
Page  8, line  1, numbered_section_heading: 2. Health and Illness as a Continuum
Page  8, line  9, figure_

In [ ]:
OGDEN_STRUCTURAL_TYPES = {
    "unit_heading",
    "section_heading",
    "numbered_section_heading",
    "figure_heading",
}


def build_ogden_page_blocks(
    page: dict,
) -> list[dict]:
    blocks = []
    current_body_lines = []

    def flush_body():
        nonlocal current_body_lines

        if not current_body_lines:
            return

        body_text = re.sub(
            r"\s+",
            " ",
            " ".join(current_body_lines),
        ).strip()

        if body_text:
            blocks.append(
                {
                    "type": "body",
                    "page_start": page["page_number"],
                    "page_end": page["page_number"],
                    "text": body_text,
                }
            )

        current_body_lines = []

    for line in page["lines"]:
        line_type = classify_ogden_line(line)

        if line_type in OGDEN_STRUCTURAL_TYPES:
            flush_body()

            blocks.append(
                {
                    "type": line_type,
                    "page_start": page["page_number"],
                    "page_end": page["page_number"],
                    "text": line.strip(),
                }
            )
        else:
            current_body_lines.append(line)

    flush_body()

    return blocks


ogden_semantic_blocks = []

for page in ogden_pages_with_canonical_units:
    page_blocks = build_ogden_page_blocks(
        page
    )

    unit_number = get_unit_number_for_page(
        page["page_number"]
    )

    for block in page_blocks:
        block["unit_number"] = unit_number

    ogden_semantic_blocks.extend(
        page_blocks
    )


semantic_block_counts = Counter(
    block["type"]
    for block in ogden_semantic_blocks
)

print(
    "Total semantic blocks:",
    len(ogden_semantic_blocks)
)

for block_type, count in semantic_block_counts.items():
    print(
        f"{block_type}: {count}"
    )


assert semantic_block_counts["unit_heading"] == 8
assert semantic_block_counts["figure_heading"] == 15
assert (
    semantic_block_counts[
        "numbered_section_heading"
    ]
    == 15
)

print(
    "\nOgden semantic block construction passed"
)

Total semantic blocks: 206
section_heading: 35
body: 133
unit_heading: 8
numbered_section_heading: 15
figure_heading: 15

Ogden semantic block construction passed


In [ ]:
def looks_like_sentence_continuation(
    previous_text: str,
    current_text: str,
) -> bool:
    previous_text = previous_text.strip()
    current_text = current_text.strip()

    if not previous_text or not current_text:
        return False

    previous_ends_sentence = bool(
        re.search(r'[.!?]["”’)]?$', previous_text)
    )

    current_starts_lowercase = bool(
        re.match(r'^[a-z]', current_text)
    )

    current_starts_continuation_word = bool(
        re.match(
            r"^(?:and|but|or|because|which|that|"
            r"with|of|to|in|on|for|from|as|"
            r"by|than|whereas|while)\b",
            current_text,
            flags=re.IGNORECASE,
        )
    )

    return (
        not previous_ends_sentence
        and (
            current_starts_lowercase
            or current_starts_continuation_word
        )
    )


ogden_cross_page_candidates = []

for index in range(
    1,
    len(ogden_semantic_blocks),
):
    previous_block = ogden_semantic_blocks[
        index - 1
    ]

    current_block = ogden_semantic_blocks[
        index
    ]

    if (
        previous_block["type"] == "body"
        and current_block["type"] == "body"
        and current_block["page_start"]
        == previous_block["page_end"] + 1
        and previous_block["unit_number"]
        == current_block["unit_number"]
        and looks_like_sentence_continuation(
            previous_block["text"],
            current_block["text"],
        )
    ):
        ogden_cross_page_candidates.append(
            {
                "left_index": index - 1,
                "right_index": index,
                "left_page": previous_block[
                    "page_end"
                ],
                "right_page": current_block[
                    "page_start"
                ],
                "left_text": previous_block[
                    "text"
                ],
                "right_text": current_block[
                    "text"
                ],
            }
        )


print(
    "Cross-page continuation candidates:",
    len(ogden_cross_page_candidates)
)

for item in ogden_cross_page_candidates:
    print(
        f'\nPages {item["left_page"]}'
        f' → {item["right_page"]}'
    )

    print(
        "LEFT:",
        item["left_text"][-180:],
    )

    print(
        "RIGHT:",
        item["right_text"][:180],
    )

Cross-page continuation candidates: 36

Pages 5 → 6
LEFT: s and not by a single causal factor. Health psychology, therefore, attempts to move away from a simple linear model of health and claims that illness can be caused by a combination
RIGHT: of biological (e.g. a virus), psychological (e.g. behaviors, beliefs) and social (e.g. social support) factors. § Who is responsible for illness? Because illness is regarded as a r

Pages 8 → 9
LEFT: t impact upon their body through changes in their physiology which can change their health status. The indirect pathway is reflected more in the behavioral literature and from this
RIGHT: perspective, the ways a person thinks (“I am feeling stressed”) influences their behavior (“I will have a cigarette”) which in turn can impact upon their health. The direct and ind

Pages 9 → 10
LEFT:  within a month. This variability indicates that health and illness cannot only be explained by illness severity (type of cancer, severity of heart attack) or knowle

In [ ]:
from copy import deepcopy


ogden_merged_blocks = deepcopy(
    ogden_semantic_blocks
)

merge_pairs = sorted(
    {
        (
            item["left_index"],
            item["right_index"],
        )
        for item in ogden_cross_page_candidates
    },
    reverse=True,
)


for left_index, right_index in merge_pairs:
    left_block = ogden_merged_blocks[
        left_index
    ]

    right_block = ogden_merged_blocks[
        right_index
    ]

    assert right_index == left_index + 1

    assert left_block["type"] == "body"
    assert right_block["type"] == "body"

    assert (
        right_block["page_start"]
        == left_block["page_end"] + 1
    )

    assert (
        left_block["unit_number"]
        == right_block["unit_number"]
    )

    left_block["text"] = re.sub(
        r"\s+",
        " ",
        (
            left_block["text"].rstrip()
            + " "
            + right_block["text"].lstrip()
        ),
    ).strip()

    left_block["page_end"] = (
        right_block["page_end"]
    )

    del ogden_merged_blocks[
        right_index
    ]


print(
    "Blocks before cross-page merge:",
    len(ogden_semantic_blocks),
)

print(
    "Cross-page merges applied:",
    len(merge_pairs),
)

print(
    "Blocks after cross-page merge:",
    len(ogden_merged_blocks),
)


assert (
    len(ogden_merged_blocks)
    == len(ogden_semantic_blocks)
    - len(merge_pairs)
)

print(
    "\nCross-page paragraph merge passed"
)

Blocks before cross-page merge: 206
Cross-page merges applied: 36
Blocks after cross-page merge: 170

Cross-page paragraph merge passed


In [ ]:
OGDEN_SUSPICIOUS_SYMBOLS = {
    "section_sign": "§",
    "bullet": "•",
    "unicode_artifact": "￾",
}


print("Suspicious symbols after merging:")

for symbol_name, symbol in OGDEN_SUSPICIOUS_SYMBOLS.items():
    occurrence_count = sum(
        block["text"].count(symbol)
        for block in ogden_merged_blocks
    )

    affected_blocks = sum(
        symbol in block["text"]
        for block in ogden_merged_blocks
    )

    print(
        f"{symbol_name}: "
        f"{occurrence_count} occurrences "
        f"in {affected_blocks} blocks"
    )


print("\nBlocks containing §:")

for block in ogden_merged_blocks:
    if "§" in block["text"]:
        print(
            f'pages {block["page_start"]}'
            f'–{block["page_end"]} | '
            f'{block["text"][:350]}'
        )
        print("-" * 80)

Suspicious symbols after merging:
section_sign: 28 occurrences in 8 blocks
bullet: 45 occurrences in 8 blocks
unicode_artifact: 0 occurrences in 0 blocks

Blocks containing §:
pages 4–4 | The biomedical model can be understood in terms of its answers to the following 5 questions: § What causes illness? According to the biomedical model, diseases either come from outside the body, invading the body and causing physical changes within the body, or originate as internal physical changes. Such diseases may be caused by several factors s
--------------------------------------------------------------------------------
pages 5–5 | § Who is responsible for illness? Because illness is seen as arising from biological changes beyond their control, individuals are not seen as responsible for their illness. They are regarded as victims of some external force causing internal changes. § How should illness be treated? The biomedical model regards treatment in terms of vaccination, m
-----------------

In [ ]:
def normalize_ogden_bullets(
    text: str,
) -> str:
    text = text.strip()

    # Перетворюємо PDF-маркери § і •
    # на Markdown bullets.
    text = re.sub(
        r"\s*[§•]\s*",
        "\n\n- ",
        text,
    )

    # Нормалізуємо bullet на початку блока.
    text = re.sub(
        r"^\s*-\s*",
        "- ",
        text,
    )

    # Прибираємо зайві порожні рядки.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


ogden_bullet_cleaned_blocks = deepcopy(
    ogden_merged_blocks
)


for block in ogden_bullet_cleaned_blocks:
    if block["type"] == "body":
        block["text"] = normalize_ogden_bullets(
            block["text"]
        )


remaining_section_signs = sum(
    block["text"].count("§")
    for block in ogden_bullet_cleaned_blocks
)

remaining_bullet_symbols = sum(
    block["text"].count("•")
    for block in ogden_bullet_cleaned_blocks
)


MARKDOWN_BULLET_PATTERN = re.compile(
    r"(?m)^- "
)

total_markdown_bullets = sum(
    len(
        MARKDOWN_BULLET_PATTERN.findall(
            block["text"]
        )
    )
    for block in ogden_bullet_cleaned_blocks
)


print(
    "Remaining §:",
    remaining_section_signs
)

print(
    "Remaining •:",
    remaining_bullet_symbols
)

print(
    "Markdown bullets:",
    total_markdown_bullets
)


assert remaining_section_signs == 0
assert remaining_bullet_symbols == 0

assert all(
    "§" not in block["text"]
    and "•" not in block["text"]
    for block in ogden_bullet_cleaned_blocks
)

print(
    "\nOgden bullet normalization passed"
)

Remaining §: 0
Remaining •: 0
Markdown bullets: 73

Ogden bullet normalization passed


In [ ]:
MARKDOWN_BULLET_PATTERN = re.compile(
    r"(?m)^- "
)


bullet_blocks = [
    block
    for block in ogden_bullet_cleaned_blocks
    if MARKDOWN_BULLET_PATTERN.search(
        block["text"]
    )
]

total_markdown_bullets = sum(
    len(
        MARKDOWN_BULLET_PATTERN.findall(
            block["text"]
        )
    )
    for block in bullet_blocks
)


print(
    "Blocks containing Markdown bullets:",
    len(bullet_blocks)
)

print(
    "Total Markdown bullets:",
    total_markdown_bullets
)


for block in bullet_blocks:
    print(
        f'\nPages {block["page_start"]}'
        f'–{block["page_end"]}'
    )

    print(
        block["text"][:700]
    )

    print("-" * 80)

Blocks containing Markdown bullets: 16
Total Markdown bullets: 73

Pages 4–4
The biomedical model can be understood in terms of its answers to the following 5 questions:

- What causes illness? According to the biomedical model, diseases either come from outside the body, invading the body and causing physical changes within the body, or originate as internal physical changes. Such diseases may be caused by several factors such as chemical imbalances, bacteria, viruses, and genetic predisposition.
--------------------------------------------------------------------------------

Pages 5–5
- Who is responsible for illness? Because illness is seen as arising from biological changes beyond their control, individuals are not seen as responsible for their illness. They are regarded as victims of some external force causing internal changes.

- How should illness be treated? The biomedical model regards treatment in terms of vaccination, medication, chemotherapy, and surgery, all of which aim

In [ ]:
from pathlib import Path


# 1. Mapping між figures у PDF та PNG-файлами.
OGDEN_FIGURE_ASSETS = {
    (7, 1): "assets/ogden_p7_figure_1.png",
    (8, 2): "assets/ogden_p8_figure_2.png",
    (9, 3): "assets/ogden_p9_figure_3.png",
    (10, 4): "assets/ogden_p10_figure_4.png",
    (22, 3): "assets/ogden_p22_figure_3.png",
    (39, 1): "assets/ogden_p39_figure_1.png",
    (54, 1): "assets/ogden_p54_figure_1.png",
    (55, 2): "assets/ogden_p55_figure_2.png",
}


# 2. Знаходимо всі реальні Ogden PNG у папці assets.
ogden_asset_files = sorted(
    OGDEN_ASSETS_DIR.glob("ogden_*.png")
)


print(
    "Discovered Ogden image assets:",
    len(ogden_asset_files)
)

for asset_path in ogden_asset_files:
    print(asset_path.name)


# 3. Перевіряємо кожен шлях із mapping.
print(
    "\nMapped Ogden figures:",
    len(OGDEN_FIGURE_ASSETS)
)

for (
    page_number,
    figure_number,
), relative_path in OGDEN_FIGURE_ASSETS.items():
    asset_path = NORMALIZED_DIR / relative_path

    print(
        f"Page {page_number}, "
        f"figure {figure_number} → "
        f"{relative_path} | "
        f"exists={asset_path.exists()}"
    )

    assert asset_path.exists(), (
        f"Missing asset: {asset_path}"
    )


# 4. Перевіряємо, що папка і mapping повністю синхронні.
mapped_asset_names = {
    Path(relative_path).name
    for relative_path in OGDEN_FIGURE_ASSETS.values()
}

discovered_asset_names = {
    asset_path.name
    for asset_path in ogden_asset_files
}

missing_from_mapping = (
    discovered_asset_names
    - mapped_asset_names
)

missing_from_folder = (
    mapped_asset_names
    - discovered_asset_names
)

assert not missing_from_mapping, (
    "PNG files exist but are absent from "
    f"OGDEN_FIGURE_ASSETS: "
    f"{sorted(missing_from_mapping)}"
)

assert not missing_from_folder, (
    "OGDEN_FIGURE_ASSETS references missing files: "
    f"{sorted(missing_from_folder)}"
)

print(
    "\nOgden figure asset mapping validation passed"
)

Discovered Ogden image assets: 8
ogden_p10_figure_4.png
ogden_p22_figure_3.png
ogden_p39_figure_1.png
ogden_p54_figure_1.png
ogden_p55_figure_2.png
ogden_p7_figure_1.png
ogden_p8_figure_2.png
ogden_p9_figure_3.png

Mapped Ogden figures: 8
Page 7, figure 1 → assets/ogden_p7_figure_1.png | exists=True
Page 8, figure 2 → assets/ogden_p8_figure_2.png | exists=True
Page 9, figure 3 → assets/ogden_p9_figure_3.png | exists=True
Page 10, figure 4 → assets/ogden_p10_figure_4.png | exists=True
Page 22, figure 3 → assets/ogden_p22_figure_3.png | exists=True
Page 39, figure 1 → assets/ogden_p39_figure_1.png | exists=True
Page 54, figure 1 → assets/ogden_p54_figure_1.png | exists=True
Page 55, figure 2 → assets/ogden_p55_figure_2.png | exists=True

Ogden figure asset mapping validation passed


In [ ]:
assert len(ogden_figure_inventory) > 0

assert all(
    item["page_number"] >= 1
    and item["figure_number"] >= 1
    and item["heading"]
    for item in ogden_figure_inventory
)

mapped_asset_count = sum(
    item["asset_path"] is not None
    for item in ogden_figure_inventory
)

description_only_count = (
    len(ogden_figure_inventory)
    - mapped_asset_count
)

assert (
    mapped_asset_count
    == len(OGDEN_FIGURE_ASSETS)
)

assert all(
    item["asset_path"] is None
    or (
        NORMALIZED_DIR
        / item["asset_path"]
    ).exists()
    for item in ogden_figure_inventory
)

print(
    "\nFigure inventory validation passed"
)

print(
    "Total figures:",
    len(ogden_figure_inventory)
)

print(
    "Figures with image assets:",
    mapped_asset_count
)

print(
    "Figures without image assets:",
    description_only_count
)


Figure inventory validation passed
Total figures: 15
Figures with image assets: 8
Figures without image assets: 7


In [ ]:
OGDEN_FIGURE_DESCRIPTIONS = {
    (
        item["page_number"],
        item["figure_number"],
    ): ""
    for item in ogden_figure_inventory
}


print(
    "Figure description slots created:",
    len(OGDEN_FIGURE_DESCRIPTIONS)
)


assert (
    set(OGDEN_FIGURE_DESCRIPTIONS)
    == {
        (
            item["page_number"],
            item["figure_number"],
        )
        for item in ogden_figure_inventory
    }
)

print(
    "\nFigure description registry validation passed"
)

Figure description slots created: 15

Figure description registry validation passed


In [ ]:
broken_ogden_path = (
    NORMALIZED_DIR
    / "ogden_2019_health_psychology_BROKEN.md"
)

broken_ogden_text = broken_ogden_path.read_text(
    encoding="utf-8"
)

existing_figure_descriptions = re.findall(
    r"^Figure description:\s*(.+)$",
    broken_ogden_text,
    flags=re.MULTILINE,
)

figure_keys = [
    (
        item["page_number"],
        item["figure_number"],
    )
    for item in ogden_figure_inventory
]

assert len(existing_figure_descriptions) == len(
    figure_keys
), (
    "Figure descriptions and figure inventory "
    "have different lengths"
)

OGDEN_FIGURE_DESCRIPTIONS = dict(
    zip(
        figure_keys,
        existing_figure_descriptions,
    )
)

assert all(
    description.strip()
    for description
    in OGDEN_FIGURE_DESCRIPTIONS.values()
)

print(
    "Figure descriptions loaded:",
    len(OGDEN_FIGURE_DESCRIPTIONS)
)

print("\nLoaded figure descriptions:\n")

for (
    page_number,
    figure_number,
), description in OGDEN_FIGURE_DESCRIPTIONS.items():
    print(
        f"Page {page_number}, "
        f"figure {figure_number}"
    )

    print(description)

    print("-" * 80)

print(
    "\nOgden figure descriptions passed"
)

Figure descriptions loaded: 15

Loaded figure descriptions:

Page 7, figure 1
The biopsychosocial model explains health and illness through the interaction of biological, psychological, and social factors.
--------------------------------------------------------------------------------
Page 8, figure 2
Health and illness are presented as a continuum. Psychological factors influence illness onset, help-seeking, adaptation, illness progression, and health outcomes.
--------------------------------------------------------------------------------
Page 9, figure 3
The direct pathway shows psychological states influencing health through physiological changes. The indirect pathway shows psychological states influencing behavior, which then affects health.
--------------------------------------------------------------------------------
Page 10, figure 4
The model highlights variability between people in knowledge, behavior, illness progression, and health outcomes.
----------------------------

In [ ]:
ogden_normalized_path = (
    NORMALIZED_DIR
    / "ogden_2019_health_psychology.md"
)


markdown_parts = [
    "---",
    f'document_id: "{OGDEN_METADATA["document_id"]}"',
    f'source_file: "{OGDEN_METADATA["source_file"]}"',
    f'title: "{OGDEN_METADATA["title"]}"',
    f'author: "{OGDEN_METADATA["author"]}"',
    f'publication_year: {OGDEN_METADATA["publication_year"]}',
    f'document_type: "{OGDEN_METADATA["document_type"]}"',
    f'content_role: "{OGDEN_METADATA["content_role"]}"',
    f'domain: "{OGDEN_METADATA["domain"]}"',
    f'language: "{OGDEN_METADATA["language"]}"',
    f'license: "{OGDEN_METADATA["license"]}"',
    "---",
    "",
    f'# {OGDEN_METADATA["title"]}',
    "",
]


for block in ogden_bullet_cleaned_blocks:
    block_type = block["type"]
    block_text = block["text"].strip()

    if block_type == "unit_heading":
        markdown_parts.extend([
            f"## {block_text}",
            "",
        ])

    elif block_type in {
        "section_heading",
        "numbered_section_heading",
    }:
        markdown_parts.extend([
            f"### {block_text}",
            "",
        ])

    elif block_type == "figure_heading":
        page_number = block["page_start"]
        figure_number = extract_figure_number(
            block_text
        )

        figure_key = (
            page_number,
            figure_number,
        )

        markdown_parts.extend([
            f"#### {block_text}",
            "",
        ])

        asset_path = OGDEN_FIGURE_ASSETS.get(
            figure_key
        )

        if asset_path:
            markdown_parts.extend([
                f"![{block_text}]({asset_path})",
                "",
            ])

        markdown_parts.extend([
            (
                "Figure description: "
                + OGDEN_FIGURE_DESCRIPTIONS[
                    figure_key
                ]
            ),
            "",
        ])

    else:
        markdown_parts.extend([
            block_text,
            "",
        ])


ogden_normalized_text = "\n".join(
    markdown_parts
).strip() + "\n"

ogden_normalized_path.write_text(
    ogden_normalized_text,
    encoding="utf-8",
)


print(
    "Saved:",
    ogden_normalized_path
)

print(
    "Characters:",
    len(ogden_normalized_text)
)

print(
    "Figure descriptions:",
    ogden_normalized_text.count(
        "Figure description:"
    )
)

print(
    "Image links:",
    ogden_normalized_text.count(
        "]("
    )
)


assert ogden_normalized_path.exists()

assert (
    ogden_normalized_text.count(
        "Figure description:"
    )
    == len(ogden_figure_inventory)
)

assert (
    ogden_normalized_text.count(
        "]("
    )
    == len(OGDEN_FIGURE_ASSETS)
)

print(
    "\nOgden normalized Markdown created"
)

Saved: /content/health-psychology-rag-kb/data/normalized/ogden_2019_health_psychology.md
Characters: 152468
Figure descriptions: 15
Image links: 8

Ogden normalized Markdown created


In [ ]:
final_ogden_text = ogden_normalized_path.read_text(
    encoding="utf-8"
)

unit_heading_count = len(
    re.findall(
        r"(?m)^## ",
        final_ogden_text,
    )
)

section_heading_count = len(
    re.findall(
        r"(?m)^### ",
        final_ogden_text,
    )
)

markdown_bullet_count = len(
    re.findall(
        r"(?m)^- ",
        final_ogden_text,
    )
)

figure_description_count = (
    final_ogden_text.count(
        "Figure description:"
    )
)

remaining_section_signs = (
    final_ogden_text.count("§")
)

remaining_bullet_symbols = (
    final_ogden_text.count("•")
)


print(
    "Unit headings:",
    unit_heading_count
)

print(
    "Section headings:",
    section_heading_count
)

print(
    "Markdown bullets:",
    markdown_bullet_count
)

print(
    "Figure descriptions:",
    figure_description_count
)

print(
    "Remaining §:",
    remaining_section_signs
)

print(
    "Remaining •:",
    remaining_bullet_symbols
)


assert final_ogden_text.startswith("---\n")

assert (
    figure_description_count
    == len(ogden_figure_inventory)
)

assert remaining_section_signs == 0
assert remaining_bullet_symbols == 0

print(
    "\nFinal Ogden Markdown validation passed"
)

Unit headings: 8
Section headings: 50
Markdown bullets: 73
Figure descriptions: 15
Remaining §: 0
Remaining •: 0

Final Ogden Markdown validation passed


# Document 4: Ogden (2019) — The Psychology of Health and Illness

In [ ]:
OGDEN_RAW_PDF_PATH = (
    RAW_DIR
    / "ogden_2019_health_psychology.pdf"
)

OGDEN_NORMALIZED_PATH = (
    NORMALIZED_DIR
    / "ogden_2019_health_psychology.md"
)

OGDEN_NORMALIZED_MD_PATH = OGDEN_NORMALIZED_PATH

OGDEN_ASSETS_DIR = ASSETS_DIR


assert OGDEN_RAW_PDF_PATH.exists(), (
    f"Missing Ogden PDF: {OGDEN_RAW_PDF_PATH}"
)

assert OGDEN_ASSETS_DIR.exists(), (
    f"Missing assets directory: {OGDEN_ASSETS_DIR}"
)


print("PDF exists:", OGDEN_RAW_PDF_PATH.exists())
print("PDF size:", OGDEN_RAW_PDF_PATH.stat().st_size)
print("Normalized path:", OGDEN_NORMALIZED_PATH)
print("Assets directory:", OGDEN_ASSETS_DIR)

PDF exists: True
PDF size: 1155544
Normalized path: /content/health-psychology-rag-kb/data/normalized/ogden_2019_health_psychology.md
Assets directory: /content/health-psychology-rag-kb/data/normalized/assets


In [ ]:
OGDEN_EXPECTED_ASSETS = [
    "ogden_p7_figure_1.png",
    "ogden_p8_figure_2.png",
    "ogden_p9_figure_3.png",
    "ogden_p10_figure_4.png",
    "ogden_p22_figure_3.png",
    "ogden_p39_figure_1.png",
    "ogden_p54_figure_1.png",
    "ogden_p55_figure_2.png",
]


missing_ogden_assets = []

for asset_name in OGDEN_EXPECTED_ASSETS:
    asset_path = (
        OGDEN_ASSETS_DIR
        / asset_name
    )

    exists = asset_path.exists()

    print(
        asset_name,
        "exists:",
        exists,
        "size:",
        asset_path.stat().st_size
        if exists
        else 0,
    )

    if not exists:
        missing_ogden_assets.append(
            asset_name
        )


assert not missing_ogden_assets, (
    "Missing Ogden assets: "
    + ", ".join(missing_ogden_assets)
)

print("\nOgden source and assets validation passed")

ogden_p7_figure_1.png exists: True size: 169013
ogden_p8_figure_2.png exists: True size: 340635
ogden_p9_figure_3.png exists: True size: 204578
ogden_p10_figure_4.png exists: True size: 24832
ogden_p22_figure_3.png exists: True size: 327717
ogden_p39_figure_1.png exists: True size: 347488
ogden_p54_figure_1.png exists: True size: 176207
ogden_p55_figure_2.png exists: True size: 204044

Ogden source and assets validation passed


In [ ]:
!pip install pymupdf

In [ ]:
OGDEN_RAW_PDF_PATH = (
    RAW_DIR
    / "ogden_2019_health_psychology.pdf"
)

assert OGDEN_RAW_PDF_PATH.exists(), (
    f"Missing PDF: {OGDEN_RAW_PDF_PATH}"
)

import fitz

ogden_pdf = fitz.open(
    OGDEN_RAW_PDF_PATH
)

print("Pages:", len(ogden_pdf))
print("Metadata:", ogden_pdf.metadata)

Pages: 108
Metadata: {'format': 'PDF 1.3', 'title': 'Microsoft Word - Ogden - The Psychology of Health and Illness_v2.docx', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Word', 'producer': 'macOS Version 10.14.1 (Build 18B75) Quartz PDFContext', 'creationDate': 'D:20190315203022Z', 'modDate': "D:20210415172059-04'00'", 'trapped': '', 'encryption': None}


In [ ]:
pages_to_check = [1, 2, 3, 4]

for printed_page in pages_to_check:
    pdf_index = printed_page - 1
    page = ogden_pdf[pdf_index]
    page_text = page.get_text("text")

    print(f"\n{'=' * 80}")
    print(f"PRINTED PAGE {printed_page}")
    print(f"{'=' * 80}\n")
    print(page_text[:2000])


PRINTED PAGE 1

 
The Psychology of Health and Illness: 
 
 
 
An Open Access Course 
  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Jane Ogden 
 
This course is distributed under the terms of the Creative Commons Attribution 4.0 International License 
 
(http://creativecommons.org/licenses/by/4.0/), which permits unrestricted use, distribution, and reproduction 
in any medium, provided you give appropriate credit to the original author and the source, provide a link to 
the Creative Commons license, and indicate if changes were made. 
 
*2019 – edited by University of the People Course Development, under the terms of the license, to correct 
typos, grammar issues, and to convert British English spellings to American English. Removed For Discussion 
Questions at the end of each chapter.


PRINTED PAGE 2

The Psychology of Health and Illness: Jane Ogden 
 
 
Overview 
 
For centuries health professionals have recognized that there are psychol

In [ ]:
OGDEN_METADATA = {
    "document_id": "ogden_2019_health_psychology",
    "source_file": "ogden_2019_health_psychology.pdf",
    "title": "The Psychology of Health and Illness: An Open Access Course",
    "author": "Jane Ogden",
    "publication_year": 2019,
    "source_type": "pdf",
    "document_type": "textbook",
    "resource_type": "textbook",
    "content_role": "core_reading",
    "domain": "health_psychology",
    "language": "en",
    "access_level": "open_access",
    "license": "CC BY 4.0",
}

In [ ]:
OGDEN_UNIT_RANGES = {
    1: {
        "title": (
            "An Introduction to the Key Theoretical Frameworks "
            "of Psychology and Health"
        ),
        "page_start": 4,
        "page_end": 12,
    },
    2: {
        "title": "The Role of Behavior in Health",
        "page_start": 13,
        "page_end": 28,
    },
    3: {
        "title": "Behavior Change",
        "page_start": 29,
        "page_end": 36,
    },
    4: {
        "title": (
            "Becoming Ill and the Role of Illness Cognitions, "
            "Help-Seeking, and Communication"
        ),
        "page_start": 37,
        "page_end": 51,
    },
    5: {
        "title": "Being Ill and the Experience of Stress and Pain",
        "page_start": 52,
        "page_end": 67,
    },
    6: {
        "title": (
            "The Role of Psychology in Chronic Illnesses such as "
            "Obesity, Coronary Heart Disease, and Cancer"
        ),
        "page_start": 69,
        "page_end": 82,
    },
    7: {
        "title": "Gender, Health, and Illness",
        "page_start": 83,
        "page_end": 91,
    },
    8: {
        "title": "Health Outcomes and Quality of Life",
        "page_start": 92,
        "page_end": 99,
    },
}

In [ ]:
OGDEN_FRONT_MATTER_PAGES = [2, 3]

In [ ]:
OGDEN_EXCLUDED_PAGES = [
    1,          # cover and license page
    68,         # blank page
    *range(100, 109),  # Further Reading
]

In [ ]:
selected_pages = set(OGDEN_FRONT_MATTER_PAGES)

for unit_data in OGDEN_UNIT_RANGES.values():
    selected_pages.update(
        range(
            unit_data["page_start"],
            unit_data["page_end"] + 1
        )
    )

selected_pages = sorted(selected_pages)

print("Selected pages:", len(selected_pages))
print("First selected page:", selected_pages[0])
print("Last selected page:", selected_pages[-1])
print("Page 68 included:", 68 in selected_pages)
print("Page 100 included:", 100 in selected_pages)

Selected pages: 97
First selected page: 2
Last selected page: 99
Page 68 included: False
Page 100 included: False


In [ ]:
ogden_raw_pages = []

for printed_page in selected_pages:
    pdf_index = printed_page - 1
    page = ogden_pdf[pdf_index]

    ogden_raw_pages.append({
        "page_number": printed_page,
        "text": page.get_text("text"),
    })

print("Extracted pages:", len(ogden_raw_pages))
print("First page:", ogden_raw_pages[0]["page_number"])
print("Last page:", ogden_raw_pages[-1]["page_number"])

Extracted pages: 97
First page: 2
Last page: 99


In [ ]:
total_characters = sum(
    len(page_data["text"])
    for page_data in ogden_raw_pages
)

empty_pages = [
    page_data["page_number"]
    for page_data in ogden_raw_pages
    if not page_data["text"].strip()
]

print("Total characters:", total_characters)
print("Empty selected pages:", empty_pages)

Total characters: 158931
Empty selected pages: []


In [ ]:
import re

OGDEN_PAGE_HEADERS = {
    "The Psychology of Health and Illness: Jane Ogden",
    "The psychology of health and illness: Jane Ogden",
}


def clean_ogden_page_text(
    text: str,
    printed_page: int
) -> str:
    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        line = line.strip()

        # Прибираємо порожні рядки на цьому етапі.
        # Абзаци відновимо пізніше.
        if not line:
            continue

        # Прибираємо повторюваний header.
        if line in OGDEN_PAGE_HEADERS:
            continue

        # Прибираємо номер сторінки внизу.
        if line == str(printed_page):
            continue

        # Прибираємо службовий Unicode-артефакт,
        # який трапляється всередині слів.
        line = line.replace("￾", "")

        # Перетворюємо PDF-маркер списку на Markdown bullet.
        if line.startswith("§"):
            line = "- " + line[1:].strip()

        # Нормалізуємо зайві пробіли.
        line = re.sub(r"\s+", " ", line)

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()

In [ ]:
ogden_clean_pages = []

for page_data in ogden_raw_pages:
    printed_page = page_data["page_number"]

    cleaned_text = clean_ogden_page_text(
        page_data["text"],
        printed_page
    )

    ogden_clean_pages.append({
        "page_number": printed_page,
        "text": cleaned_text,
    })

print("Cleaned pages:", len(ogden_clean_pages))

Cleaned pages: 97


In [ ]:
pages_to_preview = [2, 3, 7, 13, 29]

for printed_page in pages_to_preview:
    page_data = next(
        item
        for item in ogden_clean_pages
        if item["page_number"] == printed_page
    )

    print(f"\n{'=' * 80}")
    print(f"CLEANED PAGE {printed_page}")
    print(f"{'=' * 80}\n")
    print(page_data["text"][:2000])


CLEANED PAGE 2

Overview
For centuries health professionals have recognized that there are psychological
consequences of being ill. A diagnosis of cancer or diabetes can make people anxious or
depressed. This course will draw upon health psychology, public health, and community
psychology to emphasize how psychology can also contribute to the cause, progression,
experience, and outcomes of any physical illness. This course will highlight the many roles
that psychology plays in physical illness from i) being and staying well and the role of health
behaviors and behavior change; ii) becoming ill with a focus on illness beliefs, symptom
perception, help-seeking and communication with health professionals; iii) being ill in terms
of stress, pain, and chronic illnesses such as obesity, coronary heart disease, and cancer; iv)
the role of gender in health, and v) health outcomes in terms of Quality of Life and
longevity.
Learning objectives and outcomes
By the end of this course students wil

In [ ]:
def is_ogden_heading(line: str) -> bool:
    line = line.strip()

    heading_patterns = [
        r"^Unit \d+[.:]",
        r"^Overview$",
        r"^Contents$",
        r"^Learning objectives and outcomes$",
        r"^About the author$",
        r"^Questions$",
        r"^To Conclude$",
        r"^In Summary$",
        r"^Summary$",
        r"^Final Take-Home Message$",
    ]

    if any(re.match(pattern, line) for pattern in heading_patterns):
        return True

    # Короткі рядки у Title Case розглядаємо як section headings.
    words = line.split()

    if (
        2 <= len(words) <= 12
        and not line.endswith((".", ",", ";", ":"))
        and not re.match(r"^\d+[.)]\s", line)
        and not line.startswith("- ")
    ):
        capitalized_words = sum(
            word[0].isupper()
            for word in words
            if word and word[0].isalpha()
        )

        if capitalized_words >= max(2, len(words) // 2):
            return True

    return False


def is_ogden_list_item(line: str) -> bool:
    return bool(
        re.match(
            r"^(?:- |\d+[.)]\s)",
            line
        )
    )


def is_ogden_figure_caption(line: str) -> bool:
    return bool(
        re.match(
            r"^(?:Fig(?:ure)?\.?\s*\d+)",
            line,
            flags=re.IGNORECASE
        )
    )

In [ ]:
def is_ogden_inline_subheading(line: str) -> bool:
    """
    Определяет короткие содержательные подзаголовки,
    которые PDF извлёк как обычный текст.
    """

    line = line.strip()

    if not line:
        return False

    if line.startswith((
        "#",
        "- ",
        "Figure ",
        "Fig. ",
    )):
        return False

    if re.match(r"^\d+[.)]\s", line):
        return False

    if line.endswith((".", "?", "!", ",", ";", ":")):
        return False

    words = line.split()

    if not 2 <= len(words) <= 12:
        return False

    if len(line) > 120:
        return False

    stopwords = {
        "a",
        "an",
        "and",
        "as",
        "at",
        "by",
        "for",
        "from",
        "in",
        "of",
        "on",
        "or",
        "the",
        "to",
        "with",
    }

    meaningful_words = []

    for word in words:
        cleaned_word = re.sub(
            r"^[^A-Za-z]+|[^A-Za-z]+$",
            "",
            word,
        )

        if (
            cleaned_word
            and cleaned_word.lower() not in stopwords
        ):
            meaningful_words.append(cleaned_word)

    if len(meaningful_words) < 2:
        return False

    capitalized_words = sum(
        word[0].isupper()
        for word in meaningful_words
    )

    return (
        capitalized_words
        / len(meaningful_words)
        >= 0.75
    )

In [ ]:
def rebuild_ogden_paragraphs(text: str) -> str:
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    blocks = []
    current_block = []
    current_type = None

    def flush_block():
        nonlocal current_block, current_type

        if current_block:
            block = " ".join(current_block)
            block = re.sub(r"\s+", " ", block).strip()
            blocks.append(block)

        current_block = []
        current_type = None

    for line in lines:

        # Звичайні заголовки
        if is_ogden_heading(line):
            flush_block()
            blocks.append(line)
            continue

        # Короткі нумеровані заголовки розділів:
        # 1. Learning Theory (with added cognitions)
        # 2. Social Cognition Theory and the Use of Planning
        if is_ogden_numbered_section_heading(line):
            flush_block()
            blocks.append(line)
            continue

        # Підписи до рисунків
        if is_ogden_figure_caption(line):
            flush_block()
            blocks.append(line)
            continue

        # Короткі підзаголовки, які займають окремий рядок
        if is_ogden_inline_subheading(line):
            flush_block()
            blocks.append(line)
            continue

        # Новий пункт списку
        if is_ogden_list_item(line):
            flush_block()
            current_block = [line]
            current_type = "list_item"
            continue

        # Продовження пункту списку
        if current_type == "list_item":
            current_block.append(line)
            continue

        # Звичайний абзац
        if current_type != "paragraph":
            flush_block()
            current_type = "paragraph"

        current_block.append(line)

    flush_block()

    return "\n\n".join(blocks).strip()

In [ ]:
ogden_rebuilt_pages = []

for page_data in ogden_clean_pages:
    rebuilt_text = rebuild_ogden_paragraphs(
        page_data["text"]
    )

    ogden_rebuilt_pages.append({
        "page_number": page_data["page_number"],
        "text": rebuilt_text,
    })

print("Rebuilt pages:", len(ogden_rebuilt_pages))

assert len(ogden_rebuilt_pages) == len(ogden_clean_pages)
assert len(ogden_rebuilt_pages) == 97

Rebuilt pages: 97


In [ ]:
pages_to_preview = [2, 29]

for printed_page in pages_to_preview:
    page_data = next(
        item
        for item in ogden_rebuilt_pages
        if item["page_number"] == printed_page
    )

    print(f"\n{'=' * 80}")
    print(f"REBUILT PAGE {printed_page}")
    print(f"{'=' * 80}\n")
    print(page_data["text"][:2500])


REBUILT PAGE 2

Overview

For centuries health professionals have recognized that there are psychological consequences of being ill. A diagnosis of cancer or diabetes can make people anxious or depressed. This course will draw upon health psychology, public health, and community psychology to emphasize how psychology can also contribute to the cause, progression, experience, and outcomes of any physical illness. This course will highlight the many roles that psychology plays in physical illness from i) being and staying well and the role of health behaviors and behavior change; ii) becoming ill with a focus on illness beliefs, symptom perception, help-seeking and communication with health professionals; iii) being ill in terms of stress, pain, and chronic illnesses such as obesity, coronary heart disease, and cancer; iv) the role of gender in health, and v) health outcomes in terms of Quality of Life and longevity.

Learning objectives and outcomes

By the end of this course students 

In [ ]:
ogden_rebuild_findings = []

for page_data in ogden_rebuilt_pages:
    page_number = page_data["page_number"]
    text = page_data["text"]

    findings = []

    if not text.strip():
        findings.append("empty_text")

    if text.startswith((
        "and ",
        "but ",
        "or ",
        "because ",
        "which ",
        "that ",
        "harmful)",
    )):
        findings.append("possible_cross_page_continuation")

    if re.search(r"\b\w+-\n\n\w+", text):
        findings.append("possible_split_hyphenated_word")

    if re.search(
        r"^Fig(?:ure)?\.?\s*\d+",
        text,
        flags=re.IGNORECASE | re.MULTILINE
    ):
        findings.append("contains_figure_caption")

    if re.search(
        r"\n\n[A-Za-z]{1,20}(?: [A-Za-z]{1,20}){2,8}\n\n",
        text
    ):
        findings.append("possible_short_layout_fragment")

    if findings:
        ogden_rebuild_findings.append({
            "page_number": page_number,
            "findings": findings,
        })

print("Pages with findings:", len(ogden_rebuild_findings))

for item in ogden_rebuild_findings:
    print(
        f'Page {item["page_number"]}: '
        f'{", ".join(item["findings"])}'
    )

Pages with findings: 34
Page 2: possible_short_layout_fragment
Page 4: possible_short_layout_fragment
Page 7: contains_figure_caption
Page 8: contains_figure_caption
Page 9: contains_figure_caption
Page 10: possible_cross_page_continuation, contains_figure_caption
Page 13: possible_short_layout_fragment
Page 14: possible_short_layout_fragment
Page 16: possible_short_layout_fragment
Page 18: contains_figure_caption
Page 20: contains_figure_caption
Page 22: contains_figure_caption
Page 24: possible_short_layout_fragment
Page 29: possible_short_layout_fragment
Page 30: possible_cross_page_continuation
Page 39: contains_figure_caption
Page 45: possible_short_layout_fragment
Page 46: possible_cross_page_continuation
Page 47: contains_figure_caption
Page 53: possible_short_layout_fragment
Page 54: contains_figure_caption
Page 55: contains_figure_caption
Page 56: contains_figure_caption
Page 62: possible_split_hyphenated_word, contains_figure_caption
Page 70: possible_split_hyphenated_word
Pa

In [ ]:
pages_to_inspect = [10, 30, 46, 62, 70, 80]

for current_page_number in pages_to_inspect:
    current_index = next(
        index
        for index, page_data in enumerate(ogden_rebuilt_pages)
        if page_data["page_number"] == current_page_number
    )

    current_page = ogden_rebuilt_pages[current_index]

    previous_page = (
        ogden_rebuilt_pages[current_index - 1]
        if current_index > 0
        else None
    )

    print(f"\n{'=' * 90}")
    print(f"BOUNDARY BEFORE PAGE {current_page_number}")
    print(f"{'=' * 90}")

    if previous_page is not None:
        print(
            f'\nEND OF PAGE {previous_page["page_number"]}:\n'
        )
        print(previous_page["text"][-700:])

    print(
        f'\nSTART OF PAGE {current_page_number}:\n'
    )
    print(current_page["text"][:700])


BOUNDARY BEFORE PAGE 10

END OF PAGE 9:

f health outcomes (“I have cancer and therefore will die”). Health psychology, however, argues that there is much more variability between people than this and this variability is our focus. For example, two people might both know that smoking is bad for them but only one stops smoking. Similarly, two people might find a lump in their breast but only one goes to the doctor. Further, two people might both have a heart attack but whilst one has another in 6 months time, the other is perfectly healthy and back to work within a month. This variability indicates that health and illness cannot only be explained by illness severity (type of cancer, severity of heart attack) or knowledge (smoking is

START OF PAGE 10:

harmful) but that other factors must have a key role to play. For a health psychologist, these factors include a wide range of psychological variables such as cognitions, emotions, expectations, learning, peer pressure, social norms, cop

In [ ]:
def repair_ogden_hyphen_line_breaks(text: str) -> str:
    """
    Прибирає помилковий розрив абзацу після дефіса.

    Приклади:
    stimulus-

    response
    -> stimulus-response

    18.5-

    24.9
    -> 18.5-24.9

    Сам дефіс зберігається.
    """

    result = re.sub(
        r"(?<=\w)-\s*\n\s*\n\s*(?=\w)",
        "-",
        text
    )

    result = re.sub(
        r"\n{3,}",
        "\n\n",
        result
    )

    return result.strip()

In [ ]:
ogden_repaired_pages = []

for page_data in ogden_rebuilt_pages:
    repaired_text = repair_ogden_hyphen_line_breaks(
        page_data["text"]
    )

    ogden_repaired_pages.append({
        "page_number": page_data["page_number"],
        "text": repaired_text,
    })

print("Repaired pages:", len(ogden_repaired_pages))

assert len(ogden_repaired_pages) == 97
assert len(ogden_repaired_pages) == len(ogden_rebuilt_pages)

Repaired pages: 97


In [ ]:
FIGURE_PREFIX_RE = re.compile(
    r"^Fig(?:ure)?\.?\s*(\d+)\b[:.]?",
    flags=re.IGNORECASE,
)


def is_short_title_like(text: str) -> bool:
    """
    Перевіряє, чи схожий рядок на коротку назву,
    а не на звичайне речення.
    """

    text = text.strip()

    if not text:
        return False

    words = text.split()

    if not 2 <= len(words) <= 18:
        return False

    # Повне речення зазвичай містить кілька речень
    # або закінчується крапкою.
    if text.endswith((".", "?", "!")):
        return False

    # Не беремо довгі sentence-like fragments.
    if len(text) > 160:
        return False

    return True


def extract_ogden_figure_candidates(
    page_number: int,
    text: str
) -> list[dict]:
    """
    Універсально знаходить імовірні captions рисунків.

    Підтримує:
    - повний caption в одному рядку;
    - номер рисунка в одному блоці, а назву — у наступному;
    - відкидає in-text references типу:
      'Figure 4. It suggested that...'
    """

    lines = text.splitlines()
    candidates = []

    for index, raw_line in enumerate(lines):
        line = raw_line.strip()

        if not line:
            continue

        match = FIGURE_PREFIX_RE.match(line)

        if not match:
            continue

        figure_number = int(match.group(1))

        remainder = line[match.end():].strip(" :.-")

        # Варіант 1: caption повністю міститься в одному рядку.
        if remainder and is_short_title_like(remainder):
            candidates.append({
                "page_number": page_number,
                "figure_number": figure_number,
                "raw_start_index": index,
                "raw_end_index": index,
                "raw_text": line,
                "caption": line,
                "candidate_type": "single_line",
            })
            continue

        # Якщо після Figure N іде довге речення,
        # це in-text reference, а не caption.
        if remainder:
            continue

        # Варіант 2: рядок містить тільки Figure N.
        # Шукаємо наступний непорожній короткий title-like рядок.
        next_nonempty_index = None

        for next_index in range(
            index + 1,
            min(index + 5, len(lines))
        ):
            next_line = lines[next_index].strip()

            if next_line:
                next_nonempty_index = next_index
                break

        if next_nonempty_index is None:
            continue

        title_line = lines[next_nonempty_index].strip()

        if not is_short_title_like(title_line):
            continue

        canonical_caption = (
            f"Figure {figure_number} {title_line}"
        )

        candidates.append({
            "page_number": page_number,
            "figure_number": figure_number,
            "raw_start_index": index,
            "raw_end_index": next_nonempty_index,
            "raw_text": (
                f"{line}\n\n{title_line}"
            ),
            "caption": canonical_caption,
            "candidate_type": "split_caption",
        })

    return candidates

In [ ]:
ogden_figure_candidates = []

for page_data in ogden_repaired_pages:
    page_candidates = extract_ogden_figure_candidates(
        page_number=page_data["page_number"],
        text=page_data["text"],
    )

    ogden_figure_candidates.extend(
        page_candidates
    )

print(
    "Figure candidates found:",
    len(ogden_figure_candidates)
)

for candidate in ogden_figure_candidates:
    print(
        f'Page {candidate["page_number"]}, '
        f'Figure {candidate["figure_number"]}, '
        f'{candidate["candidate_type"]}: '
        f'{candidate["caption"]}'
    )

Figure candidates found: 15
Page 7, Figure 1, single_line: Fig 1 The biopsychosocial model of health and illness (after Engel 1977, 1980)
Page 8, Figure 2, single_line: Fig 2: Health and Illness as a Continuum
Page 9, Figure 3, single_line: Fig. 3 The Direct and Indirect Pathways between Psychology and Health
Page 10, Figure 4, single_line: Fig 4: The focus on variability
Page 18, Figure 1, single_line: Figure 1 Basics of the Health Belief Model
Page 20, Figure 2, single_line: Figure 2 Basics of the Protection Motivation Theory
Page 22, Figure 3, single_line: Figure 3 Basics of the Theory of Planned Behavior
Page 39, Figure 1, single_line: Figure 1 Leventhal’s Self Regulatory Model of Illness Behavior (SRM)
Page 47, Figure 2, single_line: Figure 2 Diagnoses as a Form of Problem-Solving
Page 54, Figure 1, single_line: Figure 1 The Role of Appraisal in Stress
Page 55, Figure 2, single_line: Figure 2: The Stress Illness Link: Direct and Indirect Pathways
Page 56, Figure 3, single_line: Fi

In [ ]:
OGDEN_FIGURE_ASSETS = {
    (7, 1): "assets/ogden_p7_figure_1.png",
    (8, 2): "assets/ogden_p8_figure_2.png",
    (9, 3): "assets/ogden_p9_figure_3.png",
    (10, 4): "assets/ogden_p10_figure_4.png",
    (22, 3): "assets/ogden_p22_figure_3.png",
    (39, 1): "assets/ogden_p39_figure_1.png",
    (54, 1): "assets/ogden_p54_figure_1.png",
    (55, 2): "assets/ogden_p55_figure_2.png",
}

In [ ]:
OGDEN_FIGURE_DESCRIPTIONS = {
    (7, 1): (
        "The biopsychosocial model explains health and illness through "
        "the interaction of biological, psychological, and social factors."
    ),

    (8, 2): (
        "Health and illness are presented as a continuum. Psychological "
        "factors influence illness onset, help-seeking, adaptation, "
        "illness progression, and health outcomes."
    ),

    (9, 3): (
        "The direct pathway shows psychological states influencing health "
        "through physiological changes. The indirect pathway shows "
        "psychological states influencing behavior, which then affects health."
    ),

    (10, 4): (
        "The model highlights variability between people in knowledge, "
        "behavior, illness progression, and health outcomes."
    ),

    (18, 1): (
        "The Health Belief Model links perceived susceptibility, severity, "
        "benefits, costs, cues to action, health motivation, and perceived "
        "control to health behavior."
    ),

    (20, 2): (
        "Protection Motivation Theory proposes that severity, susceptibility, "
        "fear, response effectiveness, and self-efficacy influence behavioral "
        "intentions and health behavior."
    ),

    (22, 3): (
        "The Theory of Planned Behavior proposes that attitudes, subjective "
        "norms, and perceived behavioral control influence behavioral "
        "intentions. Intentions and perceived behavioral control influence behavior."
    ),

    (39, 1): (
        "Leventhal's Self-Regulatory Model describes illness management as "
        "a cycle of interpretation, coping, and appraisal. Illness beliefs "
        "and emotional responses influence coping strategies."
    ),

    (47, 2): (
        "Clinical diagnosis is presented as a problem-solving process involving "
        "information gathering, hypothesis development, searching for confirming "
        "or disconfirming attributes, and making a management decision."
    ),

    (54, 1): (
        "The appraisal model of stress distinguishes primary appraisal of the "
        "event from secondary appraisal of coping resources. These appraisals "
        "shape the stress response."
    ),

    (55, 2): (
        "Stress can affect illness directly through physiological changes and "
        "indirectly through changes in health behaviors."
    ),

    (56, 3): (
        "The relationship between stress and illness is moderated by coping, "
        "social support, personality, and perceived control."
    ),

    (62, 4): (
        "The Gate Control Theory of Pain proposes that pain perception is "
        "regulated by a gate influenced by physiological input and psychological "
        "factors such as attention, mood, and previous experience."
    ),

    (84, 1): (
        "The figure compares gender differences in disease, showing conditions "
        "reported more often among women, conditions reported more often among "
        "men, and conditions with little or no gender difference."
    ),

    (85, 2): (
        "The figure compares gender differences in physical symptoms, including "
        "symptoms reported more often by women and symptoms showing little or "
        "no gender difference."
    ),
}

In [ ]:
OGDEN_FIGURES = []

for candidate in ogden_figure_candidates:
    figure_key = (
        candidate["page_number"],
        candidate["figure_number"],
    )

    description = OGDEN_FIGURE_DESCRIPTIONS.get(
        figure_key
    )

    OGDEN_FIGURES.append({
        **candidate,
        "figure_id": (
            f'ogden_p{candidate["page_number"]}_'
            f'fig{candidate["figure_number"]}'
        ),
        "description": description,
    })

print("Registered figures:", len(OGDEN_FIGURES))

Registered figures: 15


In [ ]:
missing_descriptions = [
    figure["figure_id"]
    for figure in OGDEN_FIGURES
    if not figure["description"]
]

duplicate_figure_ids = sorted({
    figure["figure_id"]
    for figure in OGDEN_FIGURES
    if sum(
        item["figure_id"] == figure["figure_id"]
        for item in OGDEN_FIGURES
    ) > 1
})

print("Figures:", len(OGDEN_FIGURES))
print("Missing descriptions:", missing_descriptions)
print("Duplicate figure IDs:", duplicate_figure_ids)

assert len(OGDEN_FIGURES) == 15
assert missing_descriptions == []
assert duplicate_figure_ids == []

Figures: 15
Missing descriptions: []
Duplicate figure IDs: []


In [ ]:
def enrich_ogden_figures_fail_closed(
    page_number: int,
    text: str,
    figures: list[dict],
) -> tuple[str, list[dict]]:
    """
    Вставляє Markdown-блоки для рисунків.

    Працює fail-closed:
    - якщо raw caption знайдено рівно один раз — збагачує;
    - якщо не знайдено або знайдено кілька разів — не змінює текст
      і додає finding для ручного review.
    """

    result = text
    findings = []

    page_figures = [
        figure
        for figure in figures
        if figure["page_number"] == page_number
    ]

    for figure in page_figures:
        raw_text = figure["raw_text"]
        caption = figure["caption"]
        description = figure["description"]

        match_count = result.count(raw_text)

        if match_count == 1:
            figure_block = (
                f"#### {caption}\n\n"
                f"Figure description: {description}"
            )

            result = result.replace(
                raw_text,
                figure_block,
                1
            )

            findings.append({
                "figure_id": figure["figure_id"],
                "page_number": page_number,
                "status": "enriched_exact",
                "match_count": match_count,
            })

        else:
            findings.append({
                "figure_id": figure["figure_id"],
                "page_number": page_number,
                "status": "requires_review",
                "match_count": match_count,
                "raw_text": raw_text,
            })

    result = re.sub(
        r"\n{3,}",
        "\n\n",
        result
    )

    return result.strip(), findings

In [ ]:
ogden_enriched_pages = []
ogden_figure_enrichment_findings = []

for page_data in ogden_repaired_pages:
    enriched_text, findings = enrich_ogden_figures_fail_closed(
        page_number=page_data["page_number"],
        text=page_data["text"],
        figures=OGDEN_FIGURES,
    )

    ogden_enriched_pages.append({
        "page_number": page_data["page_number"],
        "text": enriched_text,
    })

    ogden_figure_enrichment_findings.extend(
        findings
    )

print("Enriched pages:", len(ogden_enriched_pages))
print(
    "Figure findings:",
    len(ogden_figure_enrichment_findings)
)

assert len(ogden_enriched_pages) == 97

Enriched pages: 97
Figure findings: 15


In [ ]:
status_counts = {}

for finding in ogden_figure_enrichment_findings:
    status = finding["status"]

    status_counts[status] = (
        status_counts.get(status, 0) + 1
    )

print("Status counts:", status_counts)

for finding in ogden_figure_enrichment_findings:
    if finding["status"] != "enriched_exact":
        print(finding)

Status counts: {'enriched_exact': 15}


In [ ]:
pages_to_preview = [10, 22, 62]

for printed_page in pages_to_preview:
    page_data = next(
        item
        for item in ogden_enriched_pages
        if item["page_number"] == printed_page
    )

    print(f"\n{'=' * 80}")
    print(f"ENRICHED PAGE {printed_page}")
    print(f"{'=' * 80}\n")
    print(page_data["text"][:3000])


ENRICHED PAGE 10

harmful) but that other factors must have a key role to play. For a health psychologist, these factors include a wide range of psychological variables such as cognitions, emotions, expectations, learning, peer pressure, social norms, coping, and social support. These constructs are the nuts and bolts of psychology and are covered in the units in this book. The notion of variability is shown in Figure 4.

#### Fig 4: The focus on variability

Figure description: The model highlights variability between people in knowledge, behavior, illness progression, and health outcomes.

Knowledge behavior Variability illness outcome The 4 key theoretical frameworks form the basis of health psychology and reflect the emphasis on psychology as having a role at all stages of being healthy and becoming ill. These frameworks can be illustrated by the case example of Mr. A.

Mr. A: The example of lung cancer

Mr. A grew up in a poor area of India. Both of his parents smoked because it 

In [ ]:
ogden_merged_bullet_findings = []

for page_data in ogden_enriched_pages:
    page_number = page_data["page_number"]

    for block_index, block in enumerate(
        page_data["text"].split("\n\n")
    ):
        block = block.strip()

        # Блок починається як bullet, але всередині
        # містить ще один PDF/Markdown bullet.
        internal_bullets = list(
            re.finditer(
                r"(?<=\.)\s+-\s+(?=[A-Z])",
                block
            )
        )

        if block.startswith("- ") and internal_bullets:
            ogden_merged_bullet_findings.append({
                "page_number": page_number,
                "block_index": block_index,
                "internal_bullet_count": len(
                    internal_bullets
                ),
                "text": block,
            })

print(
    "Merged bullet blocks:",
    len(ogden_merged_bullet_findings)
)

for finding in ogden_merged_bullet_findings:
    print(f"\n{'=' * 80}")
    print(
        f'PAGE {finding["page_number"]}, '
        f'BLOCK {finding["block_index"]}, '
        f'INTERNAL BULLETS: '
        f'{finding["internal_bullet_count"]}'
    )
    print(f"{'=' * 80}\n")
    print(finding["text"][:2000])

Merged bullet blocks: 2

PAGE 22, BLOCK 4, INTERNAL BULLETS: 1

- Attitude towards a behavior, which is composed of either a positive or negative evaluation of a particular behavior and beliefs about the outcome of the behavior (e.g. “exercising is fun and will improve my health”). - Subjective norm, which is composed of the perception of social norms and pressures to perform a behavior and an evaluation of whether the individual is motivated to

PAGE 61, BLOCK 2, INTERNAL BULLETS: 1

- It was observed that medical treatments for pain (for example drugs and surgery) were useful only for treating acute pain (pain of short duration). Such treatments were fairly ineffective for treating chronic pain (pain which lasts for a long time). This suggested that there must be something else involved in the pain experience that was not included in the simple stimulus-response models. - It was also observed that individuals with the same degree of tissue damage differed in their reporting of the pa

In [ ]:
def repair_ogden_merged_bullets(text: str) -> str:
    """
    Розділяє кілька bullet items, які PDF склеїв
    в один текстовий блок.

    Приклад:
    - First item. - Second item.

    перетворюється на:

    - First item.

    - Second item.
    """

    result = re.sub(
        r"(?<=\.)\s+-\s+(?=[A-Z])",
        "\n\n- ",
        text
    )

    result = re.sub(
        r"\n{3,}",
        "\n\n",
        result
    )

    return result.strip()

In [ ]:
ogden_bullet_repaired_pages = []

for page_data in ogden_enriched_pages:
    repaired_text = repair_ogden_merged_bullets(
        page_data["text"]
    )

    ogden_bullet_repaired_pages.append({
        "page_number": page_data["page_number"],
        "text": repaired_text,
    })

print(
    "Bullet-repaired pages:",
    len(ogden_bullet_repaired_pages)
)

assert len(ogden_bullet_repaired_pages) == 97
assert (
    len(ogden_bullet_repaired_pages)
    == len(ogden_enriched_pages)
)

Bullet-repaired pages: 97


In [ ]:
for printed_page in [22, 61]:
    page_data = next(
        item
        for item in ogden_bullet_repaired_pages
        if item["page_number"] == printed_page
    )

    print(f"\n{'=' * 80}")
    print(f"BULLET-REPAIRED PAGE {printed_page}")
    print(f"{'=' * 80}\n")
    print(page_data["text"][:3000])


BULLET-REPAIRED PAGE 22

behaviors and was central to the debate within social psychology concerning the relationship between attitudes and behavior (Fishbein and Ajzen 1975). The Theory of Planned Behavior (TPB) (see Figure 3) was developed by Ajzen and colleagues (Ajzen and Madden 1986) and represented a progression from the TRA.

#### Figure 3 Basics of the Theory of Planned Behavior

Figure description: The Theory of Planned Behavior proposes that attitudes, subjective norms, and perceived behavioral control influence behavioral intentions. Intentions and perceived behavioral control influence behavior.

The TPB emphasizes behavioral intentions as the outcome of a combination of several beliefs. The theory proposes that intentions should be conceptualized as “plans of action in pursuit of behavioral goals” (Ajzen and Madden 1986) and are a result of the following beliefs:

- Attitude towards a behavior, which is composed of either a positive or negative evaluation of a particular 

In [ ]:
remaining_merged_bullet_findings = []

for page_data in ogden_bullet_repaired_pages:
    page_number = page_data["page_number"]

    for block_index, block in enumerate(
        page_data["text"].split("\n\n")
    ):
        block = block.strip()

        internal_bullets = list(
            re.finditer(
                r"(?<=\.)\s+-\s+(?=[A-Z])",
                block
            )
        )

        if block.startswith("- ") and internal_bullets:
            remaining_merged_bullet_findings.append({
                "page_number": page_number,
                "block_index": block_index,
                "internal_bullet_count": len(
                    internal_bullets
                ),
                "text": block,
            })

print(
    "Remaining merged bullet blocks:",
    len(remaining_merged_bullet_findings)
)

for finding in remaining_merged_bullet_findings:
    print(finding)

Remaining merged bullet blocks: 0


In [ ]:
cross_page_candidates = []

for index in range(1, len(ogden_bullet_repaired_pages)):
    previous_page = ogden_bullet_repaired_pages[index - 1]
    current_page = ogden_bullet_repaired_pages[index]

    previous_text = previous_page["text"].strip()
    current_text = current_page["text"].strip()

    if not previous_text or not current_text:
        continue

    previous_last_block = previous_text.split("\n\n")[-1].strip()
    current_first_block = current_text.split("\n\n")[0].strip()

    previous_ends_sentence = bool(
        re.search(r'[.!?]["”’\']?$', previous_last_block)
    )

    current_starts_like_continuation = bool(
    re.match(
        r"^(?:"
        r"[a-z]|"
        r"and\b|"
        r"but\b|"
        r"or\b|"
        r"because\b|"
        r"which\b|"
        r"that\b|"
        r"of\b|"
        r"to\b|"
        r"from\b|"
        r"with\b|"
        r"between\b|"
        r"through\b|"
        r"depending\b|"
        r"relapse\b|"
        r"comply\b|"
        r"action\b|"
        r"indirect\b|"
        r"adaptive\b|"
        r"factors\b|"
        r"environment\b|"
        r"outcomes\b|"
        r"originate\b|"
        r"defined\b|"
        r"outcome\b|"
        r"may\b"
        r")",
        current_first_block
    )
)

    current_starts_new_structure = bool(
        re.match(
            r"^(?:"
            r"#|"
            r"-\s|"
            r"Unit\s+\d+[.:]|"
            r"Overview$|"
            r"Contents$|"
            r"Questions$|"
            r"Summary$|"
            r"Figure\s+\d+|"
            r"Fig\.?\s*\d+"
            r")",
            current_first_block,
            flags=re.IGNORECASE
        )
    )

    if (
        not previous_ends_sentence
        and current_starts_like_continuation
        and not current_starts_new_structure
    ):
        cross_page_candidates.append({
            "previous_page": previous_page["page_number"],
            "current_page": current_page["page_number"],
            "previous_last_block": previous_last_block,
            "current_first_block": current_first_block,
        })

print(
    "Cross-page continuation candidates:",
    len(cross_page_candidates)
)

for candidate in cross_page_candidates:
    print(f"\n{'=' * 80}")
    print(
        f'PAGE {candidate["previous_page"]} '
        f'→ PAGE {candidate["current_page"]}'
    )
    print(f"{'=' * 80}\n")

    print("END OF PREVIOUS PAGE:\n")
    print(candidate["previous_last_block"][-600:])

    print("\nSTART OF CURRENT PAGE:\n")
    print(candidate["current_first_block"][:600])

Cross-page continuation candidates: 36

PAGE 5 → PAGE 6

END OF PREVIOUS PAGE:

- What causes illness? Health psychology suggests that human beings should be seen as complex systems and that illness is caused by a multitude of factors and not by a single causal factor. Health psychology, therefore, attempts to move away from a simple linear model of health and claims that illness can be caused by a combination

START OF CURRENT PAGE:

of biological (e.g. a virus), psychological (e.g. behaviors, beliefs) and social (e.g. social support) factors.

PAGE 8 → PAGE 9

END OF PREVIOUS PAGE:

Health psychologists consider both a direct and indirect pathway between psychology and health. The direct pathway is reflected in the physiological literature and from this perspective, the way a person experiences their life (“I am feeling stressed”) has a direct impact upon their body through changes in their physiology which can change their health status. The indirect pathway is reflected more in the

In [ ]:
def get_ogden_unit_number(page_number: int) -> int | None:
    """
    Повертає номер unit для сторінки.
    Для Course Overview and Contents повертає None.
    """

    for unit_number, unit_data in OGDEN_UNIT_RANGES.items():
        if (
            unit_data["page_start"]
            <= page_number
            <= unit_data["page_end"]
        ):
            return unit_number

    return None


cross_page_candidates_with_units = []

for candidate in cross_page_candidates:
    previous_page = candidate["previous_page"]
    current_page = candidate["current_page"]

    previous_unit = get_ogden_unit_number(
        previous_page
    )

    current_unit = get_ogden_unit_number(
        current_page
    )

    cross_page_candidates_with_units.append({
        **candidate,
        "previous_unit": previous_unit,
        "current_unit": current_unit,
        "same_unit": previous_unit == current_unit,
    })


cross_unit_candidates = [
    candidate
    for candidate in cross_page_candidates_with_units
    if not candidate["same_unit"]
]


print(
    "Cross-page candidates:",
    len(cross_page_candidates_with_units)
)

print(
    "Candidates inside the same unit:",
    sum(
        candidate["same_unit"]
        for candidate in cross_page_candidates_with_units
    )
)

print(
    "Candidates crossing unit boundaries:",
    len(cross_unit_candidates)
)


for candidate in cross_unit_candidates:
    print(
        f'Page {candidate["previous_page"]} '
        f'(Unit {candidate["previous_unit"]}) '
        f'→ Page {candidate["current_page"]} '
        f'(Unit {candidate["current_unit"]})'
    )

Cross-page candidates: 36
Candidates inside the same unit: 36
Candidates crossing unit boundaries: 0


In [ ]:
cross_page_pairs = {
    (
        candidate["previous_page"],
        candidate["current_page"],
    )
    for candidate in cross_page_candidates_with_units
    if candidate["same_unit"]
}

print("Approved cross-page pairs:", len(cross_page_pairs))

assert len(cross_page_pairs) == 36

Approved cross-page pairs: 36


In [ ]:
def build_ogden_unit_blocks(
    pages: list[dict],
    unit_number: int,
    unit_data: dict,
    approved_cross_page_pairs: set[tuple[int, int]],
) -> list[dict]:
    """
    Збирає текстові блоки однієї unit.

    Якщо останній блок попередньої сторінки та перший блок
    наступної сторінки утворюють підтверджене продовження,
    вони склеюються.

    Для кожного блока зберігаються:
    - unit_number;
    - unit_title;
    - page_start;
    - page_end;
    - text.
    """

    unit_pages = [
        page_data
        for page_data in pages
        if (
            unit_data["page_start"]
            <= page_data["page_number"]
            <= unit_data["page_end"]
        )
    ]

    unit_pages = sorted(
        unit_pages,
        key=lambda item: item["page_number"]
    )

    unit_blocks = []

    for page_index, page_data in enumerate(unit_pages):
        page_number = page_data["page_number"]

        page_blocks = [
            block.strip()
            for block in page_data["text"].split("\n\n")
            if block.strip()
        ]

        if not page_blocks:
            continue

        previous_page_number = (
            unit_pages[page_index - 1]["page_number"]
            if page_index > 0
            else None
        )

        should_merge_boundary = (
            previous_page_number is not None
            and (
                previous_page_number,
                page_number,
            ) in approved_cross_page_pairs
            and bool(unit_blocks)
        )

        if should_merge_boundary:
            first_block = page_blocks.pop(0)

            unit_blocks[-1]["text"] = (
                f'{unit_blocks[-1]["text"]} '
                f'{first_block}'
            )

            unit_blocks[-1]["text"] = re.sub(
                r"\s+",
                " ",
                unit_blocks[-1]["text"]
            ).strip()

            unit_blocks[-1]["page_end"] = page_number

        for block in page_blocks:
            unit_blocks.append({
                "unit_number": unit_number,
                "unit_title": unit_data["title"],
                "page_start": page_number,
                "page_end": page_number,
                "text": block,
            })

    return unit_blocks

In [ ]:
ogden_unit_blocks = {}

for unit_number, unit_data in OGDEN_UNIT_RANGES.items():
    blocks = build_ogden_unit_blocks(
        pages=ogden_bullet_repaired_pages,
        unit_number=unit_number,
        unit_data=unit_data,
        approved_cross_page_pairs=cross_page_pairs,
    )

    ogden_unit_blocks[unit_number] = blocks

print("Units assembled:", len(ogden_unit_blocks))

for unit_number, blocks in ogden_unit_blocks.items():
    cross_page_blocks = [
        block
        for block in blocks
        if block["page_start"] != block["page_end"]
    ]

    print(
        f"Unit {unit_number}: "
        f"{len(blocks)} blocks, "
        f"{len(cross_page_blocks)} cross-page blocks"
    )

assert len(ogden_unit_blocks) == 8
assert all(
    len(blocks) > 0
    for blocks in ogden_unit_blocks.values()
)

Units assembled: 8
Unit 1: 50 blocks, 4 cross-page blocks
Unit 2: 68 blocks, 5 cross-page blocks
Unit 3: 21 blocks, 2 cross-page blocks
Unit 4: 56 blocks, 8 cross-page blocks
Unit 5: 59 blocks, 6 cross-page blocks
Unit 6: 48 blocks, 6 cross-page blocks
Unit 7: 36 blocks, 0 cross-page blocks
Unit 8: 23 blocks, 3 cross-page blocks


In [ ]:
multi_page_blocks = []

for unit_number, blocks in ogden_unit_blocks.items():
    for block in blocks:
        page_span = (
            block["page_end"]
            - block["page_start"]
            + 1
        )

        if page_span > 2:
            multi_page_blocks.append({
                "unit_number": unit_number,
                "page_start": block["page_start"],
                "page_end": block["page_end"],
                "page_span": page_span,
                "text": block["text"],
            })

print(
    "Blocks spanning more than 2 pages:",
    len(multi_page_blocks)
)

for block in multi_page_blocks:
    print(
        f'Unit {block["unit_number"]}: '
        f'pages {block["page_start"]}'
        f'–{block["page_end"]}, '
        f'span={block["page_span"]}'
    )

Blocks spanning more than 2 pages: 2
Unit 3: pages 29–31, span=3
Unit 3: pages 32–34, span=3


In [ ]:
covered_cross_page_pairs = set()

for blocks in ogden_unit_blocks.values():
    for block in blocks:
        page_start = block["page_start"]
        page_end = block["page_end"]

        if page_start == page_end:
            continue

        # Розгортаємо багатосторінковий block
        # у всі сусідні межі, які він охоплює.
        for page_number in range(
            page_start,
            page_end
        ):
            covered_cross_page_pairs.add(
                (
                    page_number,
                    page_number + 1,
                )
            )

missing_merged_pairs = sorted(
    cross_page_pairs
    - covered_cross_page_pairs
)

unexpected_merged_pairs = sorted(
    covered_cross_page_pairs
    - cross_page_pairs
)

print(
    "Covered cross-page pairs:",
    len(covered_cross_page_pairs)
)

print(
    "Expected approved pairs:",
    len(cross_page_pairs)
)

print(
    "Missing approved pairs:",
    missing_merged_pairs
)

print(
    "Unexpected merged pairs:",
    unexpected_merged_pairs
)

assert covered_cross_page_pairs == cross_page_pairs

Covered cross-page pairs: 36
Expected approved pairs: 36
Missing approved pairs: []
Unexpected merged pairs: []


In [ ]:
ogden_front_matter_blocks = []

for page_data in ogden_bullet_repaired_pages:
    page_number = page_data["page_number"]

    if page_number not in OGDEN_FRONT_MATTER_PAGES:
        continue

    page_blocks = [
        block.strip()
        for block in page_data["text"].split("\n\n")
        if block.strip()
    ]

    for block in page_blocks:
        ogden_front_matter_blocks.append({
            "unit_number": None,
            "unit_title": "Course Overview and Contents",
            "page_start": page_number,
            "page_end": page_number,
            "text": block,
        })

print(
    "Course Overview and Contents blocks:",
    len(ogden_front_matter_blocks)
)

assert len(ogden_front_matter_blocks) > 0

Course Overview and Contents blocks: 25


In [ ]:
ogden_all_blocks = []

ogden_all_blocks.extend(
    ogden_front_matter_blocks
)

for unit_number in sorted(ogden_unit_blocks):
    ogden_all_blocks.extend(
        ogden_unit_blocks[unit_number]
    )

print("All Ogden blocks:", len(ogden_all_blocks))

assert len(ogden_all_blocks) > 0
assert all(
    block["text"].strip()
    for block in ogden_all_blocks
)

All Ogden blocks: 386


In [ ]:
covered_pages = set()

for block in ogden_all_blocks:
    covered_pages.update(
        range(
            block["page_start"],
            block["page_end"] + 1
        )
    )

missing_selected_pages = sorted(
    set(selected_pages) - covered_pages
)

unexpected_pages = sorted(
    covered_pages - set(selected_pages)
)

print("Covered pages:", len(covered_pages))
print("Selected pages:", len(selected_pages))
print("Missing selected pages:", missing_selected_pages)
print("Unexpected pages:", unexpected_pages)

assert missing_selected_pages == []
assert unexpected_pages == []

Covered pages: 97
Selected pages: 97
Missing selected pages: []
Unexpected pages: []


In [ ]:
ogden_normalized_markdown = build_ogden_normalized_markdown(
    metadata=OGDEN_METADATA,
    front_matter_blocks=ogden_front_matter_blocks,
    unit_blocks=ogden_unit_blocks,
)

OGDEN_NORMALIZED_MD_PATH.write_text(
    ogden_normalized_markdown,
    encoding="utf-8"
)

print(
    "Saved:",
    OGDEN_NORMALIZED_MD_PATH
)

print(
    "Characters:",
    len(ogden_normalized_markdown)
)

print(
    "File size:",
    OGDEN_NORMALIZED_MD_PATH.stat().st_size
)

Saved: /content/health-psychology-rag-kb/data/normalized/ogden_2019_health_psychology.md
Characters: 160856
File size: 162592


In [ ]:
print("=" * 80)
print("START OF NORMALIZED MARKDOWN")
print("=" * 80)
print(ogden_normalized_markdown[:3000])

print("\n" + "=" * 80)
print("END OF NORMALIZED MARKDOWN")
print("=" * 80)
print(ogden_normalized_markdown[-3000:])

START OF NORMALIZED MARKDOWN
# The Psychology of Health and Illness: An Open Access Course

- Author: Jane Ogden
- Publication year: 2019
- Document type: textbook
- Content role: core_reading
- Domain: health_psychology
- Language: en
- Access level: open_access
- License: CC BY 4.0
- Source file: ogden_2019_health_psychology.pdf

## Front Matter

<!-- pages: 2–2 -->

Overview

<!-- pages: 2–2 -->

For centuries health professionals have recognized that there are psychological consequences of being ill. A diagnosis of cancer or diabetes can make people anxious or depressed. This course will draw upon health psychology, public health, and community psychology to emphasize how psychology can also contribute to the cause, progression, experience, and outcomes of any physical illness. This course will highlight the many roles that psychology plays in physical illness from i) being and staying well and the role of health behaviors and behavior change; ii) becoming ill with a focus on illne

In [ ]:
figure_heading_matches = re.findall(
    r"^#### (?:Figure|Fig\.?)\s+\d+.*$",
    ogden_normalized_markdown,
    flags=re.MULTILINE
)

figure_description_matches = re.findall(
    r"^Figure description:.*$",
    ogden_normalized_markdown,
    flags=re.MULTILINE
)

print(
    "Figure headings in Markdown:",
    len(figure_heading_matches)
)

print(
    "Figure descriptions in Markdown:",
    len(figure_description_matches)
)

assert len(figure_heading_matches) == 15
assert len(figure_description_matches) == 15

Figure headings in Markdown: 15
Figure descriptions in Markdown: 15


In [ ]:
cleanup_audit = {
    "duplicate_unit_titles": [],
    "inline_bullet_blocks": [],
    "possible_same_page_splits": [],
}


def normalize_for_comparison(text: str) -> str:
    return re.sub(
        r"\s+",
        " ",
        text
    ).strip().lower()


def looks_like_heading(text: str) -> bool:
    text = text.strip()

    return bool(
        re.match(
            r"^(?:"
            r"#|"
            r"Unit\s+\d+[\.:]|"
            r"Overview$|"
            r"Contents$|"
            r"Questions$|"
            r"Summary$|"
            r"In Summary$|"
            r"To Conclude$|"
            r"Final Take-Home Message$|"
            r"####\s+(?:Figure|Fig\.?)"
            r")",
            text,
            flags=re.IGNORECASE
        )
    )


# 1. Дубльовані назви Units
for unit_number, blocks in ogden_unit_blocks.items():
    expected_title = (
        f"Unit {unit_number}: "
        f'{OGDEN_UNIT_RANGES[unit_number]["title"]}'
    )

    expected_normalized = normalize_for_comparison(
        expected_title
    )

    for block_index, block in enumerate(blocks[:5]):
        block_normalized = normalize_for_comparison(
            block["text"]
        )

        if (
            block_normalized.startswith(
                f"unit {unit_number}"
            )
            and (
                expected_normalized in block_normalized
                or block_normalized in expected_normalized
            )
        ):
            cleanup_audit[
                "duplicate_unit_titles"
            ].append({
                "unit_number": unit_number,
                "block_index": block_index,
                "page_start": block["page_start"],
                "page_end": block["page_end"],
                "text": block["text"],
            })


# 2. Списки, склеєні символом • усередині абзацу
for block_index, block in enumerate(ogden_all_blocks):
    bullet_count = block["text"].count("•")

    if bullet_count > 0:
        cleanup_audit[
            "inline_bullet_blocks"
        ].append({
            "block_index": block_index,
            "unit_number": block["unit_number"],
            "page_start": block["page_start"],
            "page_end": block["page_end"],
            "bullet_count": bullet_count,
            "text": block["text"],
        })


# 3. Ймовірні розриви абзацу всередині однієї сторінки
for block_index in range(
    len(ogden_all_blocks) - 1
):
    current_block = ogden_all_blocks[block_index]
    next_block = ogden_all_blocks[block_index + 1]

    same_page = (
        current_block["page_end"]
        == next_block["page_start"]
    )

    same_unit = (
        current_block["unit_number"]
        == next_block["unit_number"]
    )

    current_text = current_block["text"].strip()
    next_text = next_block["text"].strip()

    current_ends_sentence = bool(
        re.search(
            r'[.!?]["”’\']?$',
            current_text
        )
    )

    next_starts_lowercase = bool(
        re.match(
            r"^[a-z]",
            next_text
        )
    )

    next_is_structure = (
        looks_like_heading(next_text)
        or next_text.startswith("- ")
        or bool(
            re.match(
                r"^\d+[\.\)]?\s+",
                next_text
            )
        )
    )

    if (
        same_page
        and same_unit
        and not current_ends_sentence
        and next_starts_lowercase
        and not next_is_structure
    ):
        cleanup_audit[
            "possible_same_page_splits"
        ].append({
            "current_block_index": block_index,
            "next_block_index": block_index + 1,
            "unit_number": current_block[
                "unit_number"
            ],
            "page_number": current_block[
                "page_end"
            ],
            "current_text": current_text,
            "next_text": next_text,
        })


print(
    "Duplicate unit titles:",
    len(cleanup_audit["duplicate_unit_titles"])
)

print(
    "Blocks with inline bullets:",
    len(cleanup_audit["inline_bullet_blocks"])
)

print(
    "Possible same-page paragraph splits:",
    len(cleanup_audit["possible_same_page_splits"])
)


print("\nDUPLICATE UNIT TITLES")

for finding in cleanup_audit[
    "duplicate_unit_titles"
]:
    print(
        f'Unit {finding["unit_number"]}, '
        f'page {finding["page_start"]}: '
        f'{finding["text"]}'
    )


print("\nINLINE BULLET BLOCKS")

for finding in cleanup_audit[
    "inline_bullet_blocks"
]:
    print(
        f'Page {finding["page_start"]}'
        f'–{finding["page_end"]}, '
        f'bullets={finding["bullet_count"]}'
    )
    print(finding["text"][:500])
    print("-" * 80)


print("\nPOSSIBLE SAME-PAGE SPLITS")

for finding in cleanup_audit[
    "possible_same_page_splits"
]:
    print(
        f'Page {finding["page_number"]}, '
        f'Unit {finding["unit_number"]}'
    )
    print("LEFT:")
    print(finding["current_text"][-300:])
    print("\nRIGHT:")
    print(finding["next_text"][:300])
    print("=" * 80)

Duplicate unit titles: 3
Blocks with inline bullets: 8
Possible same-page paragraph splits: 2

DUPLICATE UNIT TITLES
Unit 1, page 4: Unit 1: An Introduction to the Key Theoretical Frameworks of
Unit 4, page 37: Unit 4: Becoming Ill and the Role of Illness Cognitions, Help-Seeking, and
Unit 7, page 83: Unit 7: Gender, Health, and Illness

INLINE BULLET BLOCKS
Page 10–11, bullets=6
Mr. A grew up in a poor area of India. Both of his parents smoked because it helped them relax after a difficult day. Mr. A was given his first cigarette by his friend when he was 12 and they had great fun learning to smoke without coughing too much. If they were lucky, they found half-smoked cigarettes lying around that they could smoke, but as they grew older his parents would give him one of theirs. Sitting with his dad having a cigarette was a chance to chat with him. Smoking then became a h
--------------------------------------------------------------------------------
Page 12–12, bullets=1
• To improve 

In [ ]:
from copy import deepcopy


def repair_inline_bullets(text: str) -> str:
    """
    Перетворює PDF-bullets виду:
    текст • item 1 • item 2

    у Markdown:
    текст

    - item 1

    - item 2
    """

    text = text.strip()

    # Bullet на початку блока.
    text = re.sub(
        r"^\s*•\s*",
        "- ",
        text
    )

    # Bullets усередині блока.
    text = re.sub(
        r"\s+•\s+",
        "\n\n- ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


ogden_cleanup_blocks = deepcopy(
    ogden_all_blocks
)


# 1. Склеюємо підтверджені same-page splits.
# Ідемо з кінця, щоб індекси не зсунулися.
same_page_merge_pairs = sorted(
    {
        (
            finding["current_block_index"],
            finding["next_block_index"],
        )
        for finding in cleanup_audit[
            "possible_same_page_splits"
        ]
    },
    reverse=True
)


for left_index, right_index in same_page_merge_pairs:
    left_block = ogden_cleanup_blocks[left_index]
    right_block = ogden_cleanup_blocks[right_index]

    assert right_index == left_index + 1
    assert (
        left_block["page_end"]
        == right_block["page_start"]
    )
    assert (
        left_block["unit_number"]
        == right_block["unit_number"]
    )

    left_block["text"] = re.sub(
        r"\s+",
        " ",
        (
            left_block["text"].rstrip()
            + " "
            + right_block["text"].lstrip()
        )
    ).strip()

    left_block["page_end"] = max(
        left_block["page_end"],
        right_block["page_end"]
    )

    del ogden_cleanup_blocks[right_index]


# 2. Перетворюємо всі символи • на Markdown bullets.
for block in ogden_cleanup_blocks:
    if "•" in block["text"]:
        block["text"] = repair_inline_bullets(
            block["text"]
        )


print(
    "Blocks before cleanup:",
    len(ogden_all_blocks)
)

print(
    "Same-page merges applied:",
    len(same_page_merge_pairs)
)

print(
    "Blocks after cleanup:",
    len(ogden_cleanup_blocks)
)

print(
    "Remaining bullet symbols:",
    sum(
        block["text"].count("•")
        for block in ogden_cleanup_blocks
    )
)


expected_cleanup_block_count = (
    len(ogden_all_blocks)
    - len(same_page_merge_pairs)
)

assert (
    len(ogden_cleanup_blocks)
    == expected_cleanup_block_count
)

assert all(
    "•" not in block["text"]
    for block in ogden_cleanup_blocks
)

assert all(
    "•" not in block["text"]
    for block in ogden_cleanup_blocks
)

Blocks before cleanup: 386
Same-page merges applied: 2
Blocks after cleanup: 384
Remaining bullet symbols: 0


In [ ]:
remaining_cleanup_findings = {
    "same_page_splits": [],
    "duplicate_unit_titles": [],
}


def looks_like_heading_or_list(text: str) -> bool:
    text = text.strip()

    return bool(
        re.match(
            r"^(?:"
            r"#|"
            r"-\s|"
            r"Unit\s+\d+[\.:]|"
            r"Overview$|"
            r"Contents$|"
            r"Questions$|"
            r"Summary$|"
            r"In Summary$|"
            r"To Conclude$|"
            r"Final Take-Home Message$|"
            r"####\s+(?:Figure|Fig\.?)"
            r")",
            text,
            flags=re.IGNORECASE
        )
    )


# 1. Залишкові same-page splits
for block_index in range(
    len(ogden_cleanup_blocks) - 1
):
    left = ogden_cleanup_blocks[block_index]
    right = ogden_cleanup_blocks[block_index + 1]

    same_page = (
        left["page_end"] == right["page_start"]
    )

    same_unit = (
        left["unit_number"] == right["unit_number"]
    )

    left_text = left["text"].strip()
    right_text = right["text"].strip()

    left_ends_sentence = bool(
        re.search(
            r'[.!?]["”’\']?$',
            left_text
        )
    )

    right_is_structure = (
        looks_like_heading_or_list(right_text)
        or bool(
            re.match(
                r"^\d+[\.\)]?\s+",
                right_text
            )
        )
    )

    if (
        same_page
        and same_unit
        and not left_ends_sentence
        and not right_is_structure
    ):
        remaining_cleanup_findings[
            "same_page_splits"
        ].append({
            "left_index": block_index,
            "right_index": block_index + 1,
            "page_number": left["page_end"],
            "unit_number": left["unit_number"],
            "left_text": left_text,
            "right_text": right_text,
        })


# 2. Дубльовані unit titles
for unit_number in range(1, 9):
    expected_prefix = f"unit {unit_number}"

    unit_blocks = [
        block
        for block in ogden_cleanup_blocks
        if block["unit_number"] == unit_number
    ]

    for block_index, block in enumerate(
        unit_blocks[:4]
    ):
        normalized = re.sub(
            r"\s+",
            " ",
            block["text"]
        ).strip().lower()

        if normalized.startswith(expected_prefix):
            remaining_cleanup_findings[
                "duplicate_unit_titles"
            ].append({
                "unit_number": unit_number,
                "block_index": block_index,
                "page_start": block["page_start"],
                "page_end": block["page_end"],
                "text": block["text"],
            })


print(
    "Remaining same-page splits:",
    len(
        remaining_cleanup_findings[
            "same_page_splits"
        ]
    )
)

print(
    "Duplicate unit titles:",
    len(
        remaining_cleanup_findings[
            "duplicate_unit_titles"
        ]
    )
)


print("\nREMAINING SAME-PAGE SPLITS")

for finding in remaining_cleanup_findings[
    "same_page_splits"
]:
    print(
        f'Page {finding["page_number"]}, '
        f'Unit {finding["unit_number"]}'
    )
    print("LEFT:")
    print(finding["left_text"][-300:])
    print("\nRIGHT:")
    print(finding["right_text"][:300])
    print("=" * 80)


print("\nDUPLICATE UNIT TITLES")

for finding in remaining_cleanup_findings[
    "duplicate_unit_titles"
]:
    print(
        f'Unit {finding["unit_number"]}, '
        f'page {finding["page_start"]}: '
        f'{finding["text"]}'
    )

Remaining same-page splits: 113
Duplicate unit titles: 8

REMAINING SAME-PAGE SPLITS
Page 2, Unit None
LEFT:
Overview

RIGHT:
For centuries health professionals have recognized that there are psychological consequences of being ill. A diagnosis of cancer or diabetes can make people anxious or depressed. This course will draw upon health psychology, public health, and community psychology to emphasize how psychology can als
Page 2, Unit None
LEFT:
Learning objectives and outcomes

RIGHT:
By the end of this course students will be able to:
Page 2, Unit None
LEFT:
8. Understand the importance of psychological health outcomes including Quality of Life and health status

RIGHT:
About the author
Page 2, Unit None
LEFT:
About the author

RIGHT:
Jane Ogden is a Professor in Health Psychology at the University of Surrey in the UK where she teaches psychology, dietician, nutrition, medical and vet students to think more psychologically about health and illness. Jane’s research interests focus on

In [ ]:
from copy import deepcopy


ogden_final_blocks = deepcopy(
    ogden_cleanup_blocks
)


def merge_exact_adjacent_blocks(
    blocks: list[dict],
    page_number: int,
    left_ending: str,
    right_beginning: str,
) -> None:
    """
    Склеює тільки одну точно визначену пару
    сусідніх блоків на заданій сторінці.
    """

    matching_pairs = []

    for index in range(len(blocks) - 1):
        left = blocks[index]
        right = blocks[index + 1]

        if (
            left["page_end"] == page_number
            and right["page_start"] == page_number
            and left["unit_number"] == right["unit_number"]
            and left["text"].rstrip().endswith(left_ending)
            and right["text"].lstrip().startswith(right_beginning)
        ):
            matching_pairs.append(index)

    assert len(matching_pairs) == 1, (
        f"Expected exactly one match on page {page_number}, "
        f"found {len(matching_pairs)}"
    )

    left_index = matching_pairs[0]
    right_index = left_index + 1

    left = blocks[left_index]
    right = blocks[right_index]

    left["text"] = re.sub(
        r"\s+",
        " ",
        (
            left["text"].rstrip()
            + " "
            + right["text"].lstrip()
        )
    ).strip()

    left["page_end"] = max(
        left["page_end"],
        right["page_end"]
    )

    del blocks[right_index]

In [ ]:
merge_exact_adjacent_blocks(
    blocks=ogden_final_blocks,
    page_number=2,
    left_ending="(published by",
    right_beginning="McGraw Hill)",
)

merge_exact_adjacent_blocks(
    blocks=ogden_final_blocks,
    page_number=62,
    left_ending="illustrated in",
    right_beginning="Figure 4. It suggested",
)

print(
    "Blocks after exact paragraph repairs:",
    len(ogden_final_blocks)
)

Blocks after exact paragraph repairs: 382


In [ ]:
removed_unit_title_blocks = []

for unit_number in range(1, 9):
    unit_indices = [
        index
        for index, block in enumerate(ogden_final_blocks)
        if block["unit_number"] == unit_number
    ]

    overview_indices = [
        index
        for index in unit_indices
        if (
            ogden_final_blocks[index]["text"]
            .strip()
            .lower()
            == "overview"
        )
    ]

    assert len(overview_indices) == 1

    overview_index = overview_indices[0]

    leading_unit_indices = [
        index
        for index in unit_indices
        if index < overview_index
    ]

    assert len(leading_unit_indices) >= 1

    for index in leading_unit_indices:
        removed_unit_title_blocks.append({
            "unit_number": unit_number,
            "text": ogden_final_blocks[index]["text"],
        })

    for index in sorted(
        leading_unit_indices,
        reverse=True
    ):
        del ogden_final_blocks[index]

In [ ]:
print(
    "Removed duplicate unit-title fragments:",
    len(removed_unit_title_blocks)
)

for item in removed_unit_title_blocks:
    print(
        f'Unit {item["unit_number"]}: '
        f'{item["text"]}'
    )

print(
    "Final cleaned blocks:",
    len(ogden_final_blocks)
)

assert len(removed_unit_title_blocks) == 11

expected_final_block_count = (
    len(ogden_cleanup_blocks)
    - 2
    - len(removed_unit_title_blocks)
)

assert (
    len(ogden_final_blocks)
    == expected_final_block_count
)

assert any(
    block["text"].strip() == "About the author"
    for block in ogden_final_blocks
)

assert any(
    block["text"].strip() == "Final Take-Home Message"
    for block in ogden_final_blocks
)

print(
    "\nFinal Ogden block cleanup passed"
)

Removed duplicate unit-title fragments: 11
Unit 1: Unit 1: An Introduction to the Key Theoretical Frameworks of
Unit 1: Psychology and Health
Unit 2: Unit 2. The Role of Behavior in Health
Unit 3: Unit 3. Behavior Change
Unit 4: Unit 4: Becoming Ill and the Role of Illness Cognitions, Help-Seeking, and
Unit 4: Communication
Unit 5: Unit 5. Being Ill and the Experience of Stress and Pain
Unit 6: Unit 6. The Role of Psychology in Chronic Illnesses such as Obesity,
Unit 6: Coronary Heart Disease (CHD), and Cancer.
Unit 7: Unit 7: Gender, Health, and Illness
Unit 8: Unit 8. Health Outcomes and Quality of Life (QoL)
Final cleaned blocks: 371

Final Ogden block cleanup passed


In [ ]:
from collections import defaultdict


final_front_matter_blocks = [
    block
    for block in ogden_final_blocks
    if block["unit_number"] is None
]

final_unit_blocks = defaultdict(list)

for block in ogden_final_blocks:
    if block["unit_number"] is not None:
        final_unit_blocks[
            block["unit_number"]
        ].append(block)


print(
    "Final Course Overview and Contents blocks:",
    len(final_front_matter_blocks)
)

for unit_number in range(1, 9):
    print(
        f"Unit {unit_number}:",
        len(final_unit_blocks[unit_number]),
        "blocks"
    )

Final Course Overview and Contents blocks: 23
Unit 1: 48 blocks
Unit 2: 67 blocks
Unit 3: 20 blocks
Unit 4: 54 blocks
Unit 5: 57 blocks
Unit 6: 45 blocks
Unit 7: 35 blocks
Unit 8: 22 blocks


In [ ]:
def compact_repeated_page_comments(markdown_text: str) -> str:
    """
    Залишає тільки перший provenance-коментар
    у послідовності блоків з однаковим page range.

    Різні діапазони, наприклад 8–8 і 8–9,
    зберігаються окремо.
    """

    page_comment_pattern = re.compile(
        r"^<!-- pages:\s*(\d+)–(\d+)\s*-->$"
    )

    output_lines = []
    last_page_range = None
    removed_comments = 0

    lines = markdown_text.splitlines()

    for line in lines:
        match = page_comment_pattern.fullmatch(
            line.strip()
        )

        if match:
            current_page_range = (
                int(match.group(1)),
                int(match.group(2)),
            )

            if current_page_range == last_page_range:
                removed_comments += 1
                continue

            last_page_range = current_page_range

        output_lines.append(line)

    compacted_text = "\n".join(output_lines)

    # Прибираємо зайві порожні рядки,
    # які залишилися після видалення comments.
    compacted_text = re.sub(
        r"\n{3,}",
        "\n\n",
        compacted_text
    ).strip() + "\n"

    return compacted_text, removed_comments

In [ ]:
ogden_normalized_markdown = (
    build_ogden_normalized_markdown(
        metadata=OGDEN_METADATA,
        front_matter_blocks=final_front_matter_blocks,
        unit_blocks=final_unit_blocks,
    )
)

ogden_compact_markdown, removed_page_comments = (
    compact_repeated_page_comments(
        ogden_normalized_markdown
    )
)

OGDEN_NORMALIZED_PATH.write_text(
    ogden_compact_markdown,
    encoding="utf-8",
)

print(
    "Repeated page comments removed:",
    removed_page_comments,
)

print(
    "Saved:",
    OGDEN_NORMALIZED_PATH,
)

print(
    "Characters:",
    len(ogden_compact_markdown),
)

print(
    "File size:",
    OGDEN_NORMALIZED_PATH.stat().st_size,
)

Repeated page comments removed: 254
Saved: /content/health-psychology-rag-kb/data/normalized/ogden_2019_health_psychology.md
Characters: 154307
File size: 155415


In [ ]:
for path in sorted(
    OGDEN_ASSETS_DIR.glob("ogden_*.png")
):
    print(
        path.name,
        "→",
        path.stat().st_size,
        "bytes"
    )

ogden_p10_figure_4.png → 24832 bytes
ogden_p22_figure_3.png → 327717 bytes
ogden_p39_figure_1.png → 347488 bytes
ogden_p54_figure_1.png → 176207 bytes
ogden_p55_figure_2.png → 204044 bytes
ogden_p7_figure_1.png → 169013 bytes
ogden_p8_figure_2.png → 340635 bytes
ogden_p9_figure_3.png → 204578 bytes


In [ ]:
ogden_blocks_for_chunking = [
    block.strip()
    for block in ogden_final_text.split("\n\n")
    if block.strip()
]


empty_blocks = [
    block
    for block in ogden_blocks_for_chunking
    if not block.strip()
]

comment_only_blocks = [
    block
    for block in ogden_blocks_for_chunking
    if re.fullmatch(
        r"<!--.*?-->",
        block,
        flags=re.DOTALL,
    )
]

standalone_figure_descriptions = [
    block
    for block in ogden_blocks_for_chunking
    if block.startswith(
        "Figure description:"
    )
]

very_short_content_blocks = [
    block
    for block in ogden_blocks_for_chunking
    if (
        len(block) < 40
        and not block.startswith("#")
        and not block.startswith("<!--")
    )
]


print(
    "Total Markdown blocks:",
    len(ogden_blocks_for_chunking),
)

print(
    "Empty blocks:",
    len(empty_blocks),
)

print(
    "Comment-only blocks:",
    len(comment_only_blocks),
)

print(
    "Standalone figure descriptions:",
    len(standalone_figure_descriptions),
)

print(
    "Very short content blocks:",
    len(very_short_content_blocks),
)


assert empty_blocks == []

assert len(
    re.findall(
        r"^## Unit \d+:",
        ogden_final_text,
        flags=re.MULTILINE,
    )
) == 8

assert len(
    re.findall(
        r"^#### (?:Figure|Fig\.?)\s+\d+",
        ogden_final_text,
        flags=re.MULTILINE,
    )
) == 15

print(
    "\nOgden structure is ready "
    "for the chunking stage"
)

# Stage 2: Input Manifest and Chunking

In [ ]:
from pathlib import Path


PROJECT_ROOT = Path(
    "/content/health-psychology-rag-kb"
)

NORMALIZED_DIR = (
    PROJECT_ROOT
    / "data"
    / "normalized"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

CHUNKS_PATH = (
    PROCESSED_DIR
    / "chunks.jsonl"
)


CHUNK_MIN_CHARS = 500
CHUNK_TARGET_CHARS = 800
CHUNK_MAX_CHARS = 1000
CHUNK_OVERLAP_CHARS = 120


PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Normalized directory:",
    NORMALIZED_DIR
)

print(
    "Processed directory:",
    PROCESSED_DIR
)

print(
    "Chunks output:",
    CHUNKS_PATH
)

print(
    "Chunk configuration:",
    {
        "min_chars": CHUNK_MIN_CHARS,
        "target_chars": CHUNK_TARGET_CHARS,
        "max_chars": CHUNK_MAX_CHARS,
        "overlap_chars": CHUNK_OVERLAP_CHARS,
    }
)


assert NORMALIZED_DIR.exists()
assert CHUNK_MIN_CHARS < CHUNK_TARGET_CHARS
assert CHUNK_TARGET_CHARS <= CHUNK_MAX_CHARS
assert CHUNK_OVERLAP_CHARS < CHUNK_MIN_CHARS

print(
    "\nStage 2 configuration passed"
)

Normalized directory: /content/health-psychology-rag-kb/data/normalized
Processed directory: /content/health-psychology-rag-kb/data/processed
Chunks output: /content/health-psychology-rag-kb/data/processed/chunks.jsonl
Chunk configuration: {'min_chars': 500, 'target_chars': 800, 'max_chars': 1000, 'overlap_chars': 120}

Stage 2 configuration passed


In [ ]:
NORMALIZED_DOCUMENTS = [
    {
        "document_id": "health_psychology_course_syllabus",
        "title": "Syllabus for Introduction to Health Psychology",
        "source_file": "data/raw/course_syllabus.pdf",
        "normalized_file": "course_syllabus.md",
        "source_type": "pdf",
        "document_type": "syllabus",
        "content_role": "course_structure_and_assessment",
        "domain": "health_psychology",
        "language": "en",
    },
    {
        "document_id": "wright_2019_3p_disease_model",
        "title": (
            "A Framework for Understanding the Role of Psychological "
            "Processes in Disease Development, Maintenance, and Treatment: "
            "The 3P-Disease Model"
        ),
        "source_file": (
            "data/raw/wright_2019_3p_disease_model.html"
        ),
        "normalized_file": (
            "wright_2019_3p_disease_model.md"
        ),
        "source_type": "html",
        "document_type": "journal_article",
        "content_role": "supplementary_research",
        "domain": "health_psychology",
        "language": "en",
    },
    {
        "document_id": "michie_2011_behaviour_change_wheel",
        "title": (
            "The Behaviour Change Wheel: A New Method for "
            "Characterising and Designing Behaviour Change Interventions"
        ),
        "source_file": (
            "data/raw/michie_2011_behaviour_change_wheel.html"
        ),
        "normalized_file": (
            "michie_2011_behaviour_change_wheel.md"
        ),
        "source_type": "html",
        "document_type": "journal_article",
        "content_role": "core_behaviour_change_framework",
        "domain": "health_psychology",
        "language": "en",
    },
    {
        "document_id": "ogden_2019_health_psychology",
        "title": (
            "The Psychology of Health and Illness: "
            "An Open Access Course"
        ),
        "source_file": (
            "data/raw/ogden_2019_health_psychology.pdf"
        ),
        "normalized_file": (
            "ogden_2019_health_psychology.md"
        ),
        "source_type": "pdf",
        "document_type": "textbook",
        "content_role": "core_reading",
        "domain": "health_psychology",
        "language": "en",
    },
]


for document in NORMALIZED_DOCUMENTS:
    document["normalized_path"] = (
        NORMALIZED_DIR
        / document["normalized_file"]
    )

    document["source_path"] = (
        PROJECT_ROOT
        / document["source_file"]
    )


print(
    "Input manifest documents:",
    len(NORMALIZED_DOCUMENTS)
)

for document in NORMALIZED_DOCUMENTS:
    print(
        f'{document["document_id"]} → '
        f'normalized={document["normalized_file"]} | '
        f'source={document["source_file"]}'
    )

Input manifest documents: 4
health_psychology_course_syllabus → normalized=course_syllabus.md | source=data/raw/course_syllabus.pdf
wright_2019_3p_disease_model → normalized=wright_2019_3p_disease_model.md | source=data/raw/wright_2019_3p_disease_model.html
michie_2011_behaviour_change_wheel → normalized=michie_2011_behaviour_change_wheel.md | source=data/raw/michie_2011_behaviour_change_wheel.html
ogden_2019_health_psychology → normalized=ogden_2019_health_psychology.md | source=data/raw/ogden_2019_health_psychology.pdf


In [ ]:
REQUIRED_DOCUMENT_FIELDS = {
    "document_id",
    "title",
    "source_file",
    "normalized_file",
    "source_type",
    "document_type",
    "content_role",
    "domain",
    "language",
    "normalized_path",
    "source_path",
}


document_ids = []


for document in NORMALIZED_DOCUMENTS:
    missing_fields = (
        REQUIRED_DOCUMENT_FIELDS
        - set(document)
    )

    assert not missing_fields, (
        f'Missing fields for '
        f'{document.get("document_id", "unknown")}: '
        f'{sorted(missing_fields)}'
    )

    assert all(
        str(document[field]).strip()
        for field in REQUIRED_DOCUMENT_FIELDS
    ), (
        f'Empty metadata field in '
        f'{document["document_id"]}'
    )

    assert document["normalized_path"].exists(), (
        f'Missing normalized file: '
        f'{document["normalized_path"]}'
    )

    assert document["source_path"].exists(), (
        f'Missing source file: '
        f'{document["source_path"]}'
    )

    document_ids.append(
        document["document_id"]
    )

    print(
        f'{document["document_id"]} → '
        f'normalized=True | source=True'
    )


assert len(document_ids) == len(
    set(document_ids)
), "Duplicate document_id found"


print(
    "\nInput manifest validation passed"
)

health_psychology_course_syllabus → normalized=True | source=True
wright_2019_3p_disease_model → normalized=True | source=True
michie_2011_behaviour_change_wheel → normalized=True | source=True
ogden_2019_health_psychology → normalized=True | source=True

Input manifest validation passed


In [ ]:
import re


def remove_yaml_front_matter(
    text: str,
) -> str:
    return re.sub(
        r"\A---\s*\n.*?\n---\s*\n",
        "",
        text,
        count=1,
        flags=re.DOTALL,
    ).strip()


normalized_texts = {}


for document in NORMALIZED_DOCUMENTS:
    document_id = document["document_id"]

    raw_text = document[
        "normalized_path"
    ].read_text(
        encoding="utf-8"
    )

    clean_text = remove_yaml_front_matter(
        raw_text
    )

    assert clean_text, (
        f"Empty normalized document: "
        f"{document_id}"
    )

    normalized_texts[
        document_id
    ] = clean_text

    print(
        document_id,
        "→",
        len(clean_text),
        "characters",
    )


print(
    "\nNormalized texts loaded for chunking"
)

health_psychology_course_syllabus → 14411 characters
wright_2019_3p_disease_model → 71436 characters
michie_2011_behaviour_change_wheel → 49086 characters
ogden_2019_health_psychology → 152134 characters

Normalized texts loaded for chunking


In [ ]:
HEADING_PATTERN = re.compile(
    r"^(#{1,6})\s+(.+)$"
)

HTML_COMMENT_ONLY_PATTERN = re.compile(
    r"^(?:<!--.*?-->\s*)+$",
    flags=re.DOTALL,
)


def parse_markdown_sections(
    text: str,
) -> list[dict]:
    sections = []

    current_section = "Document overview"
    current_blocks = []

    for raw_block in re.split(
        r"\n\s*\n",
        text,
    ):
        block = raw_block.strip()

        if not block:
            continue

        if HTML_COMMENT_ONLY_PATTERN.fullmatch(
            block
        ):
            continue

        heading_match = HEADING_PATTERN.fullmatch(
            block
        )

        if heading_match:
            heading_level = len(
                heading_match.group(1)
            )

            heading_text = (
                heading_match.group(2).strip()
            )

            if heading_level == 1:
                continue

            if heading_level in {2, 3}:
                if current_blocks:
                    sections.append({
                        "section": current_section,
                        "blocks": current_blocks,
                    })

                current_section = heading_text
                current_blocks = []
                continue

            current_blocks.append(block)
            continue

        current_blocks.append(block)

    if current_blocks:
        sections.append({
            "section": current_section,
            "blocks": current_blocks,
        })

    return sections

In [ ]:
document_sections = {}

for document in NORMALIZED_DOCUMENTS:
    document_id = document["document_id"]

    sections = parse_markdown_sections(
        normalized_texts[document_id]
    )

    document_sections[document_id] = sections

    print(
        f"{document_id} → "
        f"{len(sections)} sections"
    )

print(
    "\nSections rebuilt"
)

health_psychology_course_syllabus → 4 sections
wright_2019_3p_disease_model → 8 sections
michie_2011_behaviour_change_wheel → 14 sections
ogden_2019_health_psychology → 48 sections

Sections rebuilt


In [ ]:
def normalize_chunk_text(
    text: str,
) -> str:
    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def find_split_position(
    text: str,
    start: int,
) -> int:
    text_length = len(text)

    minimum_end = min(
        start + CHUNK_MIN_CHARS,
        text_length,
    )

    target_end = min(
        start + CHUNK_TARGET_CHARS,
        text_length,
    )

    maximum_end = min(
        start + CHUNK_MAX_CHARS,
        text_length,
    )

    if maximum_end == text_length:
        return text_length

    latest_valid_end = (
        text_length
        + CHUNK_OVERLAP_CHARS
        - CHUNK_MIN_CHARS
    )

    maximum_end = min(
        maximum_end,
        latest_valid_end,
    )

    target_end = min(
        target_end,
        maximum_end,
    )

    if maximum_end <= minimum_end:
        return minimum_end

    candidate = text[
        minimum_end:target_end
    ]

    sentence_boundaries = list(
        re.finditer(
            r'[.!?]["”’)\]]*\s+',
            candidate,
        )
    )

    if sentence_boundaries:
        return (
            minimum_end
            + sentence_boundaries[-1].end()
        )

    space_position = text.rfind(
        " ",
        minimum_end,
        target_end,
    )

    if space_position > start:
        return space_position

    space_position = text.find(
        " ",
        target_end,
        maximum_end,
    )

    if space_position != -1:
        return space_position

    return maximum_end


def find_overlap_start(
    text: str,
    chunk_start: int,
    chunk_end: int,
) -> int:
    desired_start = max(
        chunk_start + 1,
        chunk_end - CHUNK_OVERLAP_CHARS,
    )

    previous_space = text.rfind(
        " ",
        chunk_start + 1,
        desired_start + 1,
    )

    if previous_space != -1:
        return previous_space + 1

    return desired_start


def split_section_text(
    text: str,
) -> list[str]:
    text = normalize_chunk_text(text)

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = find_split_position(
            text,
            start,
        )

        if end <= start:
            raise RuntimeError(
                "Chunking did not advance: "
                f"start={start}, end={end}"
            )

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        next_start = find_overlap_start(
            text,
            start,
            end,
        )

        if next_start <= start:
            raise RuntimeError(
                "Overlap did not advance: "
                f"start={start}, "
                f"next_start={next_start}"
            )

        start = next_start

    return chunks


print(
    "Section chunking functions defined"
)

Section chunking functions defined


In [ ]:
section_chunks = []

for document in NORMALIZED_DOCUMENTS:
    document_id = document["document_id"]

    sections = document_sections[
        document_id
    ]

    for section in sections:
        section_text = "\n\n".join(
            section["blocks"]
        )

        chunks = split_section_text(
            section_text
        )

        for section_chunk_index, chunk_text in enumerate(
            chunks
        ):
            section_chunks.append({
                "document_id": document_id,
                "title": document["title"],
                "source_file": document["source_file"],
                "source_type": document["source_type"],
                "document_type": document["document_type"],
                "content_role": document["content_role"],
                "domain": document["domain"],
                "language": document["language"],
                "section": section["section"],
                "section_chunk_index": section_chunk_index,
                "text": chunk_text,
            })

print(
    "Section chunks rebuilt:",
    len(section_chunks)
)

Section chunks rebuilt: 474


In [ ]:
short_chunk_indexes = [
    index
    for index, item in enumerate(section_chunks)
    if len(item["text"]) < CHUNK_MIN_CHARS
]


print(
    "Short chunks to review:",
    len(short_chunk_indexes)
)


for review_number, chunk_index in enumerate(
    short_chunk_indexes,
    start=1,
):
    item = section_chunks[chunk_index]

    previous_item = (
        section_chunks[chunk_index - 1]
        if chunk_index > 0
        else None
    )

    next_item = (
        section_chunks[chunk_index + 1]
        if chunk_index + 1
        < len(section_chunks)
        else None
    )

    same_section_previous = (
        previous_item is not None
        and previous_item["document_id"]
        == item["document_id"]
        and previous_item["section"]
        == item["section"]
    )

    same_section_next = (
        next_item is not None
        and next_item["document_id"]
        == item["document_id"]
        and next_item["section"]
        == item["section"]
    )

    print("\n" + "=" * 100)

    print(
        f"SHORT CHUNK {review_number} "
        f"| global index={chunk_index}"
    )

    print(
        "Document:",
        item["document_id"]
    )

    print(
        "Section:",
        item["section"]
    )

    print(
        "Length:",
        len(item["text"])
    )

    print(
        "Previous chunk in same section:",
        same_section_previous
    )

    print(
        "Next chunk in same section:",
        same_section_next
    )

    if same_section_previous:
        print(
            "\nPREVIOUS END:\n",
            previous_item["text"][-300:],
        )

    print(
        "\nSHORT CHUNK:\n",
        item["text"],
    )

    if same_section_next:
        print(
            "\nNEXT START:\n",
            next_item["text"][:300],
        )

Short chunks to review: 10

SHORT CHUNK 1 | global index=18
Document: health_psychology_course_syllabus
Section: Grading Scale
Length: 393
Previous chunk in same section: False
Next chunk in same section: False

SHORT CHUNK:
 | Letter Grade | Percentage Grade | Grade Points | | --- | --- | --- | | A+ | 98%-100% | 4.00 | | A | 93%-97% | 4.00 | | A- | 90%-92% | 3.67 | | B+ | 88%-89% | 3.33 | | B | 83%-87% | 3.00 | | B- | 80%-82% | 2.67 | | C+ | 78%-79% | 2.33 | | C | 73%-77% | 2.00 | | C- | 70%-72% | 1.67 | | D+ | 68%-69% | 1.33 | | D | 63%-67% | 1.00 | | D- | 60%-62% | 0.67 | | F | <60% | 0.00 | | W | N/A | N/A |

SHORT CHUNK 2 | global index=145
Document: michie_2011_behaviour_change_wheel
Section: Background
Length: 496
Previous chunk in same section: False
Next chunk in same section: False

SHORT CHUNK:
 Improving the design and implementation of evidence-based practice depends on successful behaviour change interventions. This requires an appropriate method for characterising interv

In [ ]:
chunk_lengths = [
    len(item["text"])
    for item in section_chunks
]

assert section_chunks, (
    "No section chunks were created"
)

assert all(
    item["text"].strip()
    for item in section_chunks
), "Empty chunk text found"

assert all(
    length <= CHUNK_MAX_CHARS
    for length in chunk_lengths
), "Chunk exceeds CHUNK_MAX_CHARS"


print(
    "Total chunks:",
    len(section_chunks)
)

print(
    "Shortest chunk:",
    min(chunk_lengths)
)

print(
    "Longest chunk:",
    max(chunk_lengths)
)

print(
    "Chunks under minimum:",
    sum(
        length < CHUNK_MIN_CHARS
        for length in chunk_lengths
    )
)

print(
    "Chunks over maximum:",
    sum(
        length > CHUNK_MAX_CHARS
        for length in chunk_lengths
    )
)

print(
    "\nChunk length validation passed"
)

Total chunks: 474
Shortest chunk: 204
Longest chunk: 997
Chunks under minimum: 10
Chunks over maximum: 0

Chunk length validation passed


In [ ]:
import json


final_chunks = []

document_chunk_counters = {}


for item in section_chunks:
    document_id = item["document_id"]

    chunk_index = document_chunk_counters.get(
        document_id,
        0,
    )

    chunk_id = (
        f"{document_id}__"
        f"{chunk_index:04d}"
    )

    final_chunks.append({
        "chunk_id": chunk_id,
        "document_id": document_id,
        "source_file": item["source_file"],
        "chunk_index": chunk_index,
        "section": item["section"],
        "text": item["text"],
    })

    document_chunk_counters[
        document_id
    ] = chunk_index + 1


print(
    "Final chunk records:",
    len(final_chunks)
)

print(
    "Documents:",
    document_chunk_counters
)

Final chunk records: 474
Documents: {'health_psychology_course_syllabus': 23, 'wright_2019_3p_disease_model': 122, 'michie_2011_behaviour_change_wheel': 82, 'ogden_2019_health_psychology': 247}


In [ ]:
with CHUNKS_PATH.open(
    "w",
    encoding="utf-8",
) as output_file:
    for chunk in final_chunks:
        output_file.write(
            json.dumps(
                chunk,
                ensure_ascii=False,
            )
            + "\n"
        )


print(
    "Chunks written:",
    CHUNKS_PATH
)

print(
    "File exists:",
    CHUNKS_PATH.exists()
)

print(
    "File size:",
    CHUNKS_PATH.stat().st_size,
    "bytes",
)

Chunks written: /content/health-psychology-rag-kb/data/processed/chunks.jsonl
File exists: True
File size: 443697 bytes


In [ ]:
loaded_chunks = []

with CHUNKS_PATH.open(
    "r",
    encoding="utf-8",
) as input_file:
    for line_number, line in enumerate(
        input_file,
        start=1,
    ):
        line = line.strip()

        assert line, (
            f"Empty line found at line "
            f"{line_number}"
        )

        loaded_chunks.append(
            json.loads(line)
        )


REQUIRED_CHUNK_FIELDS = {
    "chunk_id",
    "document_id",
    "source_file",
    "chunk_index",
    "section",
    "text",
}


for line_number, chunk in enumerate(
    loaded_chunks,
    start=1,
):
    missing_fields = (
        REQUIRED_CHUNK_FIELDS
        - set(chunk)
    )

    assert not missing_fields, (
        f"Missing fields at line "
        f"{line_number}: "
        f"{sorted(missing_fields)}"
    )

    assert chunk["text"].strip(), (
        f"Empty text at line "
        f"{line_number}"
    )

    assert len(chunk["text"]) <= CHUNK_MAX_CHARS, (
        f"Chunk exceeds maximum at line "
        f"{line_number}"
    )


chunk_ids = [
    chunk["chunk_id"]
    for chunk in loaded_chunks
]

assert len(chunk_ids) == len(
    set(chunk_ids)
), "Duplicate chunk_id found"

assert len(loaded_chunks) == len(
    final_chunks
), "Written chunk count does not match final_chunks"


print(
    "Validated chunks:",
    len(loaded_chunks)
)

print(
    "Unique chunk IDs:",
    len(set(chunk_ids))
)

print(
    "\nchunks.jsonl validation passed"
)

Validated chunks: 474
Unique chunk IDs: 474

chunks.jsonl validation passed


In [ ]:
!git -C /content/health-psychology-rag-kb status --short

?? data/processed/


In [ ]:
!git -C /content/health-psychology-rag-kb add data/processed/chunks.jsonl

In [ ]:
!git -C /content/health-psychology-rag-kb status --short

A  data/processed/chunks.jsonl


In [ ]:
!git -C /content/health-psychology-rag-kb commit -m "Add section-aware chunked knowledge base"

[main 06669d5] Add section-aware chunked knowledge base
 1 file changed, 474 insertions(+)
 create mode 100644 data/processed/chunks.jsonl


In [ ]:
!git -C /content/health-psychology-rag-kb push

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 94.15 KiB | 3.36 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/swanksenia/health-psychology-rag-kb.git
   bf745ed..06669d5  main -> main


In [ ]:
!git -C /content/health-psychology-rag-kb status --short

# Stage 3: Embeddings and Vector Index

In [ ]:
!pip -q install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.6 MB/s eta 0:00:00


In [ ]:
# Stage 3: Embeddings and vector index

import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

In [ ]:
EMBEDDINGS_PATH = PROCESSED_DIR / "embeddings.npy"
FAISS_INDEX_PATH = PROCESSED_DIR / "faiss.index"

In [ ]:
with CHUNKS_PATH.open("r", encoding="utf-8") as input_file:
    chunks = [
        json.loads(line)
        for line in input_file
    ]

texts = [
    chunk["text"]
    for chunk in chunks
]

print("Chunks loaded:", len(chunks))

Chunks loaded: 474


In [ ]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

embeddings = embeddings.astype(
    np.float32,
    copy=False,
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings shape: (474, 384)


In [ ]:
np.save(
    EMBEDDINGS_PATH,
    embeddings,
)

print("Saved:", EMBEDDINGS_PATH)

Saved: /content/health-psychology-rag-kb/data/processed/embeddings.npy


In [ ]:
faiss_index = faiss.IndexFlatIP(
    embeddings.shape[1]
)

faiss_index.add(embeddings)

print("Vectors in index:", faiss_index.ntotal)

Vectors in index: 474


In [ ]:
faiss.write_index(
    faiss_index,
    str(FAISS_INDEX_PATH),
)

print("Saved:", FAISS_INDEX_PATH)

Saved: /content/health-psychology-rag-kb/data/processed/faiss.index


In [ ]:
loaded_faiss_index = faiss.read_index(
    str(FAISS_INDEX_PATH)
)

assert loaded_faiss_index.ntotal == len(chunks)
assert loaded_faiss_index.d == embeddings.shape[1]

print("FAISS index validation passed")
print("Vectors:", loaded_faiss_index.ntotal)
print("Dimensions:", loaded_faiss_index.d)

FAISS index validation passed
Vectors: 474
Dimensions: 384


In [ ]:
query = "How does stress affect physical health?"

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype(np.float32)

scores, indices = loaded_faiss_index.search(
    query_embedding,
    k=5,
)

for rank, (score, index) in enumerate(
    zip(scores[0], indices[0]),
    start=1,
):
    chunk = chunks[index]

    print(
        f"\n{rank}. score={score:.4f}"
    )
    print(
        f"{chunk['document_id']} | "
        f"{chunk['section']} | "
        f"{chunk['chunk_id']}"
    )
    print(
        chunk["text"][:700]
    )


1. score=0.7040
ogden_2019_health_psychology | Overview | ogden_2019_health_psychology__0135
reasons that stress has been studied so consistently is because of its potential effect on the health of the individual. In particular, research shows a link between high-stress jobs and hypertension and coronary heart disease; higher life stress and physical symptoms; that stressful lives are associated with greater recurrence of colds and flu; and that there is a link between stress and mortality. For example, Phillips, Der, and Carroll (2008) reported from their longitudinal study of 968 men and women aged 56, that the number of health-related life events at baseline and their stress load predicted mortality by 17 years (266 participants had died). Stress can cause illness through either 

2. score=0.6536
ogden_2019_health_psychology | To Conclude | ogden_2019_health_psychology__0167
Stress and pain are part of the continuum from health to illness and illustrate the key role of psychologica

In [ ]:
assert EMBEDDINGS_PATH.exists()
assert FAISS_INDEX_PATH.exists()
assert embeddings.shape[0] == len(chunks)
assert loaded_faiss_index.ntotal == len(chunks)

print("Stage 3 validation passed")

Stage 3 validation passed


In [ ]:
!git -C /content/health-psychology-rag-kb status --short

?? data/processed/embeddings.npy
?? data/processed/faiss.index


In [ ]:
!git -C /content/health-psychology-rag-kb add \
    data/processed/embeddings.npy \
    data/processed/faiss.index

In [ ]:
!git -C /content/health-psychology-rag-kb commit \
    -m "Add embeddings and FAISS vector index"

[main dc4dc27] Add embeddings and FAISS vector index
 2 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 data/processed/embeddings.npy
 create mode 100644 data/processed/faiss.index


In [ ]:
!git -C /content/health-psychology-rag-kb push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 660.09 KiB | 5.04 MiB/s, done.
Total 6 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/swanksenia/health-psychology-rag-kb.git
   06669d5..dc4dc27  main -> main


In [ ]:
!git -C /content/health-psychology-rag-kb status --short

# Stage 4: Prepare submission

In [ ]:
README_PATH = PROJECT_ROOT / "README.md"

example_chunks = []
seen_documents = set()

for chunk in chunks:
    document_id = chunk["document_id"]

    if (
        document_id not in seen_documents
        and len(chunk["text"]) >= CHUNK_MIN_CHARS
    ):
        example_chunks.append(chunk)
        seen_documents.add(document_id)

    if len(example_chunks) == 4:
        break


example_sections = []

for number, chunk in enumerate(example_chunks, start=1):
    example = {
        "chunk_id": chunk["chunk_id"],
        "document_id": chunk["document_id"],
        "source_file": chunk["source_file"],
        "chunk_index": chunk["chunk_index"],
        "section": chunk["section"],
        "text": chunk["text"],
    }

    formatted_json = json.dumps(
        example,
        ensure_ascii=False,
        indent=2,
    )

    indented_json = "\n".join(
        f"    {line}"
        for line in formatted_json.splitlines()
    )

    example_sections.append(
        "\n".join(
            [
                f"### Example {number}",
                "",
                indented_json,
                "",
                (
                    "**Why this chunk works:** "
                    "It preserves the source document, section, "
                    "sequence, and enough context to be understood "
                    "independently."
                ),
                "",
            ]
        )
    )


readme_lines = [
    "# Health Psychology RAG Knowledge Base",
    "",
    "## Subject Area",
    "",
    (
        "This project prepares a knowledge base for a future "
        "chatbot focused on **health psychology**."
    ),
    "",
    "The knowledge base covers:",
    "",
    "- relationships between psychological and physical health;",
    "- stress and illness;",
    "- health behaviour change;",
    "- behavioural intervention design;",
    "- course concepts and learning requirements.",
    "",
    "## Pipeline",
    "",
    "    raw sources",
    "    → normalized Markdown documents",
    "    → section-aware chunking",
    "    → metadata enrichment",
    "    → JSONL knowledge base",
    "    → embeddings and FAISS index",
    "",
    "## Sources",
    "",
    "The project contains four source documents:",
    "",
    "1. `ogden_2019_health_psychology.pdf`",
    "2. `wright_2019_3p_disease_model.html`",
    "3. `michie_2011_behaviour_change_wheel.html`",
    "4. `course_syllabus.pdf`",
    "",
    "Original files are stored in `data/raw/`.",
    "",
    (
        "Normalized Markdown documents are stored "
        "in `data/normalized/`."
    ),
    "",
    "## Chunking Strategy",
    "",
    (
        "The documents were split using a section-aware "
        "chunking strategy."
    ),
    "",
    f"- minimum chunk size: {CHUNK_MIN_CHARS} characters;",
    f"- target chunk size: {CHUNK_TARGET_CHARS} characters;",
    f"- maximum chunk size: {CHUNK_MAX_CHARS} characters;",
    f"- overlap: {CHUNK_OVERLAP_CHARS} characters;",
    "- section headings are preserved;",
    "- paragraph and sentence boundaries are preferred;",
    "- short self-contained sections are retained.",
    "",
    f"The final knowledge base contains **{len(chunks)} chunks**.",
    "",
    "## Metadata Structure",
    "",
    "Each JSONL record contains:",
    "",
    "| Field | Description |",
    "|---|---|",
    "| `chunk_id` | Unique chunk identifier |",
    "| `document_id` | Source document identifier |",
    "| `source_file` | Original source filename |",
    "| `chunk_index` | Sequential position within the document |",
    "| `section` | Source section heading |",
    "| `text` | Chunk content |",
    "",
    (
        "The processed knowledge base is stored in "
        "`data/processed/chunks.jsonl`."
    ),
    "",
    "## Chunk Examples",
    "",
    "\n".join(example_sections),
    "## Quality Review",
    "",
    "### What worked well",
    "",
    (
        "- Four source documents were normalized into a "
        "consistent Markdown format."
    ),
    (
        "- Chunking preserves document sections and "
        "semantic boundaries."
    ),
    (
        "- Every chunk has stable identifiers and "
        "source metadata."
    ),
    "- No chunk exceeds the configured maximum size.",
    (
        "- The JSONL file was validated for structure "
        "and unique chunk IDs."
    ),
    (
        "- Section metadata supports traceability and "
        "future citations."
    ),
    "",
    "### What could be improved",
    "",
    (
        "- Some naturally short sections remain below the "
        "preferred minimum size because they are meaningful "
        "standalone units."
    ),
    (
        "- Additional metadata such as `title`, `language`, "
        "`domain`, and `document_type` could be added later."
    ),
    (
        "- Retrieval quality should be evaluated with a "
        "dedicated test-question set."
    ),
    (
        "- Future work could compare section-aware chunking "
        "with semantic or parent-child chunking."
    ),
    "",
    "## Optional Retrieval Extension",
    "",
    (
        "Beyond the core homework requirements, "
        "the project also includes:"
    ),
    "",
    "- embeddings in `data/processed/embeddings.npy`;",
    "- a FAISS vector index in `data/processed/faiss.index`;",
    (
        f"- a successful semantic-search test over all "
        f"{len(chunks)} chunks."
    ),
    "",
]

README_PATH.write_text(
    "\n".join(readme_lines),
    encoding="utf-8",
)

print("Saved:", README_PATH)

Saved: /content/health-psychology-rag-kb/README.md


In [ ]:
readme_text = README_PATH.read_text(
    encoding="utf-8"
)

required_sections = [
    "# Health Psychology RAG Knowledge Base",
    "## Subject Area",
    "## Sources",
    "## Chunking Strategy",
    "## Metadata Structure",
    "## Chunk Examples",
    "## Quality Review",
]

for section in required_sections:
    assert section in readme_text

assert readme_text.count("### Example ") == 4

print("README validation passed")

README validation passed


In [ ]:
!git -C /content/health-psychology-rag-kb add README.md
!git -C /content/health-psychology-rag-kb commit -m "Add project README"
!git -C /content/health-psychology-rag-kb push

[main de85c2e] Add project README
 1 file changed, 144 insertions(+), 2 deletions(-)
 rewrite README.md (100%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 3.40 KiB | 3.40 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/swanksenia/health-psychology-rag-kb.git
   dc4dc27..de85c2e  main -> main


In [ ]:
!git -C /content/health-psychology-rag-kb status --short

In [ ]:
!git -C /content/health-psychology-rag-kb pull

Already up to date.


In [ ]:
!git -C /content/health-psychology-rag-kb pull

Already up to date.


In [ ]:
NOTEBOOK_PATH = (
    PROJECT_ROOT
    / "notebooks"
    / "prepare_knowledge_base.ipynb"
)

print("Exists:", NOTEBOOK_PATH.exists())

Exists: False
